In [ ]:
library(parallel)
library(dlm)
library(exdqlm)
library(mvtnorm)
library(jmuOutlier)
library(sn)
library(Matrix)
library(future)
library(future.apply)
library(numDeriv)
library(foreach)
library(doParallel)
library(dataRetrieval)
library(dplyr)
library(zoo)
library(tseries)
library(tidyverse)
library(patchwork)
library(rvest)
library(expint)
library(nimble)
library(nloptr)
library(expm)
library(numDeriv)
library(Rcpp)
library(RcppArmadillo)
library(RcppEigen)
library(ks)
library(MASS)
library(FNN)

make_df_mat = function(df,dim.df,n){
  if(sum(dim.df)!=n){ stop("sum of component dimensions given in dim.df does not match m0") }
  if(length(df)!=length(dim.df)){ stop("length of component discount factors does not match length of component dimensions") }
  dfs = rep(df,dim.df)
  n.dfs = length(dim.df)
  ind.dfs = c(0,sapply(1:length(dim.df),function(x){sum(dim.df[1:x])}),n)
  df.mat = matrix(0,n,n)
  for(j in 1:n.dfs){
    df.mat[(ind.dfs[j]+1):ind.dfs[(j+1)],(ind.dfs[j]+1):ind.dfs[(j+1)]] = (1-dfs[ind.dfs[(j+1)]])/dfs[ind.dfs[(j+1)]]
  }
  return(df.mat)
}
#
make_df_mat_k = function(df,dim.df,n,k){
  if(sum(dim.df)!=n){ stop("sum of component dimensions given in dim.df does not match m0") }
  if(length(df)!=length(dim.df)){ stop("length of component discount factors does not match length of component dimensions") }
  dfs = rep(df,dim.df)
  n.dfs = length(dim.df)
  ind.dfs = c(0,sapply(1:length(dim.df),function(x){sum(dim.df[1:x])}),n)
  df.mat = matrix(0,n,n)
  for(j in 1:n.dfs){
    df.mat[(ind.dfs[j]+1):ind.dfs[(j+1)],(ind.dfs[j]+1):ind.dfs[(j+1)]] = (1-dfs[ind.dfs[(j+1)]]^k)/dfs[ind.dfs[(j+1)]]^k
  }
  return(df.mat)
}
#
H_t_k_r <- function(GG, t, k, r){
  n <- dim(GG)[1]
  I <- diag(n)
  for (s in (t+k-r):(t+k)) {
    I <- GG[,,s] %*% I   
  }
  return(I)
}

In [ ]:
# load_variables <- function(filename, dir_path) {
#   file_path <- file.path(dir_path, filename)
#   load(file_path)
#   cat("Variables loaded from:", file_path, "\n")
# }

# # load_variables("variables.RData", "/home/jaguir26/projects/notebooks/")
# file_path <- "/home/jaguir26/projects/notebooks/variables_50_M.RData"
# load(file_path)
# file_path <- "/home/jaguir26/projects/notebooks/variables_5_M.RData"
# load(file_path)
# file_path <- "/home/jaguir26/projects/notebooks/variables_95_M.RData"
# load(file_path)
# file_path <- "/home/jaguir26/projects/notebooks/variables_M.RData"
# load(file_path)

In [ ]:
# # Attempt to load the problematic files separately
# tryCatch({
#   load("/home/jaguir26/project1_ucsc_phd/variables_5_exAL.RData")
#   cat("Successfully loaded variables_5_exAL.RData\n")
# }, error = function(e) {
#   cat("Error loading variables_5_exAL.RData\n")
#   cat("Error message:", e$message, "\n")
# })

# tryCatch({
#   load("/home/jaguir26/project1_ucsc_phd/variables_95_exAL.RData")
#   cat("Successfully loaded variables_95_exAL.RData\n")
# }, error = function(e) {
#   cat("Error loading variables_95_exAL.RData\n")
#   cat("Error message:", e$message, "\n")
# })


In [ ]:
# # Define the file paths
# files <- c("/home/jaguir26/project1_ucsc_phd/variables_5_exAL.RData", 
#            "/home/jaguir26/project1_ucsc_phd/variables_50_exAL.RData",
#            "/home/jaguir26/project1_ucsc_phd/variables_95_exAL.RData",
#            "/home/jaguir26/project1_ucsc_phd/variables_20_exAL.RData",
#            "/home/jaguir26/project1_ucsc_phd/variables_35_exAL.RData",
#            "/home/jaguir26/project1_ucsc_phd/variables_65_exAL.RData",
#            "/home/jaguir26/project1_ucsc_phd/variables_80_exAL.RData")

# # Function to load files and handle errors
# load_file <- function(file_path) {
#   tryCatch({
#     load(file_path)
#     cat("Successfully loaded:", file_path, "\n")
#   }, error = function(e) {
#     cat("Error loading file:", file_path, "\n")
#     cat("Error message:", e$message, "\n")
#   })
# }

# # Load each file one by one
# for (file_path in files) {
#   load_file(file_path)
# }


In [ ]:
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_5_exAL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_50_exAL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_95_exAL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_20_exAL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_35_exAL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_65_exAL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_80_exAL.RData"
load(file_path)


In [ ]:
file_path <- "/home/jaguir26/project1_ucsc_phd/variables__NDLM.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables__NDLM_uni.RData"
load(file_path)

In [ ]:
n.samp <- 2000
cut <- 1

df_t    <- 1
df_s    <-  1
df_s67  <- 1
df.discrep <- 1
df_trans <- 1
df_covs <-  1
lambda <- 0.6
use_covariates = TRUE

parameters_path <- "/home/jaguir26/projects/Project/Input/exAL/parameters/parameters.txt"

# Check if the file exists
if (!file.exists(parameters_path)) {
  stop("The parameters file does not exist at the specified path: ", parameters_path)
}

lines <- readLines(parameters_path)

# Check if the lines variable is empty or not as expected
if (length(lines) == 0) {
  stop("No content found in the parameters file: ", parameters_path)
}

# Process each line and assign variables
for (line in lines) {
  # Remove leading and trailing whitespaces
  line <- trimws(line)
  
  # Skip empty lines and comments
  if (nchar(line) == 0 || grepl("^#", line)) next
  
  # Evaluate and assign
  eval(parse(text = line))
}

# Read and process ELI_lon data
ELI_lon <- read.csv("/home/jaguir26/projects/Project/Input/exAL/covariates/cov_1_ELI.csv")
merged_sst_data <- read.csv("/home/jaguir26/projects/Project/Input/exAL/covariates/cov_2_ONI.csv")
ELI_lon$time <- as.Date(ELI_lon$time)
adjustment_years <- 170
ELI_lon$time <- ELI_lon$time - years(adjustment_years)
#
# Read and process USGS data
data_usgs_r <- readNWISdv(siteNumbers = site_code[1], parameterCd = "00060", statCd = "00003")
San_Lorenzo_Daily_USGS_R <- data_usgs_r %>%
  mutate(timestamp = as.Date(Date),
         data0 = log(X_00060_00003 + 1)) %>%
  filter(timestamp > as.Date("1979-01-01"))
San_Lorenzo_Daily_USGS_R$time <- San_Lorenzo_Daily_USGS_R$timestamp
#
#
# SOIL
csv_file_path <- "/home/jaguir26/project1_ucsc_phd/climate_indices/soil_moisture_daily_avg.csv"
soil_moisture_data <- read.csv(csv_file_path)
soil_moisture_data$time <- as.Date(soil_moisture_data$time)
colnames(soil_moisture_data) <- c('time','soil')
#
# Merge datasets based on 'time'
merged_data <- merge(ELI_lon, merged_sst_data, by = "time")
merged_data <- merge(merged_data, San_Lorenzo_Daily_USGS_R, by = "time")
merged_data <- merged_data[, c(1:6, 10)]
colnames(merged_data) <- c("time", "eli", "nino12", "nino3", "nino34", "nino4", "flow")
merged_data$eli_smooth <- rollmean(merged_data$eli, k = KK, align = "right", fill = NA)
merged_data$oni <- rollmean(merged_data$nino34, k = KK, align = "right", fill = NA)
merged_data$eli_smooth[1:(KK-1)] <- merged_data$eli[1:(KK-1)]#
merged_data$oni[1:(KK-1)] <- merged_data$nino34[1:(KK-1)]
merged_data$flow_log <- log(merged_data$flow + 1)
#
# Adding soil
merged_data <- merge(merged_data, soil_moisture_data, by = "time")
#
# Standardize specified columns
standardize <- function(x) {
  (x - mean(x, na.rm = TRUE)) / sd(x, na.rm = TRUE)
}
columns_to_standardize <- c("eli_smooth", "oni", "flow_log", "soil")
merged_data[columns_to_standardize] <- lapply(merged_data[columns_to_standardize], standardize)
#
# Read streamflow data and merge with covariates
# data_path <- "/home/jaguir26/project1_ucsc_phd/combined_streamflow_data_cleaned.csv"
data_path <- "/home/jaguir26/project1_ucsc_phd/retros_2022-12-25.csv"
streamflow_data <- read_csv(data_path, show_col_types = FALSE)
timestamps <- as.Date(streamflow_data$Date)
time_series_matrix <- as.matrix(streamflow_data[, c('USGS', 'NWS3.0', 'GloFAS')])
Y_usgs <- data.frame(time = timestamps, time_series_matrix)
#
plot_data <- merge(merged_data, Y_usgs, by = "time")
ppt_data <- read.csv("PPT.csv")
ppt_data$time <- as.Date(ppt_data$time)
plot_data <- merge(plot_data, ppt_data, by = "time")
########################################################
# INDECES
file_path <- "/home/jaguir26/project1_ucsc_phd/climate_indices/combined_indices_daily_standardized.csv"
combined_indices <- read_csv(file_path, show_col_types = FALSE)
combined_indices['time']  <- as.Date(combined_indices$Date )
plot_data <- merge(plot_data, combined_indices, by = "time")
#
plot_data <- plot_data[cut:nrow(plot_data),]
x_names <- c('ppt', 'soil','Niño 3','NAO','Niño 1+2','WHWP','GMT','ONI','PNA','NOI','WP','Niño 3.4','Solar Flux','AMO','ESPI','TSA','Niño 4','TNA','SOI')
X <- as.matrix(plot_data[, x_names])
########################################################
# Set up Y and X matrices
Y <- t(as.matrix(plot_data[, c('USGS', 'NWS3.0', 'GloFAS')]))
# Y <- matrix(Y[,cut:dim(Y)[2]],nrow = dim(Y)[1])
TT <- dim(Y)[2]
J <- dim(Y)[1] - 1
#
# X <- as.matrix(plot_data[, c('oni', 'ppt', 'soil')])
# X <- as.matrix(plot_data[, c('oni')])
timestamps <- plot_data[, 'time']
#
# Model setup without covariates
m_yy <- mean(Y, na.rm = TRUE)
s_yy <- sd(Y, na.rm = TRUE)  
kk <- 0.1 * s_yy
trend.comp <- polytrendMod(1, m0 = m_yy, C0 = kk)
harm <- harmonics
seas.comp <- seasMod(p = 363.5854, h = harm, C0 = 0.5 * kk * diag(2 * length(harm)))
model <- combineMods(trend.comp, seas.comp)
p <- length(model$m0)
#
y <- Y
#
if (is.null(nrow(y))) {
  JJJ <- 1
  y <- array(y, c(JJJ, length(y)))
} else {
  JJJ <- nrow(Y)
  y <- array(y, c(JJJ, ncol(y)))
}
#
gam.init <- array(rep(0, JJJ), c(JJJ, 1))
sig.init <- array(rep(1, JJJ), c(JJJ, 1))
PriorSigma <- array(NA_real_, c(JJJ, 2))
PriorGamma <- array(NA_real_, c(JJJ, 3))
verbose <- TRUE
#
m0 <- c(model$m0, rep(0, J))
C0 <- bdiag(model$C0, 0.5 * kk * diag(J))
#
##########################################
##########################################
#
df_discrep <- rep(df.discrep, J)
#
df = c(df_t, df_s, df_s67); 
dim.df = c(1, 2*length(harm)-2, 2); 
k <- 5
##########################################
df.mat <- make_df_mat(df, dim.df, p)
df.mat.k <- make_df_mat_k(df, dim.df, p, k)
if (J <= 0) {
  ex.df.mat <- df.mat
  ex.df.mat.k <- df.mat.k
} else {
#   extra_df.mat <- make_df_mat(df_discrep, rep(1, J), J)
  extra_df.mat <- make_df_mat(c(df.discrep), c(J), J)
#   extra_df.mat.k <- make_df_mat_k(df_discrep, rep(1, J), J, k)
  extra_df.mat.k<- make_df_mat_k(c(df.discrep), c(J), J, k)
  ex.df.mat <- bdiag(df.mat, extra_df.mat)
  ex.df.mat.k <- bdiag(df.mat.k, extra_df.mat.k)
}
#
model_simp <- model
df_simp <- df
dim.df_simp <- dim.df
model_simp$GG <- array(model_simp$GG, c(p, p, TT))
model_simp$FF <- array(model_simp$FF, c(p, 1, TT))
model$m0 <- m0
model$C0 <- C0
if (use_covariates) {
  # Adding covariates
  px <- dim(X)[2]
  ppx <- px + 1

  F1 <- matrix(model$FF, p, J + 1)
  F2 <- cbind(rep(0, J), diag(J))
  Fx <- rbind(rep(1, J + 1), matrix(0, nrow = px, ncol = J + 1))
  FF <- array(rbind(F1, F2, Fx), c(p + J + ppx, 1 + J, TT))

  Gx <- as.matrix(bdiag(lambda, diag(px)))
  Gx <- array(rep(Gx, TT), dim = c(ppx, ppx, TT))
  Gx[1, 2:ppx, ] <- as.matrix(t(X))

  GG <- array(bdiag(model$GG, diag(J)), c(p + J, p + J, TT))
  model$GG <- GG
  GG_dim <- dim(model$GG)[1]
  new_dim <- GG_dim + ppx
  GGG <- array(0, dim = c(new_dim, new_dim, TT))
  GGG[1:GG_dim, 1:GG_dim, ] <- model$GG
  GGG[(GG_dim + 1):new_dim, (GG_dim + 1):new_dim, ] <- Gx

  model$FF <- FF
  model$GG <- GGG

  # df.covs <- rep(df_covs, ppx)
  # extra_df.mat <- make_df_mat(df.covs, rep(1, ppx), ppx)
  # extra_df.mat.k <- make_df_mat_k(df.covs, rep(1, ppx), ppx, k)

  extra_df.mat <- make_df_mat(c(df_trans,df_covs), c(1,px), ppx)
  extra_df.mat.k <- make_df_mat_k(c(df_trans,df_covs), c(1,px), ppx, k)

  ex.df.mat <- bdiag(ex.df.mat, extra_df.mat)
  ex.df.mat.k <- bdiag(ex.df.mat.k, extra_df.mat.k)

  model$m0 <- c(model$m0, rep(0, ppx))
  model$C0 <- bdiag(model$C0, 0.5 * kk * diag(ppx))

} else {
  # Without covariates
  GG <- array(bdiag(model$GG, diag(J)), c(p + J, p + J, TT))
  model$GG <- GG
  F1 <- matrix(model$FF, p, J + 1)
  F2 <- cbind(rep(0, J), diag(J))
  FF <- array(rbind(F1, F2), c(p + J, 1 + J, TT))
  model$FF <- FF
  ppx <- 0
}
#
FF <- model$FF
GG <- model$GG
#



In [ ]:
idxxx <- (TT-1000):TT
plot.ts(Y[1,idxxx], col = 'gray', lwd = 2)
lines((new.theta.out__NDLM_uni$exps[1,idxxx]), col = 'orange')
lines((new.theta.out_50_exAL$exps[1,idxxx]), col = 'red')
lines((new.theta.out_5_exAL$exps[1,idxxx]), col = 'green')
lines((new.theta.out_95_exAL$exps[1,idxxx]), col = 'blue')
s <- 0
# lines(s+new.theta.out_50_exAL$sm[2,idxxx], col = 'blue')
# lines(s+new.theta.out_50_exAL$sm[4,idxxx], col = 'blue')
# lines(s+new.theta.out_50_exAL$sm[6,idxxx], col = 'blue')
# plot.ts(s+new.theta.out_50_exAL$sm[8,idxxx], col = 'blue')
# plot.ts(s+new.theta.out_50_exAL$sm[9,idxxx], col = 'blue')
# lines(s+new.theta.out_50_exAL$sm[10,idxxx], col = 'blue')

# plot.ts(Y[1,idxxx], col = 'gray', lwd = 2)
# lines((new.theta.out__NDLM_uni$sm[1,idxxx]), col = 'darkred')
# lines((new.theta.out__NDLM_uni$sm[2,idxxx]), col = 'darkred')
# lines((new.theta.out__NDLM_uni$sm[4,idxxx]), col = 'darkred')
# lines((new.theta.out__NDLM_uni$sm[6,idxxx]), col = 'darkred')

# lines((new.theta.out__NDLM_uni$sm[9,idxxx]), col = 'darkred')
# lines((new.theta.out__NDLM_uni$sm[10,idxxx]), col = 'darkred')


In [ ]:
# Set up the plotting area for a 6x2 matrix layout
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 3, 0))

# Define the colors for each time series
colors <- c("forestgreen", "darkorange", "darkblue")

# Plot each time series with the specified colors
ts.plot(t(seq.sigma_50_exAL), col = colors, main = "Sigma 50th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_5_exAL), col = colors, main = "Sigma 05th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_95_exAL), col = colors, main = "Sigma 95th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.gamma_50_exAL), col = colors, main = "Gamma 50th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.gamma_5_exAL), col = colors, main = "Gamma 05th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.gamma_95_exAL), col = colors, main = "Gamma 95th", xlab = "Iteration", ylab = "Gamma")

# Add a common legend to the plot
# Placing the legend at the top of the first column (adjust `oma` and `mar` for space)
mtext("Green - USGS, Orange - GLOFAS, Blue - NWS", side = 3, outer = TRUE, line = 0, cex = 0.8)

par(mfrow = c(2, 4), mar = c(4, 4, 2, 1), oma = c(0, 0, 3, 0))
# Plot each time series with the specified colors
ts.plot(t(seq.sigma_20_exAL), col = colors, main = "Sigma 20th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_35_exAL), col = colors, main = "Sigma 35th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_65_exAL), col = colors, main = "Sigma 65th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_80_exAL), col = colors, main = "Sigma 80th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.gamma_20_exAL), col = colors, main = "Gamma 20th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.gamma_35_exAL), col = colors, main = "Gamma 35th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.gamma_65_exAL), col = colors, main = "Gamma 65th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.sigma_80_exAL), col = colors, main = "Sigma 80th", xlab = "Iteration", ylab = "Sigma")

# Add a common legend to the plot
# Placing the legend at the top of the first column (adjust `oma` and `mar` for space)
mtext("Green - USGS, Orange - GLOFAS, Blue - NWS", side = 3, outer = TRUE, line = 0, cex = 0.8)

# Resetting the plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
par(mfrow = c(2, 2), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

a <- c(seq.elbo_50_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL50", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_5_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL05", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_95_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL95", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo__NDLM)
a[1:3]=NaN
plot.ts(a, main = "ELBO -NDLM", xlab = "Iteration", ylab = "ELBO")

par(mfrow = c(2, 2), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))
a <- c(seq.elbo_20_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL20", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_35_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL35", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_65_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL65", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_80_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL80", xlab = "Iteration", ylab = "ELBO")

In [ ]:
compute_xb_corrected <- function(samp_theta, FF) {
  # Determine dimensions
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  # Allocate space for the result: 3x12009x2000
  xb <- array(NA, dim = c(3, n_time, n_sim))
  
  # Loop over time points
  for (t in 1:n_time) {
    FF_t <- FF[,,t]  # 9x3 matrix for time t
    
    # Extract all simulations for time t across all components: 9x2000 matrix
    theta_t_s <- samp_theta[, t, ]
    
    # Perform matrix multiplication
    xb[, t, ] <- t(FF_t) %*% theta_t_s  # Results in a 3x2000 matrix
  }
  
  return(xb)
}


In [ ]:
xb_50_corrected <- compute_xb_corrected(samp.theta_50_exAL, FF)
xb_05_corrected <- compute_xb_corrected(samp.theta_5_exAL, FF)
xb_95_corrected <- compute_xb_corrected(samp.theta_95_exAL, FF)
quantiles_xb_50_corrected <- apply(xb_50_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_05_corrected <- apply(xb_05_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_95_corrected <- apply(xb_95_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))


In [ ]:
xb_20_corrected <- compute_xb_corrected(samp.theta_20_exAL, FF)
xb_35_corrected <- compute_xb_corrected(samp.theta_35_exAL, FF)
xb_65_corrected <- compute_xb_corrected(samp.theta_65_exAL, FF)
xb_80_corrected <- compute_xb_corrected(samp.theta_80_exAL, FF)
quantiles_xb_20_corrected <- apply(xb_20_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_35_corrected <- apply(xb_35_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_65_corrected <- apply(xb_65_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_80_corrected <- apply(xb_80_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))


In [ ]:
# Define a function to calculate quantiles for each dataset
calculate_quantiles <- function(data, variable_name, quantile_name, source_name) {
  quantile_values <- quantile(data, probs = c(0.025, 0.5, 0.975))
  tibble(
    variable = variable_name,
    source = source_name,
    quantile = quantile_name,
    quantile_025 = quantile_values["2.5%"],
    median = quantile_values["50%"],
    quantile_975 = quantile_values["97.5%"]
  )
}

# List of datasets and their metadata
data_sets <- list(
  gamma_50_M = list(data = samp.gamma_50_exAL, quantile = "50th", variable = "Gamma"),
  gamma_95_M = list(data = samp.gamma_95_exAL, quantile = "95th", variable = "Gamma"),
  gamma_05_M = list(data = samp.gamma_5_exAL, quantile = "05th", variable = "Gamma"),
  gamma_20_M = list(data = samp.gamma_20_exAL, quantile = "20th", variable = "Gamma"),
  gamma_35_M = list(data = samp.gamma_35_exAL, quantile = "35th", variable = "Gamma"),
  gamma_65_M = list(data = samp.gamma_65_exAL, quantile = "65th", variable = "Gamma"),
  gamma_80_M = list(data = samp.gamma_80_exAL, quantile = "80th", variable = "Gamma"),
  sigma_50_M = list(data = samp.sigma_50_exAL, quantile = "50th", variable = "Sigma"),
  sigma_95_M = list(data = samp.sigma_95_exAL, quantile = "95th", variable = "Sigma"),
  sigma_05_M = list(data = samp.sigma_5_exAL, quantile = "05th", variable = "Sigma"),
  sigma_20_M = list(data = samp.sigma_20_exAL, quantile = "20th", variable = "Sigma"),
  sigma_35_M = list(data = samp.sigma_35_exAL, quantile = "35th", variable = "Sigma"),
  sigma_65_M = list(data = samp.sigma_65_exAL, quantile = "65th", variable = "Sigma"),
  sigma_80_M = list(data = samp.sigma_80_exAL, quantile = "80th", variable = "Sigma")
)

# # List of datasets and their metadata
# data_sets <- list(
#   gamma_50_M = list(data = samp.gamma_50_exAL, quantile = "50th", variable = "Gamma"),
#   gamma_95_M = list(data = samp.gamma_95_exAL, quantile = "95th", variable = "Gamma"),
#   gamma_05_M = list(data = samp.gamma_5_exAL, quantile = "05th", variable = "Gamma"),
#   sigma_50_M = list(data = samp.sigma_50_exAL, quantile = "50th", variable = "Sigma"),
#   sigma_95_M = list(data = samp.sigma_95_exAL, quantile = "95th", variable = "Sigma"),
#   sigma_05_M = list(data = samp.sigma_5_exAL, quantile = "05th", variable = "Sigma")
# )

# Calculate quantiles for each dataset and source
all_quantiles <- bind_rows(
  lapply(data_sets, function(item) {
    bind_rows(
      calculate_quantiles(item$data[, 1], item$variable, item$quantile, "USGS"),
      calculate_quantiles(item$data[, 2], item$variable, item$quantile, "GLOFAS"),
      calculate_quantiles(item$data[, 3], item$variable, item$quantile, "NWS")
    )
  })
)

# Print the complete table of quantiles
print(all_quantiles, n = Inf)


In [ ]:
prepare_quantile_data <- function(v_d) {
  v_d_transposed <- aperm(v_d, c(3, 1, 2))
  q_d_transposed <- apply(v_d_transposed, 2:3, function(x) quantile(x, probs = c(0.975, 0.5, 0.025)))
  q_d <- aperm(q_d_transposed, c(2, 3, 1))
  return(q_d)
}


In [ ]:
q_d_50 <- prepare_quantile_data(samp.theta_50_exAL)
q_d_05 <- prepare_quantile_data(samp.theta_5_exAL)
q_d_95 <- prepare_quantile_data(samp.theta_95_exAL)


In [ ]:
q_d_20 <- prepare_quantile_data(samp.theta_20_exAL)
q_d_35 <- prepare_quantile_data(samp.theta_35_exAL)
q_d_65 <- prepare_quantile_data(samp.theta_65_exAL)
q_d_80 <- prepare_quantile_data(samp.theta_80_exAL)

In [ ]:
dates_ts_usgs <- timestamps

In [ ]:
# Function to plot with quantiles and dates on x-axis
# plot_quantile_component <- function(q_d_50, q_d_05, q_d_95,q_d_20,q_d_35,q_d_65,q_d_80, Y, idx, component, main_label, num_ticks) {
plot_quantile_component <- function(q_d_50, q_d_05, q_d_95, Y, idx, component, main_label, num_ticks) {

  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d_50[component, idx, ], q_d_05[component, idx, ], q_d_95[component, idx, ])) * 2
  ylims[2] <- min(1.3, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.3, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

  lines(idx, q_d_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)
  
  lines(idx, q_d_20[component, idx, 2], col = "purple", lwd = 1)
  lines(idx, q_d_35[component, idx, 2], col = "purple", lwd = 1)
  lines(idx, q_d_65[component, idx, 2], col = "purple", lwd = 1)
  lines(idx, q_d_80[component, idx, 2], col = "purple", lwd = 1)

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


In [ ]:
# Set up plotting window for a 2x3 matrix layout
par(mfrow = c(2, 1), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

# Index range and plotting
idx <- ceiling(TT/9):TT
components <- c(1, 2, 4, 6, 8, 9, 10:29)
component_labels <- c("Trend Component", "First Harmonic", "Second Harmonic", "(1/6.83) Harmonic", 
                      "Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")

for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  # plot_quantile_component(q_d_50,q_d_05, q_d_95, q_d_20, q_d_35, q_d_65, q_d_80, Y, idx, components[i], component_labels[i], num_ticks = 8)
  plot_quantile_component(q_d_50,q_d_05, q_d_95, Y, idx, components[i], component_labels[i], num_ticks = 8)

}

# Add a common legend or note at the bottom
mtext("Legend: Forest Green - 50_M, Dark Red - 05_M, Dark Blue - 95_M", side = 1, outer = TRUE, line = 2, cex = 0.8)

# Reset plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
prepare_quantile_data <- function(v_d) {
  v_d_transposed <- aperm(v_d, c(3, 1, 2))
  q_d_transposed <- apply(v_d_transposed, 2:3, function(x) quantile(x, probs = c(0.975, 0.5, 0.025)))
  q_d <- aperm(q_d_transposed, c(2, 3, 1))
  return(q_d)
}

# Apply the function to each dataset
q_d_NDLM <- prepare_quantile_data(samp.theta__NDLM)


In [ ]:
# Function to plot with quantiles and dates on x-axis
plot_quantile_component_NDLM <- function(q_d, Y, idx, component, main_label, num_ticks) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d[component, idx, ])) * 3
  ylims[2] <- min(2, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.5, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

 # Adding NDLM quantiles
  lines(idx, q_d[component, idx, 1], col = "darkorange", lwd = 0.5, lty=2)  # Lower bound
  lines(idx, q_d[component, idx, 3], col = "darkorange", lwd = 0.5, lty=2)  # Upper bound
  lines(idx, q_d[component, idx, 2], col = "darkorange", lwd = 1)            # Median line

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


In [ ]:
# Set up plotting window for a 2x3 matrix layout
par(mfrow = c(2, 1), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

# Index range and plotting
idx <- 1:TT
components <- c(1, 2, 4, 6, 8, 9)
component_labels <- c("Trend Component", "First Harmonic", "Second Harmonic", "(1/6.83) Harmonic", 
                      "Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")

for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component_NDLM (q_d_NDLM, Y, idx, components[i], component_labels[i], num_ticks = 8)
}

# Reset plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
# Function to plot with quantiles and dates on x-axis
plot_quantile_component_all <- function(q_d, q_d_50, q_d_05, q_d_95, Y, idx, component, main_label, num_ticks) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d[component, idx, ], q_d_50[component, idx, ], q_d_05[component, idx, ], q_d_95[component, idx, ])) * 2
  ylims[2] <- min(1.3, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.3, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

  lines(idx, q_d_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d[component, idx, 1], col = "darkorange", lwd = 0.5, lty=2)  # Lower bound
  lines(idx, q_d[component, idx, 3], col = "darkorange", lwd = 0.5, lty=2)  # Upper bound
  lines(idx, q_d[component, idx, 2], col = "darkorange", lwd = 1)            # Median line

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


In [ ]:
# Set up plotting window for a 2x3 matrix layout
par(mfrow = c(2, 1), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

# Index range and plotting
idx <- 1:TT
components <- c(1, 2, 4, 6, 8, 9)
component_labels <- c("Trend Component", "First Harmonic", "Second Harmonic", "(1/6.83) Harmonic", 
                      "Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")

for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component_all(q_d_NDLM, q_d_50, q_d_05, q_d_95, Y, idx, components[i], component_labels[i], num_ticks = 8)
}

# Add a common legend or note at the bottom
mtext("Legend: Orange: Mean (NDLM), Forest Green - 50_M, Dark Red - 05_M, Dark Blue - 95_M", side = 1, outer = TRUE, line = 2, cex = 0.8)

# Reset plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
compute_xb_corrected <- function(samp_theta, FF, sig.samp, pp) {
  # Determine dimensions
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  # Allocate space for the result: 3x12009x2000
  xb <- array(NA, dim = c(3, n_time, n_sim))
  
  # Loop over time points
  for (t in 1:n_time) {
    FF_t <- FF[,,t]  # 9x3 matrix for time t
    
    # Extract all simulations for time t across all components: 9x2000 matrix
    theta_t_s <- samp_theta[, t, ]
    
    # Perform matrix multiplication
    xb[, t, ] <- t(FF_t) %*% theta_t_s + t(sqrt(sig.samp))*qnorm(pp) 
  }
  
  return(xb)
}


In [ ]:

# # Apply the corrected function to each samp.theta matrix
# xb_M_50 <- compute_xb_corrected(samp.theta_M, FF, samp.sigma_M, 0.5)
# xb_M_05 <- compute_xb_corrected(samp.theta_M, FF, samp.sigma_M, 0.05)
# xb_M_95 <- compute_xb_corrected(samp.theta_M, FF, samp.sigma_M, 0.95)

# # Compute quantiles for xb_50 as an example (apply this logic to xb_05 and xb_95 as needed)
# quantiles_xb_M_50 <- apply(xb_M_50, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
# quantiles_xb_M_05 <- apply(xb_M_05, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
# quantiles_xb_M_95 <- apply(xb_M_95, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))




In [ ]:
compute_xb_corrected <- function(samp_theta, FF, sig.samp, pp) {
  # Determine dimensions
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  # Allocate space for the result: 3x12009x2000
  xb <- array(NA, dim = c(3, n_time, n_sim))
  
  # Loop over time points
  for (t in 1:n_time) {
    FF_t <- FF[,,t]  # 9x3 matrix for time t
    
    # Extract all simulations for time t across all components: 9x2000 matrix
    theta_t_s <- samp_theta[, t, ]
    
    # Perform matrix multiplication
    xb[, t, ] <- t(FF_t) %*% theta_t_s + t(sqrt(sig.samp))*qnorm(pp) 
  }
  
  return(xb)
}

# Apply the corrected function to each samp.theta matrix
xb_M_50 <- compute_xb_corrected(samp.theta__NDLM, FF, samp.sigma__NDLM, 0.5)
xb_M_05 <- compute_xb_corrected(samp.theta__NDLM, FF, samp.sigma__NDLM, 0.05)
xb_M_95 <- compute_xb_corrected(samp.theta__NDLM, FF, samp.sigma__NDLM, 0.95)

# Compute quantiles for xb_50 as an example (apply this logic to xb_05 and xb_95 as needed)
quantiles_xb_M_50 <- apply(xb_M_50, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_M_05 <- apply(xb_M_05, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_M_95 <- apply(xb_M_95, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))


In [ ]:
# idx <- 15810:16032
idx <- (TT-500):TT

# Function to create nice date labels
create_date_labels <- function(idx, num_labels = 10) {
    selected_dates <- dates_ts_usgs[idx]  # assuming dates_ts_usgs is an array of dates corresponding to idx
    tick_positions <- pretty(idx, num_labels)
    tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")
    return(list(tick_positions = tick_positions, tick_labels = tick_labels))
}

# Setting up the plotting window
par(mfrow = c(2, 1), mar = c(3.2, 4, 2, 1) + 0.1, oma = c(4, 0, 0, 0))

# Calculate date labels outside of the plotting function for consistency
date_info <- create_date_labels(idx)

# First plot
plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="NDLM", xaxt="n")

lines(idx, quantiles_xb_M_50[2, 1, idx], col="lightgreen", lwd=1) 
lines(idx, quantiles_xb_M_50[3, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_50[1, 1, idx], col="lightgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_M_05[2, 1, idx], col="pink", lwd=1) 
lines(idx, quantiles_xb_M_05[3, 1, idx], col="pink", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_05[1, 1, idx], col="pink", lwd=0.5, lty=2)
lines(idx, quantiles_xb_M_95[2, 1, idx], col="lightblue", lwd=1) 
lines(idx, quantiles_xb_M_95[3, 1, idx], col="lightblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_95[1, 1, idx], col="lightblue", lwd=0.5, lty=2) 
axis(1, at = date_info$tick_positions, labels = FALSE)

# Adding rotated text labels
text(x = date_info$tick_positions, y = min(par("usr")[3], -1.5), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Second plot
plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="exAL", xaxt="n")

lines(idx, quantiles_xb_50_corrected[2, 1, idx], col="forestgreen", lwd=1) 
lines(idx, quantiles_xb_50_corrected[1, 1, idx], col="forestgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_50_corrected[3, 1, idx], col="forestgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_05_corrected[2, 1, idx], col="darkred", lwd=1) 
lines(idx, quantiles_xb_05_corrected[1, 1, idx], col="darkred", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_05_corrected[3, 1, idx], col="darkred", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_95_corrected[2, 1, idx], col="darkblue", lwd=1) 
lines(idx, quantiles_xb_95_corrected[1, 1, idx], col="darkblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_95_corrected[3, 1, idx], col="darkblue", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_20_corrected[2, 1, idx], col="purple", lwd=0.5) 
lines(idx, quantiles_xb_35_corrected[2, 1, idx], col="purple", lwd=0.5) 
lines(idx, quantiles_xb_65_corrected[2, 1, idx], col="purple", lwd=0.5) 
lines(idx, quantiles_xb_80_corrected[2, 1, idx], col="purple", lwd=0.5) 

axis(1, at = date_info$tick_positions, labels = FALSE)

# Adding rotated text labels for the second plot
text(x = date_info$tick_positions, y = min(par("usr")[3], -1.5), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Reset the plotting parameters to default after plotting is done
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
idx <- (TT-1000):TT
# Function to create nice date labels
create_date_labels <- function(idx, num_labels = 8) {
    selected_dates <- dates_ts_usgs[idx]  # assuming dates_ts_usgs is an array of dates corresponding to idx
    tick_positions <- pretty(idx, num_labels)
    tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")
    return(list(tick_positions = tick_positions, tick_labels = tick_labels))
}

par(mfrow = c(3, 1), mar = c(3.2, 4, 2, 1) + 0.1, oma = c(4, 0, 0, 0))
date_info <- create_date_labels(idx)

plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="NDLM vs exAL: 95th", xaxt="n")

lines(idx, quantiles_xb_M_95[2, 1, idx], col="lightblue", lwd=1) 
lines(idx, quantiles_xb_M_95[3, 1, idx], col="lightblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_95[1, 1, idx], col="lightblue", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_95_corrected[2, 1, idx], col="darkblue", lwd=1) 
lines(idx, quantiles_xb_95_corrected[1, 1, idx], col="darkblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_95_corrected[3, 1, idx], col="darkblue", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="NDLM vs exAL: 05th", xaxt="n")

lines(idx, quantiles_xb_M_05[2, 1, idx], col="pink", lwd=1) 
lines(idx, quantiles_xb_M_05[3, 1, idx], col="pink", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_05[1, 1, idx], col="pink", lwd=0.5, lty=2)
lines(idx, quantiles_xb_05_corrected[2, 1, idx], col="darkred", lwd=1) 
lines(idx, quantiles_xb_05_corrected[1, 1, idx], col="darkred", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_05_corrected[3, 1, idx], col="darkred", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)


plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="NDLM vs exA: 50th", xaxt="n")

lines(idx, quantiles_xb_M_50[2, 1, idx], col="lightgreen", lwd=1) 
lines(idx, quantiles_xb_M_50[3, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_50[1, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_50_corrected[2, 1, idx], col="forestgreen", lwd=1) 
lines(idx, quantiles_xb_50_corrected[1, 1, idx], col="forestgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_50_corrected[3, 1, idx], col="forestgreen", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Reset the plotting parameters to default after plotting is done
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
# v <- xb_95_corrected
# boolean_array <- sweep(v[1,,], MARGIN = 1, STATS = Y[1,], FUN = ">")

# Metrics 

### cv metric?

## I. ELBO 
### $\text{ELBO}(\theta, \phi) = \mathbb{E}_{q_\phi(z)}[\log p_\theta(x, z) - \log q_\phi(z)]$

In [ ]:
elbo_values <- data.frame(
  Model = c("NDLM", "exAL-0.5", "exAL-0.05", "exAL-0.95"),
  ELBO = c(seq.elbo__NDLM[length(seq.elbo__NDLM)],
          seq.elbo_50_exAL[dim(seq.elbo_50_exAL)[2]],
           seq.elbo_5_exAL[dim(seq.elbo_5_exAL)[2]],
           seq.elbo_95_exAL[dim(seq.elbo_95_exAL)[2]])
)

print(elbo_values)


In [ ]:
post_mean_50_exAL <- apply(samp.post.pred_50_exAL, c(1, 2), mean)
post_mean_5_exAL <- apply(samp.post.pred_5_exAL, c(1, 2), mean)
post_mean_95_exAL <- apply(samp.post.pred_95_exAL, c(1, 2), mean)

In [ ]:
check_loss <- function(y, mu, tau) {
  if (!is.numeric(tau) || tau < 0 || tau > 1) {
    stop("tau must be a numeric value between 0 and 1.")
  }
  errors <- y - mu
  loss <- ifelse(errors >= 0, 
                 tau * errors,    
                 (1 - tau) * -errors)  
return(loss)
}

In [ ]:
# post_ndlm <- function(samp_theta, FF, sig.samp) {
#   n_time <- dim(samp_theta)[2]
#   n_sim <- dim(samp_theta)[3]
#   y_post <- array(NA, dim = c(3, n_time, n_sim))
  
#   for (t in 1:n_time) {
#     FF_t <- FF[,,t]  
#     theta_t_s <- samp_theta[, t, ]
#     y_post[, t, ] <- t(FF_t) %*% theta_t_s + t(sqrt(sig.samp))*matrix(rnorm(3*n_sim),3,n_sim)  
#   }
#   return(y_post)
# }

# y_M_post <- post_ndlm(samp.theta_M, FF, samp.sigma_M)

In [ ]:
y_post_mean_ndlm <- apply(samp.post.pred__NDLM, c(1, 2), mean)
y_post_qs_ndlm <- apply(samp.post.pred__NDLM, c(1, 2), function(x) quantile(x, probs = c(0.05, 0.5, 0.95)))

In [ ]:
y_post_qs_ndlm_95  <- y_post_mean_ndlm[1,] + mean(sqrt(samp.sigma__NDLM[1,]))*pnorm(0.95)
y_post_qs_ndlm_50  <- y_post_mean_ndlm[1,] + mean(sqrt(samp.sigma__NDLM[1,]))*pnorm(0)
y_post_qs_ndlm_05  <- y_post_mean_ndlm[1,] + mean(sqrt(samp.sigma__NDLM[1,]))*pnorm(0.05)

## II. Generalized PPLC 
###  $\sum_{t=1}^T \rho_{p_0}(y_{t}^{\text{usgs}} - \mathbb{E}_{q_\phi(z)}[y_{t, \text{new}}^{\text{usgs}}])$


In [ ]:
library(knitr)

# Data preparation and rolling mean calculation
data_50_M <- data.frame(Time = 1:length(Y[1,]), Loss = check_loss(Y[1,], post_mean_50_exAL[1,], 0.50))
data_05_M <- data.frame(Time = 1:length(Y[1,]), Loss = check_loss(Y[1,], post_mean_5_exAL[1,], 0.05))
data_95_M <- data.frame(Time = 1:length(Y[1,]), Loss = check_loss(Y[1,], post_mean_95_exAL[1,], 0.95))
data_ndlm <- data.frame(Time = 1:length(Y[1,]), Loss = check_loss(Y[1,], y_post_mean_ndlm[1,], 0.50))

data_50_M$RollingMean <- rollapply(data_50_M$Loss, 360, mean, partial = TRUE, fill = NA, align = "right")
data_05_M$RollingMean <- rollapply(data_05_M$Loss, 360, mean, partial = TRUE, fill = NA, align = "right")
data_95_M$RollingMean <- rollapply(data_95_M$Loss, 360, mean, partial = TRUE, fill = NA, align = "right")
data_ndlm$RollingMean <- rollapply(data_ndlm$Loss, 360, mean, partial = TRUE, fill = NA, align = "right")

data_50_M$model <- "exAL-0.5"
data_05_M$model <- "exAL-0.05"
data_95_M$model <- "exAL-0.95"
data_ndlm$model <- "NDLM"

# Combine all data into one data frame
all_data <- rbind(data_ndlm, data_50_M, data_05_M, data_95_M)

# Plotting with improved colors and cleaned x-axis
p <- ggplot(all_data, aes(x = Time)) +
  geom_line(aes(y = Loss, colour = "Actual Loss"), linewidth = 1.2) +
  geom_line(aes(y = RollingMean, colour = "Rolling Mean (360)"), linewidth = 0.8) +
  facet_wrap(~ model, nrow = 1, scales = "free_x") +  # One row of plots with free scales
  labs(title = "Check Loss Function Across Models",
       x = "",
       y = "Loss",
       colour = "Legend") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),  # Hide x-axis text
    axis.ticks.x = element_blank()  # Hide x-axis ticks
  ) +
  scale_colour_manual(values = c("Actual Loss" = "#1f77b4", "Rolling Mean (360)" = "#ff7f0e"))

# Display the plot
print(p)


mean_losses <- data.frame(
  Model = c("NDLM", "exAL-0.5", "exAL-0.05", "exAL-0.95"),
  MeanLoss = c(mean(data_ndlm$Loss), mean(data_50_M$Loss), mean(data_05_M$Loss), mean(data_95_M$Loss))
)

# Format the mean losses into a clean table
kable(mean_losses, format = "markdown", caption = "Mean Check Losses Across Models")

In [ ]:
score <- subset(all_data, model==unique(all_data[,4])[1])[,2]
plot.ts(cumsum(score)/TT, col='darkorange', ylim = c(0,0.35))
score <- subset(all_data, model==unique(all_data[,4])[2])[,2]
lines(cumsum(score)/TT, col='darkgreen')
score <- subset(all_data, model==unique(all_data[,4])[3])[,2]
lines(cumsum(score)/TT, col='darkred')
score <- subset(all_data, model==unique(all_data[,4])[4])[,2]
lines(cumsum(score)/TT, col='darkblue')

## III. Generalized MSE
###  $\sum_{t=1}^T \mathbb{E}_{q_\phi(z)}[\rho_{p_0}(y_{t}^{\text{usgs}} - y_{t, \text{new}}^{\text{usgs}})]$

In [ ]:
# Function to process each model's predictive performance
process_model <- function(y_vector, post_pred, tau, model_name) {
  n_samples <- dim(post_pred)[3]  # Number of samples from the third dimension
  y_matrix <- matrix(rep(y_vector, each = n_samples), nrow = length(y_vector), ncol = n_samples)
  
  # Compute differences and apply loss function
  diff_matrix <- y_matrix - post_pred[1, , ]
  loss_matrix <- ifelse(diff_matrix >= 0, tau * diff_matrix, (1 - tau) * -diff_matrix)
  mean_per_t <- rowMeans(loss_matrix)
  
  data_frame <- data.frame(
    Time = 1:length(mean_per_t),
    Loss = mean_per_t,
    RollingMean = rollapply(mean_per_t, 360, mean, partial = TRUE, fill = NA, align = "right"),
    Model = model_name
  )
  
  overall_mean_loss = mean(mean_per_t)
  return(list(data_frame = data_frame, mean_loss = overall_mean_loss))
}

# List of models with parameters and specific names
models <- list(
  list(post_pred = samp.post.pred__NDLM, tau = 0.50, name = "NDLM"),
  list(post_pred = samp.post.pred_50_exAL, tau = 0.5, name = "exAL-0.5"),
  list(post_pred = samp.post.pred_5_exAL, tau = 0.05, name = "exAL-0.05"),
  list(post_pred = samp.post.pred_95_exAL, tau = 0.95, name = "exAL-0.95")
)

results <- lapply(models, function(m) process_model(Y[1, ], m$post_pred, m$tau, m$name))

# Combine results into a single data frame for plotting
all_data <- do.call(rbind, lapply(results, `[[`, "data_frame"))

# Plotting using ggplot2 with the specific formatting as requested
p <- ggplot(all_data, aes(x = Time)) +
  geom_line(aes(y = Loss, colour = "Actual Loss"), linewidth = 1.2) +
  geom_line(aes(y = RollingMean, colour = "Rolling Mean (360)"), linewidth = 0.8) +
  facet_wrap(~ Model, nrow = 1, scales = "free_x") +
  labs(title = "Predictive Performance Across Models",
       x = "",
       y = "Loss",
       colour = "Legend") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),  # Hide x-axis text
    axis.ticks.x = element_blank()  # Hide x-axis ticks
  ) +
  scale_colour_manual(values = c("Actual Loss" = "#1f77b4", "Rolling Mean (360)" = "#ff7f0e"))

# Display the plot
print(p)

# Displaying the table of mean losses
# Assuming results are generated as follows:
results <- lapply(models, function(m) process_model(Y[1, ], m$post_pred, m$tau, m$name))
# Extracting mean losses to a data frame
mean_losses <- do.call(rbind, lapply(results, function(x) {
  data.frame(Model = x$data_frame$Model[1], MeanLoss = x$mean_loss, stringsAsFactors = FALSE)
}))
# Formatting and printing the table using knitr::kable
kable(mean_losses, format = "markdown", caption = "Mean Losses Across Models")


In [ ]:
score <- subset(all_data, Model==unique(all_data[,4])[1])[,2]
plot.ts(cumsum(score)/TT, col='darkorange', ylim = c(0,0.6))
score <- subset(all_data, Model==unique(all_data[,4])[2])[,2]
lines(cumsum(score)/TT, col='darkgreen')
score <- subset(all_data, Model==unique(all_data[,4])[3])[,2]
lines(cumsum(score)/TT, col='darkred')
score <- subset(all_data, Model==unique(all_data[,4])[4])[,2]
lines(cumsum(score)/TT, col='darkblue')

## IV. KL divergence
### $KL(h, \phi) = \int_{-\infty}^{\infty} h(x) \log \left(\frac{h(x)}{\phi(x)}\right) dx$


where:
- $ h(x) $ "standardize" one-step-ahead forecast.
- $ \phi(x) $ is the standard normal density.




In [ ]:
# Function to process and plot KL Divergence and forecast errors
process_errors <- function(errors, model_name) {
  s <-0
  n <- 100
  for(k in 1:n){
  T_e <- length(errors)
  ref <- stats::rnorm(T_e)  # Reference normal distribution
  kl_divergence <- mean(FNN::KL.divergence(ref, errors))
  s <- s + kl_divergence/n
  }
  # Creating a dataframe for ggplot
  data_frame <- data.frame(
    Time = 1:T_e,
    NormalizedErrors = stats::pnorm(errors),
    RollingMean = rollapply(stats::pnorm(errors), 90, mean, partial = TRUE, fill = NA, align = "right"),
    Model = model_name
  )

  return(list(data_frame = data_frame, kl_divergence = kl_divergence))
}

# List of models with parameters and specific names
models <- list(
  list(errors = new.theta.out__NDLM$standard_forecast_errors[1,], name = "NDLM Errors"),
  list(errors = new.theta.out_50_exAL$standard_forecast_errors[1,], name = "exAL-0.5 errors"),
  list(errors = new.theta.out_5_exAL$standard_forecast_errors[1,], name = "exAL-0.05 errors"),
  list(errors = new.theta.out_95_exAL$standard_forecast_errors[1,], name = "exAL-0.95 errors")
)


results <- lapply(models, function(m) process_errors(m$errors, m$name))

# Combine results into a single data frame for plotting
all_data <- do.call(rbind, lapply(results, `[[`, "data_frame"))

# Plotting using ggplot2 with the specific formatting as requested
p <- ggplot(all_data, aes(x = Time)) +
  geom_line(aes(y = NormalizedErrors, colour = "Actual Errors"), linewidth = 1.2) +
  geom_line(aes(y = RollingMean, colour = "Rolling Mean (90)"), linewidth = 0.8) +
  facet_wrap(~ Model, nrow = 1, scales = "free_x") +
  labs(title = "KL Divergence and Forecast Errors Across Models",
       x = "",
       y = "Normalized Errors",
       colour = "Legend") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),  # Hide x-axis text
    axis.ticks.x = element_blank()  # Hide x-axis ticks
  ) +
  scale_colour_manual(values = c("Actual Errors" = "#1f77b4", "Rolling Mean (90)" = "#ff7f0e"))

# Display the plot
print(p)

# Displaying KL Divergences

# Assuming results were generated as follows:
results <- lapply(models, function(m) process_errors(m$errors, m$name))
kl_divergences <- do.call(rbind, lapply(results, function(x) data.frame(Model = x$data_frame$Model[1], KL_Divergence = x$kl_divergence)))

# Format the KL Divergence data into a clean table
kable(kl_divergences, format = "markdown", caption = "Standardize Forecast Error Models")


In [ ]:
score <- subset(all_data, Model==unique(all_data[,4])[1])[,2]
plot.ts(cumsum(score)/TT, col='darkorange', ylim = c(0,0.6))
score <- subset(all_data, Model==unique(all_data[,4])[2])[,2]
lines(cumsum(score)/TT, col='darkgreen')
score <- subset(all_data, Model==unique(all_data[,4])[3])[,2]
lines(cumsum(score)/TT, col='darkred')
score <- subset(all_data, Model==unique(all_data[,4])[4])[,2]
lines(cumsum(score)/TT, col='darkblue')

Why not the "smoothed errors"??

## V. Quantile Check: 
### $\frac{1}{T} \sum_{t=1}^{T}I(y^{usgs}_t \leq F_t'\theta^*_t) \approx p_0$


In [ ]:
# Define a function to process each model and calculate statistics
process_model_stats <- function(v, y, target_prob) {
  boolean_array <- sweep(v[1, , ], MARGIN = 1, STATS = y, FUN = ">")
  boolean_means <- apply(boolean_array, 2, mean)
  c(SD = sd(boolean_means), MAD = mean(abs(boolean_means - target_prob)))
}

# Assume 'xb_50_corrected', 'xb_05_corrected', 'xb_95_corrected',
# 'xb_M_50', 'xb_M_05', 'xb_M_95' are loaded and available

# Apply the function to each model
stats_50_corrected <- process_model_stats(xb_50_corrected, Y[1,], 0.5)
stats_05_corrected <- process_model_stats(xb_05_corrected, Y[1,], 0.05)
stats_95_corrected <- process_model_stats(xb_95_corrected, Y[1,], 0.95)

stats_M_50 <- process_model_stats(xb_M_50, Y[1,], 0.5)
stats_M_05 <- process_model_stats(xb_M_05, Y[1,], 0.05)
stats_M_95 <- process_model_stats(xb_M_95, Y[1,], 0.95)

# Combine all stats into a data frame, arranging to compare similar models
results <- data.frame(
  Model = c("exAL-0.5", "NDLM-0.5", "exAL-0.05", "NDLM-0.05",
            "exAL-0.95", "NDLM-0.95"),
  SD = c(stats_50_corrected["SD"], stats_M_50["SD"], stats_05_corrected["SD"], stats_M_05["SD"],
         stats_95_corrected["SD"], stats_M_95["SD"]),
  MAD = c(stats_50_corrected["MAD"], stats_M_50["MAD"], stats_05_corrected["MAD"], stats_M_05["MAD"],
          stats_95_corrected["MAD"], stats_M_95["MAD"])
)


# Print the table using kable
kable(results, format = "markdown", caption = "Statistical Measures for exAL and NDLM Models, Paired for Comparison")



In [ ]:
# Function to analyze and plot histograms with better visibility and aesthetics
analyze_and_plot <- function(v, dataset_name) {
  # Perform the boolean comparison with the global Y[1,]
  boolean_array <- sweep(v[1,,], MARGIN = 1, STATS = Y[1,], FUN = ">")
  boolean_means <- apply(boolean_array, 2, mean)
  
  # Plotting histogram with adjusted breaks and color
  hist(boolean_means, 
       main = paste("", dataset_name), 
       xlab = "Proportion", 
       breaks = 30, 
       col = rgb(0.1, 0.2, 0.5, 0.8),  # Adjusted alpha for better visibility
       border = "darkblue", 
       las = 1,
       ylim = c(0, max(table(cut(boolean_means, breaks = 50))) + 5))  # Adjusted ylim for clarity
  
  # Calculating percentiles
  p2.5 <- quantile(boolean_means, 0.025)
  p97.5 <- quantile(boolean_means, 0.975)

  # Adding vertical lines for percentiles
  abline(v = p2.5, col = "darkred", lwd = 0.5, lty = 2)
  abline(v = p97.5, col = "darkred", lwd = 0.5, lty = 2)
  
  # Adding annotations for percentiles
  text(x = p2.5, y = par("usr")[4] * 0.95, labels = paste("", round(p2.5, 4)), 
       srt = 0, col = "darkred", cex = 0.8, adj = 1)
  text(x = p97.5, y = par("usr")[4] * 0.95, labels = paste("", round(p97.5, 4)), 
       srt = 0, col = "darkred", cex = 0.8, adj = 1)
}

# Assuming Y and datasets are properly defined and loaded
# Setup the plot layout
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1))  # Adjusted layout to accommodate 6 plots

# Analyze and plot for each dataset
analyze_and_plot(xb_05_corrected, "exAL-0.05")
analyze_and_plot(xb_50_corrected, "exAL-0.5")
analyze_and_plot(xb_95_corrected, "exAL-0.95")
analyze_and_plot(xb_M_05, "NDLM-0.05")
analyze_and_plot(xb_M_50, "NDLM-0.5")
analyze_and_plot(xb_M_95, "NDLM-0.95")


In [ ]:
# Function to analyze and plot rolling statistics with enhanced aesthetics
analyze_and_plot_rolling <- function(v, dataset_name, roll_m, y_limits, h_line) {
  # Perform the boolean comparison with the global Y[1,]
  boolean_array <- sweep(v[1,,], MARGIN = 1, STATS = Y[1,], FUN = ">")
  
  # Convert boolean results to a time series object
  ts_data <- zoo(apply(boolean_array, 2, mean), order.by = 1:dim(boolean_array)[2])
  
  # Calculate rolling statistics
  roll_mean <- rollapply(ts_data, roll_m, mean, fill = NA, align = "right")
  roll_p2.5 <- rollapply(ts_data, roll_m, quantile, probs = 0.025, fill = NA, align = "right")
  roll_p97.5 <- rollapply(ts_data, roll_m, quantile, probs = 0.975, fill = NA, align = "right")
  
  # Plotting the time series with refined aesthetics
  plot(roll_mean, type = "l", col = "darkblue", xlab = "Time", ylab = "Probability", main = paste("Rolling Mean -", dataset_name),
       ylim = y_limits, lwd = 2)
  lines(roll_p2.5, col = "darkred", lty = 1, lwd = 1)
  lines(roll_p97.5, col = "darkred", lty = 1, lwd = 1)
  
  # Add horizontal line at the specified position
  abline(h = h_line, col = "darkorange", lwd = 2, lty = 2)
  
  # Legend for clarity
  legend("topright", legend = c("Mean", "95% Bands"), 
         col = c("darkblue", "darkred"), lty = 1, lwd = 2, cex = 0.8)
}

# Adjust layout to accommodate 6 plots
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))  # Adjust outer margin for overall plot titles or notes

# Analyze and plot for each dataset with updated aesthetic settings
analyze_and_plot_rolling(xb_95_corrected, "xb_95_corrected", 90, c(0.9, 1), 0.95)
analyze_and_plot_rolling(xb_50_corrected, "xb_50_corrected", 90, c(0.4, 0.6), 0.5)
analyze_and_plot_rolling(xb_05_corrected, "xb_05_corrected", 90, c(0.0, 1), 0.05)

analyze_and_plot_rolling(xb_M_95, "xb_95_ndlm", 90, c(0.9, 1), 0.95)
analyze_and_plot_rolling(xb_M_50, "xb_50_ndlm", 90, c(0.4, 0.6), 0.5)
analyze_and_plot_rolling(xb_M_05, "xb_05_ndlm", 90, c(0.0, 0.1), 0.05)


## VI. Posterior Predictive Quantile Check: 
### $\frac{1}{T} \sum_{t=1}^T I(y_{t}^* \leq F_t'\theta^*_t) \approx p_0$ 
### $ \mathbb{E}[I(y_{t}^{new} \leq F_t'\theta_t) | y^{usgs}_{1:T}] \approx p_0$ 


In [ ]:
# Define a function to process each model comparison and calculate statistics
process_comparison_stats <- function(predicted, corrected, target_prob) {
  # Compute the difference and the boolean array where differences are less than zero
  boolean_array <- (predicted - corrected) < 0
  boolean_means <- apply(boolean_array, 2, mean) - target_prob
  
  # Calculate SD and MAD
  SD = sd(boolean_means)
  MAD = mean(abs(boolean_means))
  
  return(c(SD = SD, MAD = MAD))
}

# Assume 'samp.post.pred_xx_M', 'xb_xx_corrected', 'y_M_post', 'xb_M_xx' are loaded and available

# Apply the function to each model and compute required statistics
stats_50_corr <- process_comparison_stats(samp.post.pred_50_exAL[1,,], xb_50_corrected[1,,], 0.5)
stats_05_corr <- process_comparison_stats(samp.post.pred_5_exAL[1,,], xb_05_corrected[1,,], 0.05)
stats_95_corr <- process_comparison_stats(samp.post.pred_95_exAL[1,,], xb_95_corrected[1,,], 0.95)

stats_M_50 <- process_comparison_stats(samp.post.pred__NDLM[1,,], xb_M_50[1,,], 0.5)
stats_M_05 <- process_comparison_stats(samp.post.pred__NDLM[1,,], xb_M_05[1,,], 0.05)
stats_M_95 <- process_comparison_stats(samp.post.pred__NDLM[1,,], xb_M_95[1,,], 0.95)

# Combine all stats into a data frame
results <- data.frame(
  Model = c("exAL-0.5", "NDLM-0.5", "exAL-0.05", "NDLM-0.05", "exAL-0.95", "NDLM-0.95"),
  SD = c(stats_50_corr["SD"], stats_M_50["SD"], stats_05_corr["SD"], stats_M_05["SD"], stats_95_corr["SD"], stats_M_95["SD"]),
  MAD = c(stats_50_corr["MAD"], stats_M_50["MAD"], stats_05_corr["MAD"], stats_M_05["MAD"], stats_95_corr["MAD"], stats_M_95["MAD"])
)

kable(results, format = "markdown", caption = "Statistical Measures for Predicted vs. Corrected Models")


In [ ]:
# Function to analyze and plot histograms using posterior predictive samples
analyze_and_plot <- function(post_pred, corrected, dataset_name) {
  # Perform the boolean comparison directly with the posterior predictive samples
  boolean_array <- (post_pred[1,,] - corrected[1,,]) < 0
  boolean_means <- apply(boolean_array, 2, mean)
  
  # Plotting histogram with adjusted aesthetics
  hist(boolean_means,
       main = paste("", dataset_name),
       xlab = "Proportion",
       breaks = 30,
       col = rgb(0.1, 0.2, 0.5, 0.8),
       border = "darkblue",
       las = 1,
       ylim = c(0, max(table(cut(boolean_means, breaks = 50))) + 5))
  
  # Calculating percentiles
  p2.5 <- quantile(boolean_means, 0.025)
  p97.5 <- quantile(boolean_means, 0.975)

  # Adding vertical lines for percentiles
  abline(v = p2.5, col = "darkred", lwd = 0.5, lty = 2)
  abline(v = p97.5, col = "darkred", lwd = 0.5, lty = 2)
  
  # Adding annotations for percentiles
  text(x = p2.5, y = par("usr")[4] * 0.95, labels = paste(round(p2.5, 4)), 
       srt = 0, col = "darkred", cex = 0.8, adj = 1)
  text(x = p97.5, y = par("usr")[4] * 0.95, labels = paste(round(p97.5, 4)), 
       srt = 0, col = "darkred", cex = 0.8, adj = 1)
}

# Setup the plot layout
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1))  # Adjusted layout to accommodate 6 plots

# Assuming that post_pred and corrected datasets are properly defined and loaded
# Analyze and plot for each dataset
analyze_and_plot(samp.post.pred_5_exAL, xb_05_corrected, "exAL-0.05")
analyze_and_plot(samp.post.pred_50_exAL, xb_50_corrected, "exAL-0.50")
analyze_and_plot(samp.post.pred_95_exAL, xb_95_corrected, "exAL-0.95")
analyze_and_plot(samp.post.pred__NDLM, xb_M_05, "NDLM-0.05")
analyze_and_plot(samp.post.pred__NDLM, xb_M_50, "NDLM-0.5")
analyze_and_plot(samp.post.pred__NDLM, xb_M_95, "NDLM-0.95")

In [ ]:
# Function to analyze and plot rolling statistics using posterior predictive samples
analyze_and_plot_rolling <- function(post_pred, corrected, dataset_name, roll_m, y_limits, h_line) {
  # Perform the boolean comparison directly with the posterior predictive samples
  boolean_array <- (post_pred[1,,] - corrected[1,,]) < 0
  
  # Convert boolean results to a time series object
  ts_data <- zoo(apply(boolean_array, 2, mean), order.by = 1:dim(boolean_array)[2])
  
  # Calculate rolling statistics
  roll_mean <- rollapply(ts_data, roll_m, mean, fill = NA, align = "right")
  roll_p2.5 <- rollapply(ts_data, roll_m, quantile, probs = 0.025, fill = NA, align = "right")
  roll_p97.5 <- rollapply(ts_data, roll_m, quantile, probs = 0.975, fill = NA, align = "right")
  
  # Plotting the time series with refined aesthetics
  plot(roll_mean, type = "l", col = "darkblue", xlab = "Time", ylab = "Probability", 
       main = paste("Rolling Mean -", dataset_name), ylim = y_limits, lwd = 2)
  lines(roll_p2.5, col = "darkred", lty = 1, lwd = 1)
  lines(roll_p97.5, col = "darkred", lty = 1, lwd = 1)
  
  # Add horizontal line at the specified position
  abline(h = h_line, col = "darkorange", lwd = 2, lty = 2)
  
  # Legend for clarity
  legend("topright", legend = c("Mean", "95% Band"), 
         col = c("darkblue", "darkred"), lty = 1, lwd = 1, cex = 0.8)
}

# Adjust layout to accommodate 6 plots
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))

# Analyze and plot for each dataset with updated aesthetic settings
analyze_and_plot_rolling(samp.post.pred_50_exAL, xb_50_corrected, "exAL-0.5", 90, c(0.45, 0.55), 0.5)
analyze_and_plot_rolling(samp.post.pred_5_exAL, xb_05_corrected, "exAL-0.05", 90, c(0, 0.1), 0.05)
analyze_and_plot_rolling(samp.post.pred_95_exAL, xb_95_corrected, "exAL-0.95", 90, c(0.9, 1), 0.95)

analyze_and_plot_rolling(samp.post.pred__NDLM, xb_M_50, "NDLM-0.5", 90, c(0.45, 0.55), 0.5)
analyze_and_plot_rolling(samp.post.pred__NDLM, xb_M_05, "NDLM-0.05", 90, c(0, 0.1), 0.05)
analyze_and_plot_rolling(samp.post.pred__NDLM, xb_M_95, "NDLM-0.95", 90, c(0.9, 1), 0.95)


## NWS Stamdardize log(Forecasts+1)

CHECK IF IM COMPUTING RIGHT ELBOS!

In [ ]:
idx <- (TT-500):TT
date_info <- create_date_labels(idx)
par(mfrow = c(2, 2), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))

new.cpp <- samp.post.pred_50_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-50"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.975), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkorange')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.025), col='darkblue')

new.cpp <- samp.post.pred_5_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-05"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.975), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.025), col='darkorange')

new.cpp <- samp.post.pred_95_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-95"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.975), col='darkorange')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.025), col='darkblue')

new.cpp <- samp.post.pred__NDLM
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("NDLM"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.975), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, mean), col='darkorange')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.025), col='darkblue')


In [ ]:
new.cpp1 <- samp.post.pred_5_exAL
new.cpp2 <- samp.post.pred_20_exAL
new.cpp3 <- samp.post.pred_35_exAL
new.cpp4 <- samp.post.pred_50_exAL
new.cpp5 <- samp.post.pred_65_exAL
new.cpp6 <- samp.post.pred_80_exAL
new.cpp7 <- samp.post.pred_95_exAL

par(mfrow = c(1, 1), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))
plot.ts(Y[1,idx], col='black', ylim = c(-3,4), main = paste("Quantiles of the Posterior Predictive"), xaxt="n")
# lines(apply(new.cpp1[1, idx, ], 1, quantile, probs = 0.05), col='darkred')
# lines(apply(new.cpp2[1, idx, ], 1, quantile, probs = 0.20), col='purple')
# lines(apply(new.cpp3[1, idx, ], 1, quantile, probs = 0.35), col='purple')
# lines(apply(new.cpp4[1, idx, ], 1, quantile, probs = 0.50), col='forestgreen')
# lines(apply(new.cpp5[1, idx, ], 1, quantile, probs = 0.65), col='purple')
# lines(apply(new.cpp6[1, idx, ], 1, quantile, probs = 0.80), col='purple')
# lines(apply(new.cpp7[1, idx, ], 1, quantile, probs = 0.95), col='darkblue')

x1 <- apply(new.cpp1[1, idx, ], 1, quantile, probs = 0.05)
x2 <- apply(new.cpp2[1, idx, ], 1, quantile, probs = 0.20)
x3 <- apply(new.cpp3[1, idx, ], 1, quantile, probs = 0.35)
x4 <- apply(new.cpp4[1, idx, ], 1, quantile, probs = 0.50)
x5 <- apply(new.cpp5[1, idx, ], 1, quantile, probs = 0.65)
x6 <- apply(new.cpp6[1, idx, ], 1, quantile, probs = 0.80)
x7 <- apply(new.cpp7[1, idx, ], 1, quantile, probs = 0.95)
lines( (x1+x2+x3+x4+x5+x6+x7)/7, col='pink')
# lines( (x1+x4+x7)/7, col='gold')
new.cpp0 <- samp.post.pred__NDLM
x0 <- apply(new.cpp0[1, idx, ], 1, quantile, probs = 0.5)
lines( x0, col='purple')

x1 <- apply(new.cpp1[1, idx, ], 1, quantile, probs = 0.5)
x2 <- apply(new.cpp2[1, idx, ], 1, quantile, probs = 0.5)
x3 <- apply(new.cpp3[1, idx, ], 1, quantile, probs = 0.5)
x4 <- apply(new.cpp4[1, idx, ], 1, quantile, probs = 0.5)
x5 <- apply(new.cpp5[1, idx, ], 1, quantile, probs = 0.5)
x6 <- apply(new.cpp6[1, idx, ], 1, quantile, probs = 0.5)
x7 <- apply(new.cpp7[1, idx, ], 1, quantile, probs = 0.5)
lines( (x1+x2+x3+x4+x5+x6+x7)/7, col='gold')
# lines( (x1+x4+x7)/3, col='gold')


In [ ]:
# MEDIAN POST PREDICTIVE for each model
idx <- (TT-500):TT
date_info <- create_date_labels(idx)
par(mfrow = c(2, 2), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))

new.cpp <- samp.post.pred_50_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-50"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='forestgreen')

new.cpp <- samp.post.pred_5_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-05"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkred')

new.cpp <- samp.post.pred_95_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-95"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkblue')

new.cpp <- samp.post.pred__NDLM
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("NDLM"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkorange')

## AV and SL

In [ ]:
load_variables <- function(filename, dir_path) {
  file_path <- file.path(dir_path, filename)
  load(file_path)
  cat("Variables loaded from:", file_path, "\n")
}

file_path <- "/home/jaguir26/project1_ucsc_phd/variables_50_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_5_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_95_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_20_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_35_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_65_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_80_AV.RData"
load(file_path)

file_path <- "/home/jaguir26/project1_ucsc_phd/variables_50_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_5_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_95_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_20_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_35_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_65_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_80_SL.RData"
load(file_path)


In [ ]:
# Set up the plotting area for a 6x2 matrix layout
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 3, 0))

# Plot each time series with the specified colors
y1 <- seq.sigma_50_SL
y2 <- seq.sigma_50_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Sigma 50th", xlab = "Iteration", ylab = "Sigma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Sigma 50th", xlab = "Iteration", ylab = "Sigma", col = 'purple')

y1 <- seq.sigma_5_SL
y2 <- seq.sigma_5_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Sigma 05th", xlab = "Iteration", ylab = "Sigma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Sigma 05th", xlab = "Iteration", ylab = "Sigma", col = 'purple')

y1 <- seq.sigma_95_SL
y2 <- seq.sigma_95_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Sigma 95th", xlab = "Iteration", ylab = "Sigma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Sigma 95th", xlab = "Iteration", ylab = "Sigma", col = 'purple')

# Plot each time series with the specified colors
y1 <- seq.gamma_50_SL
y2 <- seq.gamma_50_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Gamma 50th", xlab = "Iteration", ylab = "Gamma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Gamma 50th", xlab = "Iteration", ylab = "Gamma", col = 'purple')

y1 <- seq.gamma_5_SL
y2 <- seq.gamma_5_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Gamma 05th", xlab = "Iteration", ylab = "Gamma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Gamma 05th", xlab = "Iteration", ylab = "Gamma", col = 'purple')

y1 <- seq.gamma_95_SL
y2 <- seq.gamma_95_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Gamma 95th", xlab = "Iteration", ylab = "Gamma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Gamma 95th", xlab = "Iteration", ylab = "Gamma", col = 'purple')

# Resetting the plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
# Set up the plotting area for a 6x2 matrix layout
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 3, 0))

na_lim <- 5

y1 <- seq.elbo_50_SL
y2 <- seq.elbo_50_AV
y1[1:na_lim]=NaN
y2[1:na_lim]=NaN
YLIM <- c(min(y1,y2, na.rm = TRUE), max(y1,y2, na.rm = TRUE))
plot.ts(t(y1), main = "ELBO 50th", xlab = "Iteration", ylab = "ELBO", col = 'green', ylim = YLIM )
lines(t(y2), main = "ELBO 50th", xlab = "Iteration", ylab = "ELBO", col = 'purple')

y1 <- seq.elbo_5_SL
y2 <- seq.elbo_5_AV
y1[1:na_lim]=NaN
y2[1:na_lim]=NaN
YLIM <- c(min(y1,y2, na.rm = TRUE), max(y1,y2, na.rm = TRUE))
plot.ts(t(y1), main = "ELBO 05th", xlab = "Iteration", ylab = "ELBO", col = 'green', ylim = YLIM )
lines(t(y2), main = "ELBO 05th", xlab = "Iteration", ylab = "ELBO", col = 'purple')

y1 <- seq.elbo_95_SL
y2 <- seq.elbo_95_AV
y1[1:na_lim]=NaN
y2[1:na_lim]=NaN
YLIM <- c(min(y1,y2, na.rm = TRUE), max(y1,y2, na.rm = TRUE))
plot.ts(t(y1), main = "ELBO 95th", xlab = "Iteration", ylab = "ELBO", col = 'green', ylim = YLIM )
lines(t(y2), main = "ELBO 95th", xlab = "Iteration", ylab = "ELBO", col = 'purple')

par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
n <- dim(FF)[1]-J

In [ ]:
compute_xb_corrected <- function(samp_theta, FF) {
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  xb <- array(NA, dim = c(J+1, n_time, n_sim))
  
  for (t in 1:n_time) {
    FF_t <- FF[,,t]
    theta_t_s <- samp_theta[, t, ]
    xb[, t, ] <- t(FF_t) %*% theta_t_s
  }
  return(xb)
}

# Apply the corrected function to each samp.theta matrix
xb_50_AV_corrected <- compute_xb_corrected(samp.theta_50_AV, FF[1:n,,])
xb_05_AV_corrected <- compute_xb_corrected(samp.theta_5_AV, FF[1:n,,])
xb_95_AV_corrected <- compute_xb_corrected(samp.theta_95_AV, FF[1:n,,])
xb_20_AV_corrected <- compute_xb_corrected(samp.theta_20_AV, FF[1:n,,])
xb_35_AV_corrected <- compute_xb_corrected(samp.theta_35_AV, FF[1:n,,])
xb_65_AV_corrected <- compute_xb_corrected(samp.theta_65_AV, FF[1:n,,])
xb_80_AV_corrected <- compute_xb_corrected(samp.theta_80_AV, FF[1:n,,])

xb_50_SL_corrected <- compute_xb_corrected(samp.theta_50_AV, FF[1:n,,])
xb_05_SL_corrected <- compute_xb_corrected(samp.theta_5_AV, FF[1:n,,])
xb_95_SL_corrected <- compute_xb_corrected(samp.theta_95_AV, FF[1:n,,])
xb_20_SL_corrected <- compute_xb_corrected(samp.theta_20_AV, FF[1:n,,])
# xb_35_SL_corrected <- compute_xb_corrected(samp.theta_35_AV, FF[1:n,,])
xb_65_SL_corrected <- compute_xb_corrected(samp.theta_65_AV, FF[1:n,,])
xb_80_SL_corrected <- compute_xb_corrected(samp.theta_80_AV, FF[1:n,,])


In [ ]:
# Compute quantiles
compute_quantiles <- function(xb) {
  apply(xb, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
}

quantiles_xb_AV_50_corrected <- compute_quantiles(xb_50_AV_corrected)
quantiles_xb_AV_05_corrected <- compute_quantiles(xb_05_AV_corrected)
quantiles_xb_AV_95_corrected <- compute_quantiles(xb_95_AV_corrected)
quantiles_xb_AV_20_corrected <- compute_quantiles(xb_20_AV_corrected)
quantiles_xb_AV_35_corrected <- compute_quantiles(xb_35_AV_corrected)
quantiles_xb_AV_65_corrected <- compute_quantiles(xb_65_AV_corrected)
quantiles_xb_AV_80_corrected <- compute_quantiles(xb_80_AV_corrected)


quantiles_xb_SL_50_corrected <- compute_quantiles(xb_50_SL_corrected)
quantiles_xb_SL_05_corrected <- compute_quantiles(xb_05_SL_corrected)
quantiles_xb_SL_95_corrected <- compute_quantiles(xb_95_SL_corrected)
quantiles_xb_SL_20_corrected <- compute_quantiles(xb_20_SL_corrected)
# quantiles_xb_SL_35_corrected <- compute_quantiles(xb_35_SL_corrected)
quantiles_xb_SL_65_corrected <- compute_quantiles(xb_65_SL_corrected)
quantiles_xb_SL_80_corrected <- compute_quantiles(xb_80_SL_corrected)

In [ ]:
# Define data list
data_list <- list(
  gamma_50_AV = list(data = samp.gamma_50_AV, quantile = "50th", variable = "Gamma", source = "AV"),
  gamma_95_AV = list(data = samp.gamma_95_AV, quantile = "95th", variable = "Gamma", source = "AV"),
  gamma_05_AV = list(data = samp.gamma_5_AV, quantile = "05th", variable = "Gamma", source = "AV"),
  sigma_50_AV = list(data = samp.sigma_50_AV, quantile = "50th", variable = "Sigma", source = "AV"),
  sigma_95_AV = list(data = samp.sigma_95_AV, quantile = "95th", variable = "Sigma", source = "AV"),
  sigma_05_AV = list(data = samp.sigma_5_AV, quantile = "05th", variable = "Sigma", source = "AV"),
  gamma_50_SL = list(data = samp.gamma_50_SL, quantile = "50th", variable = "Gamma", source = "SL"),
  gamma_95_SL = list(data = samp.gamma_95_SL, quantile = "95th", variable = "Gamma", source = "SL"),
  gamma_05_SL = list(data = samp.gamma_5_SL, quantile = "05th", variable = "Gamma", source = "SL"),
  sigma_50_SL = list(data = samp.sigma_50_SL, quantile = "50th", variable = "Sigma", source = "SL"),
  sigma_95_SL = list(data = samp.sigma_95_SL, quantile = "95th", variable = "Sigma", source = "SL"),
  sigma_05_SL = list(data = samp.sigma_5_SL, quantile = "05th", variable = "Sigma", source = "SL")
)

# Function to calculate quantiles
calculate_quantiles <- function(data, variable_name, quantile_name, source_name) {
  quantile_values <- quantile(data, probs = c(0.025, 0.5, 0.975))
  tibble(
    variable = variable_name,
    source = source_name,
    quantile = quantile_name,
    quantile_025 = quantile_values["2.5%"],
    median = quantile_values["50%"],
    quantile_975 = quantile_values["97.5%"]
  )
}

# Calculate quantiles for each dataset
all_quantiles <- bind_rows(
  lapply(data_list, function(item) {
    calculate_quantiles(item$data, item$variable, item$quantile, item$source)
  })
)

# Print the complete table of quantiles
print(all_quantiles)


In [ ]:
prepare_quantile_data <- function(v_d) {
  v_d_transposed <- aperm(v_d, c(3, 1, 2))
  q_d_transposed <- apply(v_d_transposed, 2:3, function(x) quantile(x, probs = c(0.975, 0.5, 0.025)))
  q_d <- aperm(q_d_transposed, c(2, 3, 1))
  return(q_d)
}

# Apply the function to each dataset
q_d_50_AV <- prepare_quantile_data(samp.theta_50_AV)
q_d_05_AV <- prepare_quantile_data(samp.theta_5_AV)
q_d_95_AV <- prepare_quantile_data(samp.theta_95_AV)
q_d_50_SL <- prepare_quantile_data(samp.theta_50_SL)
q_d_05_SL <- prepare_quantile_data(samp.theta_5_SL)
q_d_95_SL <- prepare_quantile_data(samp.theta_95_SL)

In [ ]:
dates_ts_usgs <- timestamps

# Function to plot with quantiles and dates on x-axis
plot_quantile_component <- function(q_d_50, q_d_05, q_d_95, Y, idx, component, main_label, num_ticks) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d_50[component, idx, ], q_d_05[component, idx, ], q_d_95[component, idx, ])) * 2
  ylims[2] <- min(1.3, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.3, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

  lines(idx, q_d_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


# Function to plot with quantiles and dates on x-axis
plot_quantile_component_all <- function(q_d1_50, q_d1_05, q_d1_95, q_d2_50, q_d2_05, q_d2_95, Y, idx, component, main_label, num_ticks) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d1_50[component, idx, ], q_d1_05[component, idx, ], q_d1_95[component, idx, ])) * 2
  ylims[2] <- min(1.3, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.3, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

  lines(idx, q_d1_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d1_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d1_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d1_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d1_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d1_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d1_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d1_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d1_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)

  lines(idx, q_d2_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d2_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d2_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d2_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d2_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d2_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d2_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d2_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d2_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


In [ ]:
# Set up plotting window for a 2x3 matrix layout
par(mfrow = c(2, 1), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

# Index range and plotting
idx <- 1:TT
components <- c(1, 2, 4, 6)
component_labels <- c("SL - Trend Component", "SL - First Harmonic", "SL - Second Harmonic", "SL - (1/6.83) Harmonic")
for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component(q_d_50_SL,q_d_05_SL, q_d_95_SL, Y, idx, components[i], component_labels[i], num_ticks = 8)
}
# Add a common legend or note at the bottom
mtext("Legend: Forest Green - 50th, Dark Red - 05th, Dark Blue - 95th", side = 1, outer = TRUE, line = 2, cex = 0.8)

component_labels <- c("AV - Trend Component", "AV - First Harmonic", "AV - Second Harmonic", "AV - (1/6.83) Harmonic")
for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component(q_d_50_AV,q_d_05_AV, q_d_95_AV, Y, idx, components[i], component_labels[i], num_ticks = 8)
}
# Add a common legend or note at the bottom
mtext("Legend: Forest Green - 50th, Dark Red - 05th, Dark Blue - 95th", side = 1, outer = TRUE, line = 2, cex = 0.8)

component_labels <- c("SL vs AV - Trend Component", "SL vs AV - First Harmonic", "SL vs AV - Second Harmonic", "SL vs AV - (1/6.83) Harmonic")
for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component_all(q_d_50_SL,q_d_05_SL, q_d_95_SL, q_d_50_AV,q_d_05_AV, q_d_95_AV, Y, idx, components[i], component_labels[i], num_ticks = 8)
}
# Add a common legend or note at the bottom
mtext("Legend: Forest Green - 50th, Dark Red - 05th, Dark Blue - 95th", side = 1, outer = TRUE, line = 2, cex = 0.8)

# Reset plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
# idx <- 15810:16032
idx <- (TT-500):TT

# Function to create nice date labels
create_date_labels <- function(idx, num_labels = 10) {
    selected_dates <- dates_ts_usgs[idx]  # assuming dates_ts_usgs is an array of dates corresponding to idx
    tick_positions <- pretty(idx, num_labels)
    tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")
    return(list(tick_positions = tick_positions, tick_labels = tick_labels))
}

In [ ]:
# Setting up the plotting window
par(mfrow = c(2, 1), mar = c(3.2, 4, 2, 1) + 0.1, oma = c(4, 0, 0, 0))

# Calculate date labels outside of the plotting function for consistency
date_info <- create_date_labels(idx)

# First plot
plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="AV", xaxt="n")


lines(idx, quantiles_xb_AV_50_corrected[2, 1, idx], col="lightgreen", lwd=1) 
lines(idx, quantiles_xb_AV_50_corrected[3, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_50_corrected[1, 1, idx], col="lightgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_AV_05_corrected[2, 1, idx], col="pink", lwd=1) 
lines(idx, quantiles_xb_AV_05_corrected[3, 1, idx], col="pink", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_05_corrected[1, 1, idx], col="pink", lwd=0.5, lty=2)
lines(idx, quantiles_xb_AV_95_corrected[2, 1, idx], col="lightblue", lwd=1) 
lines(idx, quantiles_xb_AV_95_corrected[3, 1, idx], col="lightblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_95_corrected[1, 1, idx], col="lightblue", lwd=0.5, lty=2) 
axis(1, at = date_info$tick_positions, labels = FALSE)

# Adding rotated text labels
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Second plot
plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="SL", xaxt="n")

lines(idx, quantiles_xb_SL_50_corrected[2, 1, idx], col="darkgreen", lwd=1) 
lines(idx, quantiles_xb_SL_50_corrected[3, 1, idx], col="darkgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_50_corrected[1, 1, idx], col="darkgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_SL_05_corrected[2, 1, idx], col="darkred", lwd=1) 
lines(idx, quantiles_xb_SL_05_corrected[3, 1, idx], col="darkred", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_05_corrected[1, 1, idx], col="darkred", lwd=0.5, lty=2)
lines(idx, quantiles_xb_SL_95_corrected[2, 1, idx], col="darkblue", lwd=1) 
lines(idx, quantiles_xb_SL_95_corrected[3, 1, idx], col="darkblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_95_corrected[1, 1, idx], col="darkblue", lwd=0.5, lty=2) 
axis(1, at = date_info$tick_positions, labels = FALSE)

# Adding rotated text labels for the second plot
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)



In [ ]:

date_info <- create_date_labels(idx)

plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="AV vs SL: 95th", xaxt="n")

lines(idx, quantiles_xb_AV_95_corrected[2, 1, idx], col="lightblue", lwd=1) 
lines(idx, quantiles_xb_AV_95_corrected[3, 1, idx], col="lightblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_95_corrected[1, 1, idx], col="lightblue", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_SL_95_corrected[2, 1, idx], col="darkblue", lwd=1) 
lines(idx, quantiles_xb_SL_95_corrected[3, 1, idx], col="darkblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_95_corrected[1, 1, idx], col="darkblue", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)


plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="AV vs SL: 05th", xaxt="n")

lines(idx, quantiles_xb_AV_05_corrected[2, 1, idx], col="pink", lwd=1) 
lines(idx, quantiles_xb_AV_05_corrected[3, 1, idx], col="pink", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_05_corrected[1, 1, idx], col="pink", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_SL_05_corrected[2, 1, idx], col="darkred", lwd=1) 
lines(idx, quantiles_xb_SL_05_corrected[3, 1, idx], col="darkred", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_05_corrected[1, 1, idx], col="darkred", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)


plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="AV vs SL: 50th", xaxt="n")

lines(idx, quantiles_xb_AV_50_corrected[2, 1, idx], col="lightgreen", lwd=1) 
lines(idx, quantiles_xb_AV_50_corrected[3, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_50_corrected[1, 1, idx], col="lightgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_SL_50_corrected[2, 1, idx], col="darkgreen", lwd=1) 
lines(idx, quantiles_xb_SL_50_corrected[3, 1, idx], col="darkgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_50_corrected[1, 1, idx], col="darkgreen", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Reset the plotting parameters to default after plotting is done
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))



## Quantile Forecasts

In [ ]:
# df_t    <- 0.9991048
# df_s    <- 0.9982188
# df_s67  <- 0.9993456
# df_discrep <- rep(0.95, 3-1)
# data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned.csv"
# streamflow_data <- read_csv(data_path, show_col_types = FALSE)
# timestamps <- as.Date(streamflow_data$Date)
# time_series_matrix <- as.matrix(streamflow_data[, c('USGS')])
# Y <- t(time_series_matrix)
# m_yy <- mean(Y, na.rm = TRUE)
# s_yy <- sd(Y, na.rm = TRUE)  
# k <- 0.1*s_yy
# trend.comp = polytrendMod(1, m0 = m_yy, C0 = k)
# harm = c(1, 2, 1/6.8333333)   
# seas.comp = seasMod(p = 363.5854, h = harm , C0 = 0.5*k*diag(2*length(harm)))
# model = combineMods(trend.comp, seas.comp)
# y = Y;
# if(is.null(nrow(y))){ 
#     JJJ <- 1
#     y = array(y, c(JJJ,length(y)))
#  }else{
#     JJJ <- nrow(Y)
#     y = array(y, c(JJJ,ncol(y)))
#  }
# df = c(df_t,df_s, df_s67); dim.df = c(1, 2*length(harm)-2, 2); 
# n.samp = 2000; 
# verbose = TRUE; k = 5;
#   ########################
#   TT = dim(y)[2] 
#   J = dim(y)[1]-1 
#   p = length(model$m0) 
#   ########################
#   m0 = c(model$m0,rep(0,J))
#   C0 = bdiag(model$C0,diag(J))
#   ########################
#   df.mat = make_df_mat(df, dim.df, p)
#   df.mat.k = make_df_mat_k(df, dim.df, p, k)
#   ########################
#   if(J<=0){ # Chanhe to ==0
#   ex.df.mat <- df.mat
#   ex.df.mat.k <- df.mat.k 
#   }else{
#   extra_df.mat <- make_df_mat(df_discrep,rep(1,J),J)
#   extra_df.mat.k <- make_df_mat_k(df_discrep,rep(1,J),J,k)
#   ex.df.mat <- bdiag(df.mat, extra_df.mat)
#   ex.df.mat.k <- bdiag(df.mat.k, extra_df.mat.k)
#   }
#   ########################
#   GG = array( bdiag(model$GG,diag(J)), c(p+J, p+J, TT) )
#   F1 <- matrix(model$FF,p,J+1)
#   F2 <- cbind(rep(0,J),diag(J))
#   FF = array(rbind(F1,F2), c(p+J, 1+J, TT))

## NEEDS REVISION

In [ ]:
# Define parameters and initialize variables
tot_forecast <- 180
# p <- nrow(GG)  # Assuming GG is a square matrix
# J <- nrow(samp.theta_50_SL) - p  # Assuming the number of rows in samp.theta_50_SL is p + J
# n.samp <- dim(samp.theta_50_SL)[3]
# TT <- dim(GG)[3]  # Assuming TT is the last time point index

# Initialize forecast arrays
forecast_50_SL <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))
forecast_05_SL <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))
forecast_95_SL <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))

forecast_50_AV <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))
forecast_05_AV <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))
forecast_95_AV <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))

# Forecast function
forecast_quantile <- function(delta, new_theta_out, samp_theta, forecast_array) {
  # df_s67 <- 0.9999
  # df <- c(delta, df_s67)

  df <- c(delta)
  for (k in 1:tot_forecast) {
    M_k <- make_df_mat_k(df, dim.df, p, 1)
    P <- GG[,,TT] %*% new_theta_out$sC[,,TT] %*% t(GG[,,TT])
    W <- P * M_k
    W_sqrt <- sqrtm(W)
    
    for (i in 1:n.samp) {
      e <- matrix(rnorm(p + J), p + J, 1)
      if (k == 1) {
        Q <- GG[,,TT] %*% samp_theta[,TT,i]
      } else {
        Q <- GG[,,TT] %*% forecast_array[,k - 1,i]
      }
      forecast_array[,k,i] <- Q + W_sqrt %*% e
    }
  }
  
  return(forecast_array)
}

# Perform forecasting for each quantile for both models
forecast_50_SL <- forecast_quantile(delta_50_SL, new.theta.out_50_SL, samp.theta_50_SL, forecast_50_SL)
forecast_05_SL <- forecast_quantile(delta_50_SL, new.theta.out_05_SL, samp.theta_05_SL, forecast_05_SL)
forecast_95_SL <- forecast_quantile(delta_95_SL, new.theta.out_95_SL, samp.theta_95_SL, forecast_95_SL)

forecast_50_AV <- forecast_quantile(delta_50_AV, new.theta.out_50_AV, samp.theta_50_AV, forecast_50_AV)
forecast_05_AV <- forecast_quantile(delta_50_AV, new.theta.out_05_AV, samp.theta_05_AV, forecast_05_AV)
forecast_95_AV <- forecast_quantile(delta_95_AV, new.theta.out_95_AV, samp.theta_95_AV, forecast_95_AV)

# Compute xb forecast
compute_xb_forecast <- function(samp_theta, F) {
  # Determine dimensions
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  xb <- array(NA, dim = c(J + 1, n_time, n_sim))
  
  for (t in 1:n_time) {
    theta_t_s <- samp_theta[, t, ]
    xb[, t, ] <- t(F) %*% theta_t_s  # Results in a 3x2000 matrix
  }
  
  return(xb)
}

# Compute xb forecast for each quantile for both models
xb_50_SL_forecast <- compute_xb_forecast(forecast_50_SL, FF[,,TT])
xb_95_SL_forecast <- compute_xb_forecast(forecast_95_SL, FF[,,TT])
xb_05_SL_forecast <- compute_xb_forecast(forecast_05_SL, FF[,,TT])

xb_50_AV_forecast <- compute_xb_forecast(forecast_50_AV, FF[,,TT])
xb_95_AV_forecast <- compute_xb_forecast(forecast_95_AV, FF[,,TT])
xb_05_AV_forecast <- compute_xb_forecast(forecast_05_AV, FF[,,TT])

# Calculate quantiles of the forecasted values
quantiles_xb_SL_50_forecast <- apply(xb_50_SL_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_SL_95_forecast <- apply(xb_95_SL_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_SL_05_forecast <- apply(xb_05_SL_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))

quantiles_xb_AV_50_forecast <- apply(xb_50_AV_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_AV_95_forecast <- apply(xb_95_AV_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_AV_05_forecast <- apply(xb_05_AV_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))


In [ ]:
# Load and process data
init <- TT-500
idx_f <- (TT + 1):(TT + tot_forecast)
yy <- c(Y[1, init:TT], rep(NA_real_, tot_forecast))
idx <- init:(TT + tot_forecast)

data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned_extra.csv"
all_streamflow_data <- read_csv(data_path, show_col_types = FALSE)
truth <- all_streamflow_data[(TT + 1):(TT + tot_forecast), 2]

# Plotting function
plot_forecast <- function(idx, yy, init, TT, new_theta_out_50, new_theta_out_05, new_theta_out_95, idx_f, quantiles_50, quantiles_05, quantiles_95, truth, title) {
  plot.ts(idx, yy, type = 'l', main = title, ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data
  lines(init:TT, new_theta_out_50$exps[1, init:TT], col = 'forestgreen')
  lines(init:TT, new_theta_out_05$exps[1, init:TT], col = 'darkred')
  lines(init:TT, new_theta_out_95$exps[1, init:TT], col = 'darkblue')
  
  # Add lines for forecast data
  lines(idx_f, quantiles_50[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_05[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_95[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[3,1,], col = 'lightblue', lty = 2)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth", "50th qntl", "5th qntl", "95th qntl", "Frcst 50th qntl", "Frcst 5th qntl", "Frcst 95th qntl"),
         col = c("black", "orange", "forestgreen", "darkred", "darkblue", "lightgreen", "pink", "lightblue"),
         lty = c(1, NA, 1, 1, 1, 2, 2, 2), pch = c(NA, 1, NA, NA, NA, NA, NA, NA), bty = "n")
}

# Plot forecasts for SL model
plot_forecast(idx, yy, init, TT, new.theta.out_50_SL, new.theta.out_05_SL, new.theta.out_95_SL, 
              idx_f, quantiles_xb_SL_50_forecast, quantiles_xb_SL_05_forecast, quantiles_xb_SL_95_forecast, truth, "SL Model Forecast")

# Plot forecasts for Averaged model
plot_forecast(idx, yy, init, TT, new.theta.out_50_AV, new.theta.out_05_AV, new.theta.out_95_AV, 
              idx_f, quantiles_xb_AV_50_forecast, quantiles_xb_AV_05_forecast, quantiles_xb_AV_95_forecast, truth, "Averaged Model Forecast")

# Combined Plotting function
plot_combined_forecast <- function(idx, yy, init, TT, new_theta_out_50_SL, new_theta_out_05_SL, new_theta_out_95_SL, quantiles_50_SL, quantiles_05_SL, quantiles_95_SL, new_theta_out_50_AV, new_theta_out_05_AV, new_theta_out_95_AV, quantiles_50_AV, quantiles_05_AV, quantiles_95_AV, idx_f, truth) {
  plot.ts(idx, yy, type = 'l', main = "Combined SL and Averaged Model Forecast", ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data
  lines(init:TT, new_theta_out_50_SL$exps[1, init:TT], col = 'lightgreen')
  lines(init:TT, new_theta_out_05_SL$exps[1, init:TT], col = 'pink')
  lines(init:TT, new_theta_out_95_SL$exps[1, init:TT], col = 'lightblue')
  lines(init:TT, new_theta_out_50_AV$exps[1, init:TT], col = 'darkgreen')
  lines(init:TT, new_theta_out_05_AV$exps[1, init:TT], col = 'darkred')
  lines(init:TT, new_theta_out_95_AV$exps[1, init:TT], col = 'darkblue')
  
  # Add lines for forecast data
  lines(idx_f, quantiles_50_SL[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50_SL[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50_SL[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_05_SL[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05_SL[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05_SL[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_95_SL[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95_SL[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95_SL[3,1,], col = 'lightblue', lty = 2)
  
  lines(idx_f, quantiles_50_AV[1,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_50_AV[2,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_50_AV[3,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_05_AV[1,1,], col = 'darkred', lty = 3)
  lines(idx_f, quantiles_05_AV[2,1,], col = 'darkred', lty = 3)
  lines(idx_f, quantiles_05_AV[3,1,], col = 'darkred', lty = 3)
  lines(idx_f, quantiles_95_AV[1,1,], col = 'darkblue', lty = 3)
  lines(idx_f, quantiles_95_AV[2,1,], col = 'darkblue', lty = 3)
  lines(idx_f, quantiles_95_AV[3,1,], col = 'darkblue', lty = 3)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth",  "SL: 50th qntl", "SL: 5th qntl", "SL: 95th qntl", "SL: Frcst 50th qntl", "SL: Frcst 5th qntl", "SL: Frcst 95th qntl", "AV: 50th qntl", "AV: 5th qntl", "AV: 95th qntl", "AV: Frcst 50th qntl", "AV: Frcst 5th qntl", "AV: Frcst 95th qntl"),
         col = c("black", "orange", "darkgreen", "darkred", "darkblue", "lightgreen", "pink", "lightblue", "lightgreen", "pink", "lightblue", "darkgreen", "darkred", "darkblue"),
         lty = c(1, NA, 1, 1, 1, 2, 2, 2, 1, 1, 1, 3, 3, 3), pch = c(NA, 1, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA), bty = "n")
}

# Combined plot for both models
plot_combined_forecast(idx, yy, init, TT, new.theta.out_50_SL, new.theta.out_05_SL, new.theta.out_95_SL, quantiles_xb_SL_50_forecast, quantiles_xb_SL_05_forecast, quantiles_xb_SL_95_forecast, new.theta.out_50_AV, new.theta.out_05_AV, new.theta.out_95_AV, quantiles_xb_AV_50_forecast, quantiles_xb_AV_05_forecast, quantiles_xb_AV_95_forecast, idx_f, truth)


In [ ]:
# df_t    <- 0.9991048
# df_s    <- 0.9982188
# df_s67  <- 0.9993456
# df_discrep <- rep(0.95, 3-1)
# data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned.csv"
# streamflow_data <- read_csv(data_path, show_col_types = FALSE)
# timestamps <- as.Date(streamflow_data$Date)
# time_series_matrix <- as.matrix(streamflow_data[, c('USGS')])
# Y <- t(time_series_matrix)
# m_yy <- mean(Y, na.rm = TRUE)
# s_yy <- sd(Y, na.rm = TRUE)  
# k <- 0.1*s_yy
# trend.comp = polytrendMod(1, m0 = m_yy, C0 = k)
# harm = c(1, 2, 1/6.8333333)   
# seas.comp = seasMod(p = 363.5854, h = harm , C0 = 0.5*k*diag(2*length(harm)))
# model = combineMods(trend.comp, seas.comp)
# y = Y;
# if(is.null(nrow(y))){ 
#     JJJ <- 1
#     y = array(y, c(JJJ,length(y)))
#  }else{
#     JJJ <- nrow(Y)
#     y = array(y, c(JJJ,ncol(y)))
#  }
# df = c(df_t,df_s, df_s67); dim.df = c(1, 2*length(harm)-2, 2); 
# n.samp = 2000; 
# verbose = TRUE; k = 5;
#   ########################
#   TT = dim(y)[2] 
#   J = dim(y)[1]-1 
#   p = length(model$m0) 
#   ########################
#   m0 = c(model$m0,rep(0,J))
#   C0 = bdiag(model$C0,diag(J))
#   ########################
#   df.mat = make_df_mat(df, dim.df, p)
#   df.mat.k = make_df_mat_k(df, dim.df, p, k)
#   ########################
#   if(J<=0){ # Chanhe to ==0
#   ex.df.mat <- df.mat
#   ex.df.mat.k <- df.mat.k 
#   }else{
#   extra_df.mat <- make_df_mat(df_discrep,rep(1,J),J)
#   extra_df.mat.k <- make_df_mat_k(df_discrep,rep(1,J),J,k)
#   ex.df.mat <- bdiag(df.mat, extra_df.mat)
#   ex.df.mat.k <- bdiag(df.mat.k, extra_df.mat.k)
#   }
#   ########################
#   GG = array( bdiag(model$GG,diag(J)), c(p+J, p+J, TT) )
#   F1 <- matrix(model$FF,p,J+1)
#   F2 <- cbind(rep(0,J),diag(J))
#   FF = array(rbind(F1,F2), c(p+J, 1+J, TT))

In [ ]:
# # Parameters and initial setup
# df_t    <- 0.9991048
# df_s    <- 0.9982188
# df_s67  <- 0.9993456
# df_discrep <- rep(0.95, 3-1)
# ###########################################################################################
# data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned.csv"
# streamflow_data <- read_csv(data_path, show_col_types = FALSE)
# timestamps <- as.Date(streamflow_data$Date, show_col_types = FALSE)
# time_series_matrix <- as.matrix(streamflow_data[, c('USGS', 'NWS3.0',  'GloFAS')])
# Y <- t(time_series_matrix)
# ###########################################################################################
# TT <- ncol(Y)  
# y = Y;
# ###########################################################################################
# m_yy <- mean(Y, na.rm = TRUE)
# s_yy <- sd(Y, na.rm = TRUE)  
# k <- 0.1*s_yy
# trend.comp = polytrendMod(1, m0 = m_yy, C0 = k)
# harm = harmonics
# seas.comp = seasMod(p = 363.5854, h = harm , C0 = 0.5*k*diag(2*length(harm)))
# model = combineMods(trend.comp, seas.comp)
# if(is.null(nrow(y))){ 
#     JJJ <- 1
#     y = array(y, c(JJJ,length(y)))
#  }else{
#     JJJ <- nrow(Y)
#     y = array(y, c(JJJ,ncol(y)))
#  }
# df = c(df_t,df_s, df_s67); 
# dim.df = c(1, 2*length(harm)-2, 2); 
# gam.init = array(rep(0.01,JJJ), c(JJJ,1)); 
# sig.init = array(rep(0.1,JJJ), c(JJJ,1));
# n.samp = 2000; 
# PriorSigma = array(NA_real_, c(JJJ,2)); 
# PriorGamma = array(NA_real_, c(JJJ,3)); 
# verbose = TRUE; k = 5;
# ###########################################################################################
#   TT = dim(y)[2] 
#   J = dim(y)[1]-1 
#   p = length(model$m0) 
# ###########################################################################################
#   m0 = c(model$m0,rep(0,J))
#   C0 = bdiag(model$C0,diag(J))
# ###########################################################################################
#   model_simp <- model
#   df_simp <- df
#   dim.df_simp <- dim.df
#   model_simp$GG = array(model_simp$GG, c(p, p, TT))
#   model_simp$FF = array(model_simp$FF, c(p, 1, TT))
#   df.mat = make_df_mat(df, dim.df, p)
#   df.mat.k = make_df_mat_k(df, dim.df, p, k)
# ###########################################################################################
#   if(J<=0){
#   ex.df.mat <- df.mat
#   ex.df.mat.k <- df.mat.k 
#   }else{
#   extra_df.mat <- make_df_mat(df_discrep,rep(1,J),J)
#   extra_df.mat.k <- make_df_mat_k(df_discrep,rep(1,J),J,k)
#   ex.df.mat <- bdiag(df.mat, extra_df.mat)
#   ex.df.mat.k <- bdiag(df.mat.k, extra_df.mat.k)
#   }
# ###########################################################################################
#   GG = array( bdiag(model$GG,diag(J)), c(p+J, p+J, TT) )
#   model$GG = GG
#   F1 <- matrix(model$FF,p,J+1)
#   F2 <- cbind(rep(0,J),diag(J))
#   FF = array(rbind(F1,F2), c(p+J, 1+J, TT))
#   model$FF = FF


In [ ]:
# Define parameters and initialize variables
n.samp <- dim(samp.theta_50_M)[3]

# Initialize forecast arrays
forecast_50_M <- array(NA_real_, c(p + J, tot_forecast, n.samp))
forecast_05_M <- array(NA_real_, c(p + J, tot_forecast, n.samp))
forecast_95_M <- array(NA_real_, c(p + J, tot_forecast, n.samp))

# Function to compute matrix square root
sqrtm <- function(mat) {
  e <- eigen(mat)
  e$vectors %*% diag(sqrt(e$values)) %*% t(e$vectors)
}

# Forecast function
forecast_quantile <- function(delta, new_theta_out, samp_theta, forecast_array) {
  df <- delta[1:3]
  df_discrep <- rep(delta[4], J)

  for (k in 1:tot_forecast) {
    
    df.mat.k = make_df_mat_k(df, dim.df, p, 1)
    extra_df.mat.k <- make_df_mat_k(df_discrep,rep(1,J),J,k)
    M_k <- bdiag(df.mat.k, extra_df.mat.k)

    P <- GG[,,TT] %*% new_theta_out$sC[,,TT] %*% t(GG[,,TT])
    W <- P * M_k
    W_sqrt <- sqrtm(W)
    
    for (i in 1:n.samp) {
      e <- matrix(rnorm(p + J), p + J, 1)
      if (k == 1) {
        Q <- GG[,,TT] %*% samp_theta[,TT,i]
      } else {
        Q <- GG[,,TT] %*% forecast_array[,k - 1,i]
      }
      forecast_array[,k,i] <- Q + W_sqrt %*% e
    }
  }
  
  return(forecast_array)
}

# Perform forecasting for each quantile for the exAL model
forecast_50_M <- forecast_quantile(delta_50_M, new.theta.out_50_M, samp.theta_50_M, forecast_50_M)
forecast_05_M <- forecast_quantile(delta_50_M, new.theta.out_5_M, samp.theta_5_M, forecast_05_M)
forecast_95_M <- forecast_quantile(delta_95_M, new.theta.out_95_M, samp.theta_95_M, forecast_95_M)

# Compute xb forecast
compute_xb_forecast <- function(samp_theta, F) {
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  xb <- array(NA, dim = c(J + 1, n_time, n_sim))
  
  for (t in 1:n_time) {
    theta_t_s <- samp_theta[, t, ]
    xb[, t, ] <- t(F) %*% theta_t_s  # Results in a 3x2000 matrix
  }
  
  return(xb)
}

# Compute xb forecast for each quantile for the exAL model
xb_50_M_forecast <- compute_xb_forecast(forecast_50_M, FF[,,TT])
xb_95_M_forecast <- compute_xb_forecast(forecast_95_M, FF[,,TT])
xb_05_M_forecast <- compute_xb_forecast(forecast_05_M, FF[,,TT])

# Calculate quantiles of the forecasted values
quantiles_xb_M_50_forecast <- apply(xb_50_M_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_M_95_forecast <- apply(xb_95_M_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_M_05_forecast <- apply(xb_05_M_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))

# Load and process data
init <- 15900
idx_f <- (TT + 1):(TT + tot_forecast)
yy <- c(Y[1, init:TT], rep(NA_real_, tot_forecast))
idx <- init:(TT + tot_forecast)

data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned_extra.csv"
all_streamflow_data <- read_csv(data_path, show_col_types = FALSE)
truth <- all_streamflow_data[(TT + 1):(TT + tot_forecast), 2]

# Plotting function
plot_forecast <- function(idx, yy, init, TT, new_theta_out_50, new_theta_out_05, new_theta_out_95, idx_f, quantiles_50, quantiles_05, quantiles_95, truth, title) {
  plot.ts(idx, yy, type = 'l', main = title, ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data
  lines(init:TT, new_theta_out_50$exps[1, init:TT], col = 'forestgreen')
  lines(init:TT, new_theta_out_05$exps[1, init:TT], col = 'darkred')
  lines(init:TT, new_theta_out_95$exps[1, init:TT], col = 'darkblue')
  
  # Add lines for forecast data
  lines(idx_f, quantiles_50[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_05[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_95[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[3,1,], col = 'lightblue', lty = 2)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth", "50th qntl", "5th qntl", "95th qntl", "Frcst 50th qntl", "Frcst 5th qntl", "Frcst 95th qntl"),
         col = c("black", "orange", "forestgreen", "darkred", "darkblue", "lightgreen", "pink", "lightblue"),
         lty = c(1, NA, 1, 1, 1, 2, 2, 2), pch = c(NA, 1, NA, NA, NA, NA, NA, NA), bty = "n")
}

# Plot forecasts for the exAL model
plot_forecast(idx, yy, init, TT, new.theta.out_50_M, new.theta.out_5_M, new.theta.out_95_M, 
              idx_f, quantiles_xb_M_50_forecast, quantiles_xb_M_05_forecast, quantiles_xb_M_95_forecast, truth, "exAL Model Forecast")


In [ ]:

# Forecast function for NDLM
forecast_quantile_ndlm <- function(delta, new_theta_out, samp_theta, forecast_array, sig_samp, pp) {
  df <- delta[1:3]
  df_discrep <- rep(delta[4], J)
  
  for (k in 1:tot_forecast) {
    df.mat.k <- make_df_mat_k(df, dim.df, p, k)
    extra_df.mat.k <- make_df_mat_k(df_discrep, rep(1, J), J, k)
    M_k <- bdiag(df.mat.k, extra_df.mat.k)
    
    P <- GG[,,TT] %*% new_theta_out$sC[,,TT] %*% t(GG[,,TT])
    W <- P * M_k
    W_sqrt <- sqrtm(W)
    
    for (i in 1:n.samp) {
      e <- matrix(rnorm(p + J), p + J, 1)
      if (k == 1) {
        Q <- GG[,,TT] %*% samp_theta[,TT,i]
      } else {
        Q <- GG[,,TT] %*% forecast_array[,k - 1,i]
      }
      forecast_array[,k,i] <- Q + W_sqrt %*% e
    }
  }
  
  # Add quantile adjustment
  for (i in 1:n.samp) {
    forecast_array[,,i] <- forecast_array[,,i] + sqrt(sig_samp[i,1]) * qnorm(pp)
  }
  
  return(forecast_array)
}

# Parameters for NDLM model
df_t    <- 0.9992
df_s    <- 0.997
df_s67  <- 0.9992
df_discrep <- rep(0.94, 3-1)
delta_ndlm <- c(df_t, df_s, df_s67, df_discrep[1])

# Initialize forecast arrays for NDLM
forecast_50_NDLM <- array(NA_real_, c(p + J, tot_forecast, n.samp))
xb_05_NDLM_forecast <- array(NA_real_, c(1, tot_forecast, n.samp))
xb_95_NDLM_forecast <- array(NA_real_, c(1, tot_forecast, n.samp))

forecast_50_NDLM <- forecast_quantile_ndlm(delta_ndlm, new.theta.out_M, samp.theta_M, forecast_50_NDLM, samp.sigma_M, 0.5)
xb_50_NDLM_forecast <- compute_xb_forecast(forecast_50_NDLM, FF[,,TT])

for (i in 1:n.samp) {
  xb_05_NDLM_forecast[,,i] <- xb_50_NDLM_forecast[1,,i] + sqrt(samp.sigma_M[i,1]) * qnorm(0.05)
  xb_95_NDLM_forecast[,,i] <- xb_50_NDLM_forecast[1,,i] + sqrt(samp.sigma_M[i,1]) * qnorm(0.95)
}

quantiles_xb_NDLM_50_forecast <- apply(xb_50_NDLM_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_NDLM_05_forecast <- apply(xb_05_NDLM_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_NDLM_95_forecast <- apply(xb_95_NDLM_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))

# Plotting function
plot_forecast_ndlm <- function(idx, yy, init, TT, xb_50, xb_05, xb_95, idx_f, quantiles_50, quantiles_05, quantiles_95, truth, title) {
  plot.ts(idx, yy, type = 'l', main = title, ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data
  lines(init:TT, rowMeans(xb_50[1,,])[init:TT], col = 'forestgreen')
  lines(init:TT, rowMeans(xb_05[1,,])[init:TT], col = 'darkred')
  lines(init:TT, rowMeans(xb_95[1,,])[init:TT], col = 'darkblue')
  
  # Add lines for forecast data
  lines(idx_f, quantiles_50[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_05[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_95[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[3,1,], col = 'lightblue', lty = 2)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth", "50th qntl", "5th qntl", "95th qntl", "Frcst 50th qntl", "Frcst 5th qntl", "Frcst 95th qntl"),
         col = c("black", "orange", "forestgreen", "darkred", "darkblue", "lightgreen", "pink", "lightblue"),
         lty = c(1, NA, 1, 1, 1, 2, 2, 2), pch = c(NA, 1, NA, NA, NA, NA, NA, NA), bty = "n")
}
# Plot forecasts for the NDLM model
plot_forecast_ndlm(idx, yy, init, TT, xb_M_50, xb_M_05, xb_M_95, 
              idx_f, quantiles_xb_NDLM_50_forecast, quantiles_xb_NDLM_05_forecast, quantiles_xb_NDLM_95_forecast, truth, "NDLM Model Forecast")

In [ ]:
# Combined Plotting function for exAL and NDLM models
plot_combined_forecast_ndlm_exal <- function(idx, yy, init, TT, new_theta_out_50_exal, new_theta_out_05_exal, new_theta_out_95_exal, 
                                             quantiles_exal_50, quantiles_exal_05, quantiles_exal_95, 
                                             xb_50_ndlm, xb_05_ndlm, xb_95_ndlm, 
                                             quantiles_ndlm_50, quantiles_ndlm_05, quantiles_ndlm_95, 
                                             idx_f, truth) {
  plot.ts(idx, yy, type = 'l', main = "Combined exAL and NDLM Model Forecast", ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data for exAL
  lines(init:TT, new_theta_out_50_exal$exps[1, init:TT], col = 'lightgreen')
  lines(init:TT, new_theta_out_05_exal$exps[1, init:TT], col = 'pink')
  lines(init:TT, new_theta_out_95_exal$exps[1, init:TT], col = 'lightblue')
  
  lines(init:TT, rowMeans(xb_50_ndlm[1,,])[init:TT], col = 'darkgreen')
  lines(init:TT, rowMeans(xb_05_ndlm[1,,])[init:TT], col = 'darkred')
  lines(init:TT, rowMeans(xb_95_ndlm[1,,])[init:TT], col = 'darkblue')
  

  # Add lines for forecast data for exAL
  lines(idx_f, quantiles_exal_50[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_exal_50[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_exal_50[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_exal_05[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_exal_05[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_exal_05[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_exal_95[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_exal_95[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_exal_95[3,1,], col = 'lightblue', lty = 2)
  
  # Add lines for forecast data for NDLM
  lines(idx_f, quantiles_ndlm_50[1,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_ndlm_50[2,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_ndlm_50[3,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_ndlm_05[1,1,], col = 'red', lty = 3)
  lines(idx_f, quantiles_ndlm_05[2,1,], col = 'red', lty = 3)
  lines(idx_f, quantiles_ndlm_05[3,1,], col = 'red', lty = 3)
  lines(idx_f, quantiles_ndlm_95[1,1,], col = 'blue', lty = 3)
  lines(idx_f, quantiles_ndlm_95[2,1,], col = 'blue', lty = 3)
  lines(idx_f, quantiles_ndlm_95[3,1,], col = 'blue', lty = 3)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth", "exAL: 50th qntl", "exAL: 5th qntl", "exAL: 95th qntl", 
                               "NDLM: 50th qntl", "NDLM: 5th qntl", "NDLM: 95th qntl"),
         col = c("black", "orange", "lightgreen", "pink", "lightblue", "darkgreen", "darkred", "darkblue"),
         lty = c(1, NA, 1, 1, 1, 1, 1, 1), pch = c(NA, 1, NA, NA, NA, NA, NA, NA), bty = "n")
}


# Combined plot for exAL and NDLM models
plot_combined_forecast_ndlm_exal(idx, yy, init, TT, new.theta.out_50_M, new.theta.out_5_M, new.theta.out_95_M, 
                                 quantiles_xb_M_50_forecast, quantiles_xb_M_05_forecast, quantiles_xb_M_95_forecast, 
                                 xb_M_50, xb_M_05, xb_M_95, 
                                 quantiles_xb_NDLM_50_forecast, quantiles_xb_NDLM_05_forecast, quantiles_xb_NDLM_95_forecast, 
                                 idx_f, truth)


# Scores for forecasts

In [ ]:
CheckLossFn = function(p0,diff){diff*p0 - diff*as.numeric(diff<0)}

In [ ]:
a_CRPS <- function(y,q1,q2,q3,ps){
    d1 <- y-q1; d2 <- y-q2; d3 <- y-q3;
    score <- CheckLossFn(ps[1],d1)+CheckLossFn(ps[2],d2)+CheckLossFn(ps[3],d3)
    return(score)
}

IS_a <- function(l,u,a,y){
    score <- (u-l)+2/a*(l-y)*ifelse(y<l,1,0)+2/a*(y-u)*ifelse(y>=u,1,0)
    return(score)
}

WIS_a <- function(K,m,L,U,A,y){
    score <- 0.5*abs(y-m)
    for(k in 1:K){
        score <- score +  A[k]/2*IS_a(L,U,A[k],y)
    }
    score <- score/(K+0.5)
    return(score)
}

Disp_a <- function(l,u,a){
    score <- a/2*(u-l)
    return(score)
}

IC_a <- function(y,l,u,a){
    score <- (ifelse(l<y,1,0)*ifelse(y<u,1,0))
    return(score)
}

CD_a <- function(y,l,u,a){
    score <- 1-a - IC_a(y,l,u,a)
    return(score)
}

QS_a <- function(l,u){
    score <- (u-l)
    return(score)
}

QS_a <- function(l,u){
    score <- (u-l)
    return(score)
}

## TODO: QB_a



## In-Sample Scoring

### a-CRPS, IS, WIS, Disp, IC, CD

In [ ]:
IS_AV <- array(NA_real_, c(1,TT) )
IS_SL <- array(NA_real_, c(1,TT) )
IS_NDLM <- array(NA_real_, c(1,TT) )
IS_exAL <- array(NA_real_, c(1,TT) )

a_CRPS_AV <- array(NA_real_, c(1,TT) )
a_CRPS_SL <- array(NA_real_, c(1,TT) )
a_CRPS_NDLM <- array(NA_real_, c(1,TT) )
a_CRPS_exAL <- array(NA_real_, c(1,TT) )

WIS_AV <- array(NA_real_, c(1,TT) )
WIS_SL <- array(NA_real_, c(1,TT) )
WIS_NDLM <- array(NA_real_, c(1,TT) )
WIS_exAL <- array(NA_real_, c(1,TT) )

DISP_AV <- array(NA_real_, c(1,TT) )
DISP_SL <- array(NA_real_, c(1,TT) )
DISP_NDLM <- array(NA_real_, c(1,TT) )
DISP_exAL <- array(NA_real_, c(1,TT) )

IC_AV <- array(NA_real_, c(1,TT) )
IC_SL <- array(NA_real_, c(1,TT) )
IC_NDLM <- array(NA_real_, c(1,TT) )
IC_exAL <- array(NA_real_, c(1,TT) )

CD_SL <- array(NA_real_, c(1,TT) )
CD_AV <- array(NA_real_, c(1,TT) )
CD_NDLM <- array(NA_real_, c(1,TT) )
CD_exAL <- array(NA_real_, c(1,TT) )

QS_SL <- array(NA_real_, c(1,TT) )
QS_AV <- array(NA_real_, c(1,TT) )
QS_NDLM <- array(NA_real_, c(1,TT) )
QS_exAL <- array(NA_real_, c(1,TT) )

for(t in 1:TT){
    yt <- Y[1,t]
    a <- 0.1 
    A <- c(a)

    Ut <- xb_95_AV_corrected[1,t,]
    Lt <- xb_05_AV_corrected[1,t,]
    Mt <- xb_50_AV_corrected[1,t,]
    
    IS_AV[t] <- mean(IS_a(Lt,Ut,a,yt))
    a_CRPS_AV[t]<- mean(a_CRPS(yt,Lt,Mt,Ut,c(0.05,0.5,0.95)))
    WIS_AV[t] <- mean(WIS_a(1,Mt,Lt,Ut,A,yt))
    DISP_AV[t] <- mean(Disp_a(Lt,Ut,a))
    IC_AV[t] <- mean(IC_a(yt,Lt,Ut,a))
    CD_AV[t] <- mean(CD_a(yt,Lt,Ut,a))
    QS_AV[t] <- mean(QS_a(Lt,Ut))

    Ut <- xb_95_SL_corrected[1,t,]
    Lt <- xb_05_SL_corrected[1,t,]
    Mt <- xb_50_SL_corrected[1,t,]

    IS_SL[t] <- mean(IS_a(Lt,Ut,a,yt))
    a_CRPS_SL[t]<- mean(a_CRPS(yt,Lt,Mt,Ut,c(0.05,0.5,0.95)))
    WIS_SL[t] <- mean(WIS_a(1,Mt,Lt,Ut,A,yt))
    DISP_SL[t] <- mean(Disp_a(Lt,Ut,a))
    IC_SL[t] <- mean(IC_a(yt,Lt,Ut,a))
    CD_SL[t] <- mean(CD_a(yt,Lt,Ut,a))
    QS_SL[t] <- mean(QS_a(Lt,Ut))

    Ut <- xb_M_95[1,t,]
    Lt <- xb_M_05[1,t,]
    Mt <- xb_M_50[1,t,]

    IS_NDLM[t] <- mean(IS_a(Lt,Ut,a,yt))
    a_CRPS_NDLM[t]<- mean(a_CRPS(yt,Lt,Mt,Ut,c(0.05,0.5,0.95)))
    WIS_NDLM[t] <- mean(WIS_a(1,Mt,Lt,Ut,A,yt))
    DISP_NDLM[t] <- mean(Disp_a(Lt,Ut,a))
    IC_NDLM[t] <- mean(IC_a(yt,Lt,Ut,a))
    CD_NDLM[t] <- mean(CD_a(yt,Lt,Ut,a))
    QS_NDLM[t] <- mean(QS_a(Lt,Ut))

    Ut <- xb_95_corrected[1,t,]
    Lt <- xb_05_corrected[1,t,]
    Mt <- xb_50_corrected[1,t,]

    IS_exAL[t] <- mean(IS_a(Lt,Ut,a,yt))
    a_CRPS_exAL[t]<- mean(a_CRPS(yt,Lt,Mt,Ut,c(0.05,0.5,0.95)))
    WIS_exAL[t] <- mean(WIS_a(1,Mt,Lt,Ut,A,yt))
    DISP_exAL[t] <- mean(Disp_a(Lt,Ut,a))
    IC_exAL[t] <- mean(IC_a(yt,Lt,Ut,a))
    CD_exAL[t] <- mean(CD_a(yt,Lt,Ut,a))
    QS_exAL[t] <- mean(QS_a(Lt,Ut))

}

add windowed with rolling mean

In [ ]:
mean(IS_AV[1,])
mean(IS_SL[1,])
mean(IS_NDLM[1,])
mean(IS_exAL[1,])

plot.ts(IS_AV[1,])
lines(IS_SL[1,], col='darkred')
lines(IS_NDLM[1,], col='darkgreen')
lines(IS_exAL[1,], col='purple')

plot.ts(cumsum(IS_AV[1,])/TT)
lines(cumsum(IS_SL[1,])/TT, col='darkred')
lines(cumsum(IS_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(IS_exAL[1,])/TT, col='purple')

In [ ]:
mean(a_CRPS_AV[1,])
mean(a_CRPS_SL[1,])
mean(a_CRPS_NDLM[1,])
mean(a_CRPS_exAL[1,])

plot.ts(a_CRPS_AV[1,])
lines(a_CRPS_SL[1,], col='darkred')
lines(a_CRPS_NDLM[1,], col='darkgreen')
lines(a_CRPS_exAL[1,], col='purple')

plot.ts(cumsum(a_CRPS_AV[1,])/TT)
lines(cumsum(a_CRPS_SL[1,])/TT, col='darkred')
lines(cumsum(a_CRPS_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(a_CRPS_exAL[1,])/TT, col='purple')

In [ ]:
mean(WIS_AV[1,])
mean(WIS_SL[1,])
mean(WIS_NDLM[1,])
mean(WIS_exAL[1,])

plot.ts(WIS_AV[1,])
lines(WIS_SL[1,], col='darkred')
lines(WIS_NDLM[1,], col='darkgreen')
lines(WIS_exAL[1,], col='purple')

plot.ts(cumsum(WIS_AV[1,])/TT)
lines(cumsum(WIS_SL[1,])/TT, col='darkred')
lines(cumsum(WIS_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(WIS_exAL[1,])/TT, col='purple')

In [ ]:
mean(DISP_AV[1,])
mean(DISP_SL[1,])
mean(DISP_NDLM[1,])
mean(DISP_exAL[1,])

plot.ts(DISP_AV[1,])
lines(DISP_SL[1,], col='darkred')
lines(DISP_NDLM[1,], col='darkgreen')
lines(DISP_exAL[1,], col='purple')

plot.ts(cumsum(DISP_AV[1,])/TT, ylim=c(0,0.1))
lines(cumsum(DISP_SL[1,])/TT, col='darkred')
lines(cumsum(DISP_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(DISP_exAL[1,])/TT, col='purple')

In [ ]:
mean(IC_AV[1,])
mean(IC_SL[1,])
mean(IC_NDLM[1,])
mean(IC_exAL[1,])

# plot.ts(IC_AV[1,])
# lines(IC_SL[1,], col='darkred')
# lines(IC_NDLM[1,], col='darkgreen')

plot.ts(cumsum(IC_AV[1,])/TT, ylim = c(0,1))
lines(cumsum(IC_SL[1,])/TT, col='darkred')
lines(cumsum(IC_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(IC_exAL[1,])/TT, col='purple')
abline(h=0.9, col='orange')

In [ ]:
mean(CD_AV[1,])
mean(CD_SL[1,])
mean(CD_NDLM[1,])
mean(CD_exAL[1,])

# plot.ts(CD_AV[1,])
# lines(CD_SL[1,], col='darkred')
# lines(CD_NDLM[1,], col='darkgreen')

plot.ts(cumsum(CD_AV[1,])/TT,ylim = c(-0.1,0.3))
lines(cumsum(CD_SL[1,])/TT, col='darkred')
lines(cumsum(CD_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(CD_exAL[1,])/TT, col='purple')
abline(h=0, col='orange')

In [ ]:
mean(QS_AV[1,])
mean(QS_SL[1,])
mean(QS_NDLM[1,])
mean(QS_exAL[1,])

plot.ts(QS_AV[1,])
lines(QS_SL[1,], col='darkred')
lines(QS_NDLM[1,], col='darkgreen')
lines(QS_exAL[1,], col='purple')


plot.ts(cumsum(QS_AV[1,])/TT,ylim = c(0,3))
lines(cumsum(QS_SL[1,])/TT, col='darkred')
lines(cumsum(QS_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(QS_exAL[1,])/TT, col='purple')

In [ ]:
load_variables <- function(filename, dir_path) {
  file_path <- file.path(dir_path, filename)
  load(file_path)
  cat("Variables loaded from:", file_path, "\n")
}

file_path <- "/home/jaguir26/project1_ucsc_phd/variables_50_exAL.RData"
load(file_path)

In [ ]:
# library(ks)
# library(MASS)

# # Function to estimate differential entropy using KDE for univariate data
# estimate_differential_entropy_kde_univariate <- function(data) {
#   kde_result <- kde(data)
#   estimates <- kde_result$estimate
#   estimates[estimates <= 0] <- .Machine$double.eps # Prevent log(0) issues
#   log_estimates <- log(estimates)
#   log_estimates[!is.finite(log_estimates)] <- 0 # Handle non-finite values
#   entropy_estimate <- -sum(estimates * log_estimates) * diff(kde_result$eval.points)[1]
#   return(entropy_estimate)
# }

# # Function to estimate differential entropy using KDE for multivariate data
# estimate_differential_entropy_kde_multivariate <- function(data) {
#   kde_result <- kde(data)
#   estimates <- kde_result$estimate
#   estimates[estimates <= 0] <- .Machine$double.eps # Prevent log(0) issues
#   log_estimates <- log(estimates)
#   log_estimates[!is.finite(log_estimates)] <- 0 # Handle non-finite values
#   entropy_estimate <- -sum(estimates * log_estimates) * prod(diff(kde_result$eval.points[[1]]))
#   return(entropy_estimate)
# }

# # Function to estimate the KL divergence D_KL(p || N(0, I)) for univariate data
# estimate_kl_divergence_univariate <- function(data) {
#   # Estimate the differential entropy H(p)
#   H_p <- estimate_differential_entropy_kde_univariate(data)
  
#   # Compute the expected value of the squared norm of the vectors
#   E_p_x2 <- mean(data^2)
  
#   # Dimensionality is 1 for univariate data
#   k <- 1
  
#   # Compute the KL divergence
#   kl_divergence <- -H_p + (k / 2) * log(2 * pi) + (1 / 2) * E_p_x2
  
#   return(kl_divergence)
# }

# # Function to estimate the KL divergence D_KL(p || N(0, I)) for multivariate data
# estimate_kl_divergence_multivariate <- function(data) {
#   # Estimate the differential entropy H(p)
#   H_p <- estimate_differential_entropy_kde_multivariate(data)
  
#   # Dimensionality of the vectors
#   k <- ncol(data)
  
#   # Compute the expected value of the squared norm of the vectors
#   E_p_xTx <- mean(rowSums(data^2))
  
#   # Compute the KL divergence
#   kl_divergence <- -H_p + (k / 2) * log(2 * pi) + (1 / 2) * E_p_xTx
  
#   return(kl_divergence)
# }

# # Wrapper function for any sample
# compute_kl_divergence <- function(sample) {
#   # Ensure the input sample is a matrix
#   sample <- as.matrix(sample)
  
#   # Determine if the sample is univariate or multivariate
#   if (ncol(sample) == 1) {
#     kl_divergence <- estimate_kl_divergence_univariate(sample)
#   } else {
#     kl_divergence <- estimate_kl_divergence_multivariate(sample)
#   }
  
#   return(kl_divergence)
# }

# # Example usage
# set.seed(123)
# sample_size <- 1000

# # Univariate case
# data_univariate <- rnorm(sample_size, mean = 2, sd = 2)
# kl_divergence_univariate <- compute_kl_divergence(data_univariate)
# print(kl_divergence_univariate)

# # Multivariate case
# data_multivariate <- MASS::mvrnorm(sample_size, mu = c(0, 0, 0), Sigma = diag(3))
# kl_divergence_multivariate <- compute_kl_divergence(data_multivariate)
# print(kl_divergence_multivariate)


In [ ]:
errors <- t(new.theta.out_50_exAL$standard_forecast_errors)
kl_divergence <- compute_kl_divergence(errors)
print(kl_divergence)

errors <- matrix(new.theta.out_50_exAL$standard_forecast_errors[2,], ncol = 1)
kl_divergence <- compute_kl_divergence(errors)
print(kl_divergence)

In [ ]:
plot.ts(new.theta.out_50_exAL$standard_forecast_errors[1,])

In [ ]:
# Function to compute descriptive statistics
descriptive_stats <- function(data) {
  stats <- data.frame(
    Mean = apply(data, 1, mean),
    Variance = apply(data, 1, var),
    Skewness = apply(data, 1, function(x) mean((x - mean(x))^3) / (sd(x)^3)),
    Kurtosis = apply(data, 1, function(x) mean((x - mean(x))^4) / (sd(x)^4) - 3)
  )
  return(stats)
}

# Compute and print descriptive statistics for the residuals
residual_stats <- descriptive_stats(errors)
print(residual_stats)


Compare with barata

In [ ]:
# Residual plots
par(mfrow = c(3, 1))  # Set up a 3-row plot layout

# Residual plots for each dimension
for (i in 1:3) {
  plot(errors[i, ], main = paste("Residual Plot for Dimension", i), ylab = "Residuals")
  abline(h = 0, col = "red")
}

par(mfrow = c(1, 1))  # Reset to default layout


In [ ]:
# ACF plots for each dimension
par(mfrow = c(3, 1))  # Set up a 3-row plot layout

# ACF plots for each dimension
for (i in 1:3) {
  acf(errors[i, ], main = paste("ACF of Residuals for Dimension", i))
}

par(mfrow = c(1, 1))  # Reset to default layout


In [ ]:
# Histograms and Q-Q plots for each dimension
par(mfrow = c(2, 3))  # Set up a 3x2 plot layout

# Histograms
for (i in 1:3) {
  hist(errors[i, ], main = paste("Histogram of Residuals (Dimension", i, ")"), xlab = "Residual Value", breaks = 50)
}

# Q-Q plots
for (i in 1:3) {
  qqnorm(errors[i, ], main = paste("Q-Q Plot of Residuals (Dimension", i, ")"))
  qqline(errors[i, ], col = "red")
}


par(mfrow = c(1, 1))  # Reset to default layout


In [ ]:
# Required Libraries
library(zoo)

# Check Loss Function
CheckLossFn <- function(p0, diff) {
  diff * p0 - diff * as.numeric(diff < 0)
}

# Approximated CRPS
a_CRPS <- function(y, quantiles, ps) {
  score <- 0
  for (i in 1:length(ps)) {
    diff <- y - quantiles[i]
    score <- score + CheckLossFn(ps[i], diff)
  }
  return(2 / length(ps) * score)
}

# Interval Score
IS_a <- function(l, u, a, y) {
  score <- (u - l) + 2/a * (l - y) * ifelse(y < l, 1, 0) + 2/a * (y - u) * ifelse(y >= u, 1, 0)
  return(score)
}

# Weighted Interval Score
WIS_a <- function(m, quantiles, ps, y, coverages) {
  K <- length(coverages)
  score <- 0.5 * abs(y - m)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    L <- quantiles[L_index]
    U <- quantiles[U_index]
    score <- score + alpha/2 * IS_a(L, U, alpha, y)
  }
  score <- score / (K + 0.5)
  return(score)
}

# Dispersion of WIS
Disp_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  score <- 0
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    score <- score + alpha/2 * (u - l)
  }
  return(score)
}

# Interval Coverage
IC_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
  }
  return(scores)
}

# Coverage Deviation
CD_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    IC_k <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
    scores[k] <- 1 - alpha - IC_k
  }
  return(scores)
}

# Quantile Sharpness
QS_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- (u - l)
  }
  return(mean(scores))
}

# Main function to compute all scores
compute_scores_general <- function(median_samples, quantile_samples, coverages, observed) {
  # Create the ps vector based on the provided quantile samples
  ps <- c(0.50)  # Always include the median
  for (i in 1:length(coverages)) {
    ps <- c(ps, (1 - coverages[i])/2, 1 - (1 - coverages[i])/2)
  }
  ps <- sort(unique(ps))
  
  n_times <- length(observed)
  n_samples <- dim(median_samples)[2]
  n_quantiles <- length(ps)
  
  quantiles_array <- array(NA, dim = c(n_quantiles, n_times, n_samples))
  quantiles_array[which.min(abs(ps - 0.50)), , ] <- median_samples  # Assuming median is at 0.50
  
  for (i in 1:length(coverages)) {
    quantiles_array[which.min(abs(ps - (1 - coverages[i])/2)), , ] <- quantile_samples[[2 * i - 1]]
    quantiles_array[which.min(abs(ps - (1 - (1 - coverages[i])/2))), , ] <- quantile_samples[[2 * i]]
  }
  
  # Initialize results data frame with dynamic columns based on coverages
  result_columns <- c("t", "CRPS", "WIS", "Dispersion", paste0("IC_", coverages * 100), paste0("CD_", coverages * 100), "QS")
  results <- data.frame(matrix(ncol = length(result_columns), nrow = n_times))
  colnames(results) <- result_columns
  results$t <- 1:n_times
  
  # Function to compute all scores
  compute_scores <- function(y, quantiles, ps, coverages) {
    list(
      CRPS = a_CRPS(y, quantiles, ps),
      WIS = WIS_a(quantiles[which.min(abs(ps - 0.50))], quantiles, ps, y, coverages),
      Dispersion = Disp_a(quantiles, ps, coverages),
      IC = IC_a(y, quantiles, ps, coverages),
      CD = CD_a(y, quantiles, ps, coverages),
      QS = QS_a(quantiles, ps, coverages)
    )
  }
  
  # Loop through all time points
  for (t in 1:n_times) {
    quantile_values <- quantiles_array[, t, 1]  # Use the first sample for simplicity
    # print(paste("Time:", t, "Quantile values:", paste(quantile_values, collapse = ", ")))
    scores <- compute_scores(observed[t], quantile_values, ps, coverages)
    
    results$CRPS[t] <- scores$CRPS
    results$WIS[t] <- scores$WIS
    results$Dispersion[t] <- scores$Dispersion
    for (i in 1:length(coverages)) {
      results[t, paste0("IC_", coverages[i] * 100)] <- scores$IC[i]
      results[t, paste0("CD_", coverages[i] * 100)] <- scores$CD[i]
    }
    results$QS[t] <- scores$QS
  }
  
  overall_means <- colMeans(results[, -1], na.rm = TRUE)
  return(list(results = results, overall_means = overall_means))
}

# Function to compute rolling mean, cumulative mean, and cumulative sum
compute_metrics <- function(series, window_size = 360) {
  roll_mean <- rollmean(series, window_size, fill = NA)
  cum_mean <- cumsum(series) / seq_along(series)
  cum_sum <- cumsum(series) / length(series)
  list(roll_mean = roll_mean, cum_mean = cum_mean, cum_sum = cum_sum)
}

# Plotting Function with Theoretical Coverage
plot_metric_with_coverage <- function(time, series, roll_mean, cum_mean, cum_sum, metric_name, theoretical_coverage = NULL) {
  par(mfrow = c(1, 3))  # Arrange plots in 1 row, 3 columns
  
  plot(time, series, type = "l", col = "darkblue", main = paste("Time Series of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2)
  lines(time, roll_mean, col = "orange", lwd = 2)
  legend("topright", legend = c("Series", "Rolling Mean"), col = c("darkblue", "orange"), lty = 1, lwd = 2)
  abline(h=0)
  
  plot(time, cum_mean, type = "l", col = "darkred", main = paste("Cumulative Mean of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2)
  abline(h=0)
  
  plot(time, cum_sum, type = "l", col = "darkgreen", main = paste("Cumulative Sum of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2)
  abline(h=0)
  
  if (!is.null(theoretical_coverage)) {
    abline(h = theoretical_coverage, col = "purple", lwd = 2, lty = 2)
  }
}

# Function to plot all metrics
plot_all_metrics <- function(results, coverages) {
  time <- results$t
  
  # CRPS
  crps_metrics <- compute_metrics(results$CRPS)
  plot_metric_with_coverage(time, results$CRPS, crps_metrics$roll_mean, crps_metrics$cum_mean, crps_metrics$cum_sum, "CRPS")
  
  # WIS
  wis_metrics <- compute_metrics(results$WIS)
  plot_metric_with_coverage(time, results$WIS, wis_metrics$roll_mean, wis_metrics$cum_mean, wis_metrics$cum_sum, "WIS")
  
  # Dispersion
  disp_metrics <- compute_metrics(results$Dispersion)
  plot_metric_with_coverage(time, results$Dispersion, disp_metrics$roll_mean, disp_metrics$cum_mean, disp_metrics$cum_sum, "Dispersion")
  
  # IC
  for (i in 1:length(coverages)) {
    ic_metrics <- compute_metrics(results[, paste0("IC_", coverages[i] * 100)])
    plot_metric_with_coverage(time, results[, paste0("IC_", coverages[i] * 100)], ic_metrics$roll_mean, ic_metrics$cum_mean, ic_metrics$cum_sum, paste0("IC_", coverages[i] * 100), theoretical_coverage = coverages[i])
  }
  
  # CD
  for (i in 1:length(coverages)) {
    cd_metrics <- compute_metrics(results[, paste0("CD_", coverages[i] * 100)])
    plot_metric_with_coverage(time, results[, paste0("CD_", coverages[i] * 100)], cd_metrics$roll_mean, cd_metrics$cum_mean, cd_metrics$cum_sum, paste0("CD_", coverages[i] * 100))
  }
  
  # QS
  qs_metrics <- compute_metrics(results$QS)
  plot_metric_with_coverage(time, results$QS, qs_metrics$roll_mean, qs_metrics$cum_mean, qs_metrics$cum_sum, "QS")
}


median_samples <- xb_50_corrected[1, , ]
# quantile_samples <- list(xb_05_corrected[1, , ], xb_95_corrected[1, , ], xb_20_corrected[1, , ], xb_80_corrected[1, , ], xb_35_corrected[1, , ], xb_65_corrected[1, , ])
# coverages <- c(0.9, 0.6, 0.3)
quantile_samples <- list(xb_05_corrected[1, , ], xb_95_corrected[1, , ])
coverages <- c(0.9)
observed <- Y[1, ]

results <- compute_scores_general(median_samples, quantile_samples, coverages, observed)
# print(results$results)
plot_all_metrics(results$results, coverages)


In [ ]:
# Required Libraries
library(zoo)

# Check Loss Function
CheckLossFn <- function(p0, diff) {
  diff * p0 - diff * as.numeric(diff < 0)
}

# Approximated CRPS
a_CRPS <- function(y, quantiles, ps) {
  score <- 0
  for (i in 1:length(ps)) {
    diff <- y - quantiles[i]
    score <- score + CheckLossFn(ps[i], diff)
  }
  return(2 / length(ps) * score)
}

# Interval Score
IS_a <- function(l, u, a, y) {
  score <- (u - l) + 2/a * (l - y) * ifelse(y < l, 1, 0) + 2/a * (y - u) * ifelse(y >= u, 1, 0)
  return(score)
}

# Weighted Interval Score
WIS_a <- function(m, quantiles, ps, y, coverages) {
  K <- length(coverages)
  score <- 0.5 * abs(y - m)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    L <- quantiles[L_index]
    U <- quantiles[U_index]
    score <- score + alpha/2 * IS_a(L, U, alpha, y)
  }
  score <- score / (K + 0.5)
  return(score)
}

# Dispersion of WIS
Disp_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  score <- 0
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    score <- score + alpha/2 * (u - l)
  }
  return(score)
}

# Interval Coverage
IC_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
  }
  return(scores)
}

# Coverage Deviation
CD_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    IC_k <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
    scores[k] <- 1 - alpha - IC_k
  }
  return(scores)
}

# Quantile Sharpness
QS_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- (u - l)
  }
  return(mean(scores))
}

# Main function to compute all scores
compute_scores_general <- function(median_samples, quantile_samples, coverages, observed) {
  # Create the ps vector based on the provided quantile samples
  ps <- c(0.50)  # Always include the median
  for (i in 1:length(coverages)) {
    ps <- c(ps, (1 - coverages[i])/2, 1 - (1 - coverages[i])/2)
  }
  ps <- sort(unique(ps))
  
  n_times <- length(observed)
  n_samples <- dim(median_samples)[2]
  n_quantiles <- length(ps)
  
  quantiles_array <- array(NA, dim = c(n_quantiles, n_times, n_samples))
  quantiles_array[which.min(abs(ps - 0.50)), , ] <- median_samples  # Assuming median is at 0.50
  
  for (i in 1:length(coverages)) {
    quantiles_array[which.min(abs(ps - (1 - coverages[i])/2)), , ] <- quantile_samples[[2 * i - 1]]
    quantiles_array[which.min(abs(ps - (1 - (1 - coverages[i])/2))), , ] <- quantile_samples[[2 * i]]
  }
  
  # Initialize results data frame with dynamic columns based on coverages
  result_columns <- c("t", "CRPS", "WIS", "Dispersion", paste0("IC_", coverages * 100), paste0("CD_", coverages * 100), "QS")
  results <- data.frame(matrix(ncol = length(result_columns), nrow = n_times))
  colnames(results) <- result_columns
  results$t <- 1:n_times
  
  # Function to compute all scores
  compute_scores <- function(y, quantiles, ps, coverages) {
    list(
      CRPS = a_CRPS(y, quantiles, ps),
      WIS = WIS_a(quantiles[which.min(abs(ps - 0.50))], quantiles, ps, y, coverages),
      Dispersion = Disp_a(quantiles, ps, coverages),
      IC = IC_a(y, quantiles, ps, coverages),
      CD = CD_a(y, quantiles, ps, coverages),
      QS = QS_a(quantiles, ps, coverages)
    )
  }
  
  # Loop through all time points
  for (t in 1:n_times) {
    quantile_values <- quantiles_array[, t, 1]  # Use the first sample for simplicity
    scores <- compute_scores(observed[t], quantile_values, ps, coverages)
    
    results$CRPS[t] <- scores$CRPS
    results$WIS[t] <- scores$WIS
    results$Dispersion[t] <- scores$Dispersion
    for (i in 1:length(coverages)) {
      results[t, paste0("IC_", coverages[i] * 100)] <- scores$IC[i]
      results[t, paste0("CD_", coverages[i] * 100)] <- scores$CD[i]
    }
    results$QS[t] <- scores$QS
  }
  
  overall_means <- colMeans(results[, -1], na.rm = TRUE)
  return(list(results = results, overall_means = overall_means))
}

# Function to compute rolling mean, cumulative mean, and cumulative sum
compute_metrics <- function(series, window_size = 360) {
  roll_mean <- rollmean(series, window_size, fill = NA)
  cum_mean <- cumsum(series) / seq_along(series)
  cum_sum <- cumsum(series) / length(series)
  list(roll_mean = roll_mean, cum_mean = cum_mean, cum_sum = cum_sum)
}
par(mfrow = c(1, 2)) 
plot_cumulative_metrics <- function(metrics_list, metric_name, models) {
  
  plot(metrics_list[[1]]$cum_mean, type = "l", col = "darkblue", main = paste("Cumulative Mean of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2, ylim = range(sapply(metrics_list, function(x) x$cum_mean), na.rm = TRUE) * 1.1)
  for (i in 2:length(metrics_list)) {
    lines(metrics_list[[i]]$cum_mean, col = i, lwd = 2)
  }
  legend("topleft", legend = c("Av", "SL", "NDLM", "exAL"), col = 1:length(models), lty = 1, lwd = 2, cex = 0.8)
  
  plot(metrics_list[[1]]$cum_sum, type = "l", col = "darkblue", main = paste("Cumulative Sum of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2, ylim = range(sapply(metrics_list, function(x) x$cum_sum), na.rm = TRUE) * 1.1)
  for (i in 2:length(metrics_list)) {
    lines(metrics_list[[i]]$cum_sum, col = i, lwd = 2)
  }
  legend("topleft", legend = c("Av", "SL", "NDLM", "exAL"), col = 1:length(models), lty = 1, lwd = 2, cex = 0.8)
}

# Function to compute and plot results for all models
compute_and_plot_all_models <- function() {
  models <- list(
    list(
      name = "Averaged Model",
      median_samples = xb_50_AV_corrected[1, , ],
      quantile_samples = list(xb_05_AV_corrected[1, , ], xb_95_AV_corrected[1, , ]),
      coverages = c(0.9)
    ),
    list(
      name = "SL Model",
      median_samples = xb_50_SL_corrected[1, , ],
      quantile_samples = list(xb_05_SL_corrected[1, , ], xb_95_SL_corrected[1, , ]),
      coverages = c(0.9)
    ),
    list(
      name = "NDLM Model",
      median_samples = xb_M_50[1, , ],
      quantile_samples = list(xb_M_05[1, , ], xb_M_95[1, , ]),
      coverages = c(0.9)
    ),
    list(
      name = "exAL Model",
      median_samples = xb_50_corrected[1, , ],
      quantile_samples = list(xb_05_corrected[1, , ], xb_95_corrected[1, , ]),
      coverages = c(0.9)
    )
  )
  
  observed <- Y[1, ]
  all_results <- list()
  overall_means_table <- data.frame()
  
  for (model in models) {
    cat("Processing", model$name, "\n")
    results <- compute_scores_general(model$median_samples, model$quantile_samples, model$coverages, observed)
    all_results[[model$name]] <- results
    overall_means_table <- rbind(overall_means_table, c(model$name, results$overall_means))
  }
  
  colnames(overall_means_table) <- c("Model", names(results$overall_means))
  print(overall_means_table)
  
  # Plot cumulative means and sums for each score
  scores <- c("CRPS", "WIS", "Dispersion", "QS")
  for (score in scores) {
    metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[score]]))
    plot_cumulative_metrics(metrics_list, score, names(all_results))
  }
  
  coverages <- unique(unlist(lapply(models, function(model) model$coverages)))
  for (coverage in coverages) {
    ic_metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[paste0("IC_", coverage * 100)]]))
    plot_cumulative_metrics(ic_metrics_list, paste0("IC_", coverage * 100), names(all_results))
    
    cd_metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[paste0("CD_", coverage * 100)]]))
    plot_cumulative_metrics(cd_metrics_list, paste0("CD_", coverage * 100), names(all_results))
  }
}

# Run the computation and plotting for all models
compute_and_plot_all_models()


In [ ]:
# Required Libraries
library(zoo)

# Check Loss Function
CheckLossFn <- function(p0, diff) {
  diff * p0 - diff * as.numeric(diff < 0)
}

# Approximated CRPS
a_CRPS <- function(y, quantiles, ps) {
  score <- 0
  for (i in 1:length(ps)) {
    diff <- y - quantiles[i]
    score <- score + CheckLossFn(ps[i], diff)
  }
  return(2 / length(ps) * score)
}

# Interval Score
IS_a <- function(l, u, a, y) {
  score <- (u - l) + 2/a * (l - y) * ifelse(y < l, 1, 0) + 2/a * (y - u) * ifelse(y >= u, 1, 0)
  return(score)
}

# Weighted Interval Score
WIS_a <- function(m, quantiles, ps, y, coverages) {
  K <- length(coverages)
  score <- 0.5 * abs(y - m)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    L <- quantiles[L_index]
    U <- quantiles[U_index]
    score <- score + alpha/2 * IS_a(L, U, alpha, y)
  }
  score <- score / (K + 0.5)
  return(score)
}

# Dispersion of WIS
Disp_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  score <- 0
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    score <- score + alpha/2 * (u - l)
  }
  return(score)
}

# Interval Coverage
IC_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
  }
  return(scores)
}

# Coverage Deviation
CD_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    IC_k <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
    scores[k] <- 1 - alpha - IC_k
  }
  return(scores)
}

# Quantile Sharpness
QS_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- (u - l)
  }
  return(mean(scores))
}

# Main function to compute all scores
compute_scores_general <- function(median_samples, quantile_samples, coverages, observed) {
  # Create the ps vector based on the provided quantile samples
  ps <- c(0.50)  # Always include the median
  for (i in 1:length(coverages)) {
    ps <- c(ps, (1 - coverages[i])/2, 1 - (1 - coverages[i])/2)
  }
  ps <- sort(unique(ps))

  n_times <- length(observed)
  n_samples <- dim(median_samples)[2]
  n_quantiles <- length(ps)
  
  quantiles_array <- array(NA, dim = c(n_quantiles, n_times, n_samples))
  quantiles_array[which.min(abs(ps - 0.50)), , ] <- median_samples  # Assuming median is at 0.50
  
  for (i in 1:length(coverages)) {
    quantiles_array[which.min(abs(ps - (1 - coverages[i])/2)), , ] <- quantile_samples[[2 * i - 1]]
    quantiles_array[which.min(abs(ps - (1 - (1 - coverages[i])/2))), , ] <- quantile_samples[[2 * i]]
  }

  # Initialize results data frame with dynamic columns based on coverages
  result_columns <- c("t", "CRPS", "WIS", "Dispersion", paste0("IC_", coverages * 100), paste0("CD_", coverages * 100), "QS")
  results <- data.frame(matrix(ncol = length(result_columns), nrow = n_times))
  colnames(results) <- result_columns
  results$t <- 1:n_times
  
  # Function to compute all scores
  compute_scores <- function(y, quantiles, ps, coverages) {
    list(
      CRPS = a_CRPS(y, quantiles, ps),
      WIS = WIS_a(quantiles[which.min(abs(ps - 0.50))], quantiles, ps, y, coverages),
      Dispersion = Disp_a(quantiles, ps, coverages),
      IC = IC_a(y, quantiles, ps, coverages),
      CD = CD_a(y, quantiles, ps, coverages),
      QS = QS_a(quantiles, ps, coverages)
    )
  }
  
  # Loop through all time points
  for (t in 1:n_times) {
    quantile_values <- quantiles_array[, t, 1]  # Use the first sample for simplicity
    scores <- compute_scores(observed[t], quantile_values, ps, coverages)
    
    results$CRPS[t] <- scores$CRPS
    results$WIS[t] <- scores$WIS
    results$Dispersion[t] <- scores$Dispersion
    for (i in 1:length(coverages)) {
      results[t, paste0("IC_", coverages[i] * 100)] <- scores$IC[i]
      results[t, paste0("CD_", coverages[i] * 100)] <- scores$CD[i]
    }
    results$QS[t] <- scores$QS
  }
  
  overall_means <- colMeans(results[, -1], na.rm = TRUE)
  return(list(results = results, overall_means = overall_means))
}

# Function to compute rolling mean, cumulative mean, and cumulative sum
compute_metrics <- function(series, window_size = 360) {
  roll_mean <- rollmean(series, window_size, fill = NA)
  cum_mean <- cumsum(series) / seq_along(series)
  cum_sum <- cumsum(series) / length(series)
  list(roll_mean = roll_mean, cum_mean = cum_mean, cum_sum = cum_sum)
}

par(mfrow = c(3, 2))  # Arrange plots in 1 row, 2 columns
plot_cumulative_metrics <- function(metrics_list, metric_name, models) {
  
  plot(metrics_list[[1]]$cum_mean, type = "l", col = "darkblue", main = paste("Cumulative Mean of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2, ylim = range(sapply(metrics_list, function(x) x$cum_mean), na.rm = TRUE) * 1.1)
  for (i in 2:length(metrics_list)) {
    lines(metrics_list[[i]]$cum_mean, col = i, lwd = 2)
  }
  legend("topleft", legend = models, col = 1:length(models), lty = 1, lwd = 2, cex = 0.8)
  
  plot(metrics_list[[1]]$cum_sum, type = "l", col = "darkblue", main = paste("Cumulative Sum of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2, ylim = range(sapply(metrics_list, function(x) x$cum_sum), na.rm = TRUE) * 1.1)
  for (i in 2:length(metrics_list)) {
    lines(metrics_list[[i]]$cum_sum, col = i, lwd = 2)
  }
  legend("topleft", legend = models, col = 1:length(models), lty = 1, lwd = 2, cex = 0.8)
}

# Function to compute and plot results for all models
compute_and_plot_all_models <- function() {
  models <- list(
    list(
      name = "Averaged Model",
      median_samples = xb_50_AV_forecast[1,,],
      quantile_samples = list(xb_05_AV_forecast[1,,], xb_95_AV_forecast[1,,]),
      coverages = c(0.9)
    ),
    list(
      name = "SL Model",
      median_samples = xb_50_SL_forecast[1,,],
      quantile_samples = list(xb_05_SL_forecast[1,,], xb_95_SL_forecast[1,,]),
      coverages = c(0.9)
    ),
    list(
      name = "NDLM Model",
      median_samples = xb_50_NDLM_forecast[1,,],
      quantile_samples = list(xb_05_NDLM_forecast[1,,], xb_95_NDLM_forecast[1,,]),
      coverages = c(0.9)
    ),
    list(
      name = "exAL Model",
      median_samples = xb_50_M_forecast[1,,],
      quantile_samples = list(xb_05_M_forecast[1,,], xb_95_M_forecast[1,,]),
      coverages = c(0.9)
    )
  )
  
  observed <- truth$std_discharge_cms
  all_results <- list()
  overall_means_table <- data.frame()
  
  for (model in models) {
    cat("Processing", model$name, "\n")
    results <- compute_scores_general(model$median_samples, model$quantile_samples, model$coverages, observed)
    all_results[[model$name]] <- results
    overall_means_table <- rbind(overall_means_table, c(model$name, results$overall_means))
  }
  
  colnames(overall_means_table) <- c("Model", names(results$overall_means))
  print(overall_means_table)
  
  # Plot cumulative means and sums for each score
  scores <- c("CRPS", "WIS", "Dispersion", "QS")
  for (score in scores) {
    metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[score]]))
    plot_cumulative_metrics(metrics_list, score, names(all_results))
  }
  
  coverages <- unique(unlist(lapply(models, function(model) model$coverages)))
  for (coverage in coverages) {
    ic_metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[paste0("IC_", coverage * 100)]]))
    plot_cumulative_metrics(ic_metrics_list, paste0("IC_", coverage * 100), names(all_results))
    
    cd_metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[paste0("CD_", coverage * 100)]]))
    plot_cumulative_metrics(cd_metrics_list, paste0("CD_", coverage * 100), names(all_results))
  }
}

# Run the computation and plotting for all models
compute_and_plot_all_models()


## Spectral Analysis

In [ ]:
# Function to calculate w values for spectral analysis
calculate_w_values <- function(T) {
  m <- floor(T / 2)
  K <- 1:m
  2 * pi * K / T
}

periodogram <- function(w, y) {
  T <- length(y)
  n <- length(w)
  s <- matrix(exp(-1i * outer(w, 1:T, "*")), nrow = n, ncol = T)
  I <- numeric(n)
  
  for (j in 1:n) {
    sum_s <- sum(y * s[j, ])
    I[j] <- abs(sum_s)^2 * 2 / T
  }
  
  I
}


# Log-likelihood function
loglikelihood_wavelength <- function(I, y) {
  T <- length(y)
  like <- 1 - I / sum(y^2)
  ((2 - T) / 2) * log(like)
}

# Function to perform spectral analysis
perform_spectral_analysis <- function(ts_data) {
  T <- length(ts_data)
  w <- calculate_w_values(T)
  I <- periodogram(w, ts_data)
  like <- loglikelihood_wavelength(I, ts_data)
  
  x <- 2 * pi / w
  like_filtered <- like[x < 6000]
  x_filtered <- x[x < 6000]
  
  # Improved peak detection
  peaks <- which(diff(sign(diff(like_filtered))) == -2) + 1
  peak_x <- x_filtered[peaks]
  peak_y <- like_filtered[peaks]
  
  list(like = like_filtered, x = x_filtered, peak_x = peak_x, peak_y = peak_y)
}

####################################################################################
####################################################################################

# Reading the CSV file

data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned.csv"
streamflow_data <- read_csv(data_path, show_col_types = FALSE)
timestamps <- as.Date(streamflow_data$Date)
time_series_matrix <- as.matrix(streamflow_data[, c('USGS', 'NWS3.0',  'GloFAS')])

# Create time series
start_year <- as.numeric(format(min(timestamps), "%Y"))
usgs_ts <- ts(time_series_matrix[,1], start = c(start_year, 1), frequency = 365)
glofas_ts <- ts(time_series_matrix[,3], start = c(start_year, 1), frequency = 365)
nws_ts <- ts(time_series_matrix[,2], start = c(start_year, 1), frequency = 365)

####################################################################################
####################################################################################

# Perform spectral analysis
usgs_spectral <- perform_spectral_analysis(usgs_ts)
glofas_spectral <- perform_spectral_analysis(glofas_ts)
nws_spectral <- perform_spectral_analysis(nws_ts)

# Plotting spectral analysis
plot_spectral <- function(spectral_data, title, color) {
  plot(spectral_data$x, spectral_data$like, type = "l", xlab = "Period (in days)", ylab = "log-likelihood", main = title, xlim =c(0,1500))
  points(spectral_data$peak_x, spectral_data$peak_y, pch = 16, col = color)
}


# Adjusting graphical parameters for better fit and efficient use of space
par(mfrow = c(3, 1),  # Setting layout to 3 rows, 1 column
    mar = c(2, 4, 2, 1) + 0.1,  # Setting margins: bottom, left, top, right
    oma = c(0.5, 0.5, 0.5, 0.5),  # Outer margins
    omd = c(0.1, 0.9, 0.1, 0.9))  # Outer margin dimensions

# Plotting spectral analysis for each series
plot_spectral(usgs_spectral, "Spectral Analysis: USGS", "red")
plot_spectral(glofas_spectral, "Spectral Analysis: GloFAS", "blue")
plot_spectral(nws_spectral, "Spectral Analysis: NWS", "green")

dev.off()
####################################################################################
####################################################################################

# Function to extract and combine peak information
extract_peak_info <- function(spectral_data, label) {
  data.frame(
    Series = rep(label, length(spectral_data$peak_x)),
    Peak_Period_Days = spectral_data$peak_x,
    Peak_Period_Years = spectral_data$peak_x / 365,
    Peak_Value = spectral_data$peak_y
  )
}

# Function to extract top 5 peaks
extract_top_peaks <- function(peaks_df) {
  peaks_df[order(-peaks_df$Peak_Value),][1:5, ]
}

# Extract peak information for each time series
usgs_peaks <- extract_peak_info(usgs_spectral, "USGS")
glofas_peaks <- extract_peak_info(glofas_spectral, "GloFAS")
nws_peaks <- extract_peak_info(nws_spectral, "NWS")

# Extract top 5 peaks for each time series
usgs_top_peaks <- extract_top_peaks(usgs_peaks)
glofas_top_peaks <- extract_top_peaks(glofas_peaks)
nws_top_peaks <- extract_top_peaks(nws_peaks)

# Combine the top peak information into a single data frame
all_top_peaks <- rbind(usgs_top_peaks, glofas_top_peaks, nws_top_peaks)

# Print the combined top peak information
print(all_top_peaks)

In [ ]:
idxxx <- (TT-1000):TT
plot.ts(Y[1,idxxx], col = 'gray', lwd = 2)
lines((new.theta.out__NDLM_uni$exps[1,idxxx]), col = 'orange')
lines((new.theta.out_50_exAL$exps[1,idxxx]), col = 'red')
lines((new.theta.out_5_exAL$exps[1,idxxx]), col = 'green')
lines((new.theta.out_95_exAL$exps[1,idxxx]), col = 'blue')
s <- 0
# lines(s+new.theta.out_50_exAL$sm[2,idxxx], col = 'blue')
# lines(s+new.theta.out_50_exAL$sm[4,idxxx], col = 'blue')
# lines(s+new.theta.out_50_exAL$sm[6,idxxx], col = 'blue')
# plot.ts(s+new.theta.out_50_exAL$sm[8,idxxx], col = 'blue')
# plot.ts(s+new.theta.out_50_exAL$sm[9,idxxx], col = 'blue')
# lines(s+new.theta.out_50_exAL$sm[10,idxxx], col = 'blue')

# plot.ts(Y[1,idxxx], col = 'gray', lwd = 2)
# lines((new.theta.out__NDLM_uni$sm[1,idxxx]), col = 'darkred')
# lines((new.theta.out__NDLM_uni$sm[2,idxxx]), col = 'darkred')
# lines((new.theta.out__NDLM_uni$sm[4,idxxx]), col = 'darkred')
# lines((new.theta.out__NDLM_uni$sm[6,idxxx]), col = 'darkred')

# lines((new.theta.out__NDLM_uni$sm[9,idxxx]), col = 'darkred')
# lines((new.theta.out__NDLM_uni$sm[10,idxxx]), col = 'darkred')


In [ ]:
# Set up the plotting area for a 6x2 matrix layout
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 3, 0))

# Define the colors for each time series
colors <- c("forestgreen", "darkorange", "darkblue")

# Plot each time series with the specified colors
ts.plot(t(seq.sigma_50_exAL), col = colors, main = "Sigma 50th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_5_exAL), col = colors, main = "Sigma 05th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_95_exAL), col = colors, main = "Sigma 95th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.gamma_50_exAL), col = colors, main = "Gamma 50th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.gamma_5_exAL), col = colors, main = "Gamma 05th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.gamma_95_exAL), col = colors, main = "Gamma 95th", xlab = "Iteration", ylab = "Gamma")

# Add a common legend to the plot
# Placing the legend at the top of the first column (adjust `oma` and `mar` for space)
mtext("Green - USGS, Orange - GLOFAS, Blue - NWS", side = 3, outer = TRUE, line = 0, cex = 0.8)

par(mfrow = c(2, 4), mar = c(4, 4, 2, 1), oma = c(0, 0, 3, 0))
# Plot each time series with the specified colors
ts.plot(t(seq.sigma_20_exAL), col = colors, main = "Sigma 20th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_35_exAL), col = colors, main = "Sigma 35th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_65_exAL), col = colors, main = "Sigma 65th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.sigma_80_exAL), col = colors, main = "Sigma 80th", xlab = "Iteration", ylab = "Sigma")
ts.plot(t(seq.gamma_20_exAL), col = colors, main = "Gamma 20th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.gamma_35_exAL), col = colors, main = "Gamma 35th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.gamma_65_exAL), col = colors, main = "Gamma 65th", xlab = "Iteration", ylab = "Gamma")
ts.plot(t(seq.sigma_80_exAL), col = colors, main = "Sigma 80th", xlab = "Iteration", ylab = "Sigma")

# Add a common legend to the plot
# Placing the legend at the top of the first column (adjust `oma` and `mar` for space)
mtext("Green - USGS, Orange - GLOFAS, Blue - NWS", side = 3, outer = TRUE, line = 0, cex = 0.8)

# Resetting the plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
par(mfrow = c(2, 2), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

a <- c(seq.elbo_50_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL50", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_5_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL05", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_95_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL95", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo__NDLM)
a[1:3]=NaN
plot.ts(a, main = "ELBO -NDLM", xlab = "Iteration", ylab = "ELBO")

par(mfrow = c(2, 2), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))
a <- c(seq.elbo_20_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL20", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_35_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL35", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_65_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL65", xlab = "Iteration", ylab = "ELBO")
a <- c(seq.elbo_80_exAL)
a[1:19]=NaN
plot.ts(a, main = "ELBO -exAL80", xlab = "Iteration", ylab = "ELBO")

In [ ]:
compute_xb_corrected <- function(samp_theta, FF) {
  # Determine dimensions
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  # Allocate space for the result: 3x12009x2000
  xb <- array(NA, dim = c(3, n_time, n_sim))
  
  # Loop over time points
  for (t in 1:n_time) {
    FF_t <- FF[,,t]  # 9x3 matrix for time t
    
    # Extract all simulations for time t across all components: 9x2000 matrix
    theta_t_s <- samp_theta[, t, ]
    
    # Perform matrix multiplication
    xb[, t, ] <- t(FF_t) %*% theta_t_s  # Results in a 3x2000 matrix
  }
  
  return(xb)
}


In [ ]:
xb_50_corrected <- compute_xb_corrected(samp.theta_50_exAL, FF)
xb_05_corrected <- compute_xb_corrected(samp.theta_5_exAL, FF)
xb_95_corrected <- compute_xb_corrected(samp.theta_95_exAL, FF)
quantiles_xb_50_corrected <- apply(xb_50_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_05_corrected <- apply(xb_05_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_95_corrected <- apply(xb_95_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))


In [ ]:
xb_20_corrected <- compute_xb_corrected(samp.theta_20_exAL, FF)
xb_35_corrected <- compute_xb_corrected(samp.theta_35_exAL, FF)
xb_65_corrected <- compute_xb_corrected(samp.theta_65_exAL, FF)
xb_80_corrected <- compute_xb_corrected(samp.theta_80_exAL, FF)
quantiles_xb_20_corrected <- apply(xb_20_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_35_corrected <- apply(xb_35_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_65_corrected <- apply(xb_65_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_80_corrected <- apply(xb_80_corrected, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))


In [ ]:
# Define a function to calculate quantiles for each dataset
calculate_quantiles <- function(data, variable_name, quantile_name, source_name) {
  quantile_values <- quantile(data, probs = c(0.025, 0.5, 0.975))
  tibble(
    variable = variable_name,
    source = source_name,
    quantile = quantile_name,
    quantile_025 = quantile_values["2.5%"],
    median = quantile_values["50%"],
    quantile_975 = quantile_values["97.5%"]
  )
}

# List of datasets and their metadata
data_sets <- list(
  gamma_50_M = list(data = samp.gamma_50_exAL, quantile = "50th", variable = "Gamma"),
  gamma_95_M = list(data = samp.gamma_95_exAL, quantile = "95th", variable = "Gamma"),
  gamma_05_M = list(data = samp.gamma_5_exAL, quantile = "05th", variable = "Gamma"),
  gamma_20_M = list(data = samp.gamma_20_exAL, quantile = "20th", variable = "Gamma"),
  gamma_35_M = list(data = samp.gamma_35_exAL, quantile = "35th", variable = "Gamma"),
  gamma_65_M = list(data = samp.gamma_65_exAL, quantile = "65th", variable = "Gamma"),
  gamma_80_M = list(data = samp.gamma_80_exAL, quantile = "80th", variable = "Gamma"),
  sigma_50_M = list(data = samp.sigma_50_exAL, quantile = "50th", variable = "Sigma"),
  sigma_95_M = list(data = samp.sigma_95_exAL, quantile = "95th", variable = "Sigma"),
  sigma_05_M = list(data = samp.sigma_5_exAL, quantile = "05th", variable = "Sigma"),
  sigma_20_M = list(data = samp.sigma_20_exAL, quantile = "20th", variable = "Sigma"),
  sigma_35_M = list(data = samp.sigma_35_exAL, quantile = "35th", variable = "Sigma"),
  sigma_65_M = list(data = samp.sigma_65_exAL, quantile = "65th", variable = "Sigma"),
  sigma_80_M = list(data = samp.sigma_80_exAL, quantile = "80th", variable = "Sigma")
)

# # List of datasets and their metadata
# data_sets <- list(
#   gamma_50_M = list(data = samp.gamma_50_exAL, quantile = "50th", variable = "Gamma"),
#   gamma_95_M = list(data = samp.gamma_95_exAL, quantile = "95th", variable = "Gamma"),
#   gamma_05_M = list(data = samp.gamma_5_exAL, quantile = "05th", variable = "Gamma"),
#   sigma_50_M = list(data = samp.sigma_50_exAL, quantile = "50th", variable = "Sigma"),
#   sigma_95_M = list(data = samp.sigma_95_exAL, quantile = "95th", variable = "Sigma"),
#   sigma_05_M = list(data = samp.sigma_5_exAL, quantile = "05th", variable = "Sigma")
# )

# Calculate quantiles for each dataset and source
all_quantiles <- bind_rows(
  lapply(data_sets, function(item) {
    bind_rows(
      calculate_quantiles(item$data[, 1], item$variable, item$quantile, "USGS"),
      calculate_quantiles(item$data[, 2], item$variable, item$quantile, "GLOFAS"),
      calculate_quantiles(item$data[, 3], item$variable, item$quantile, "NWS")
    )
  })
)

# Print the complete table of quantiles
print(all_quantiles, n = Inf)


In [ ]:
prepare_quantile_data <- function(v_d) {
  v_d_transposed <- aperm(v_d, c(3, 1, 2))
  q_d_transposed <- apply(v_d_transposed, 2:3, function(x) quantile(x, probs = c(0.975, 0.5, 0.025)))
  q_d <- aperm(q_d_transposed, c(2, 3, 1))
  return(q_d)
}


In [ ]:
q_d_50 <- prepare_quantile_data(samp.theta_50_exAL)
q_d_05 <- prepare_quantile_data(samp.theta_5_exAL)
q_d_95 <- prepare_quantile_data(samp.theta_95_exAL)


In [ ]:
q_d_20 <- prepare_quantile_data(samp.theta_20_exAL)
q_d_35 <- prepare_quantile_data(samp.theta_35_exAL)
q_d_65 <- prepare_quantile_data(samp.theta_65_exAL)
q_d_80 <- prepare_quantile_data(samp.theta_80_exAL)

In [ ]:
dates_ts_usgs <- timestamps

In [ ]:
# Function to plot with quantiles and dates on x-axis
# plot_quantile_component <- function(q_d_50, q_d_05, q_d_95,q_d_20,q_d_35,q_d_65,q_d_80, Y, idx, component, main_label, num_ticks) {
plot_quantile_component <- function(q_d_50, q_d_05, q_d_95, Y, idx, component, main_label, num_ticks) {

  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d_50[component, idx, ], q_d_05[component, idx, ], q_d_95[component, idx, ])) * 2
  ylims[2] <- min(1.3, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.3, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

  lines(idx, q_d_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)
  
  lines(idx, q_d_20[component, idx, 2], col = "purple", lwd = 1)
  lines(idx, q_d_35[component, idx, 2], col = "purple", lwd = 1)
  lines(idx, q_d_65[component, idx, 2], col = "purple", lwd = 1)
  lines(idx, q_d_80[component, idx, 2], col = "purple", lwd = 1)

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


In [ ]:
# Set up plotting window for a 2x3 matrix layout
par(mfrow = c(2, 1), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

# Index range and plotting
idx <- ceiling(TT/2):TT
components <- c(1, 2, 4, 6, 8, 9, 10, 11, 12)
component_labels <- c("Trend Component", "First Harmonic", "Second Harmonic", "(1/6.83) Harmonic", 
                      "Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")

for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  # plot_quantile_component(q_d_50,q_d_05, q_d_95, q_d_20, q_d_35, q_d_65, q_d_80, Y, idx, components[i], component_labels[i], num_ticks = 8)
  plot_quantile_component(q_d_50,q_d_05, q_d_95, Y, idx, components[i], component_labels[i], num_ticks = 8)

}

# Add a common legend or note at the bottom
mtext("Legend: Forest Green - 50_M, Dark Red - 05_M, Dark Blue - 95_M", side = 1, outer = TRUE, line = 2, cex = 0.8)

# Reset plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
prepare_quantile_data <- function(v_d) {
  v_d_transposed <- aperm(v_d, c(3, 1, 2))
  q_d_transposed <- apply(v_d_transposed, 2:3, function(x) quantile(x, probs = c(0.975, 0.5, 0.025)))
  q_d <- aperm(q_d_transposed, c(2, 3, 1))
  return(q_d)
}

# Apply the function to each dataset
q_d_NDLM <- prepare_quantile_data(samp.theta__NDLM)


In [ ]:
# Function to plot with quantiles and dates on x-axis
plot_quantile_component_NDLM <- function(q_d, Y, idx, component, main_label, num_ticks) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d[component, idx, ])) * 3
  ylims[2] <- min(2, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.5, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

 # Adding NDLM quantiles
  lines(idx, q_d[component, idx, 1], col = "darkorange", lwd = 0.5, lty=2)  # Lower bound
  lines(idx, q_d[component, idx, 3], col = "darkorange", lwd = 0.5, lty=2)  # Upper bound
  lines(idx, q_d[component, idx, 2], col = "darkorange", lwd = 1)            # Median line

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


In [ ]:
# Set up plotting window for a 2x3 matrix layout
par(mfrow = c(2, 1), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

# Index range and plotting
idx <- 1:TT
components <- c(1, 2, 4, 6, 8, 9)
component_labels <- c("Trend Component", "First Harmonic", "Second Harmonic", "(1/6.83) Harmonic", 
                      "Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")

for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component_NDLM (q_d_NDLM, Y, idx, components[i], component_labels[i], num_ticks = 8)
}

# Reset plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
# Function to plot with quantiles and dates on x-axis
plot_quantile_component_all <- function(q_d, q_d_50, q_d_05, q_d_95, Y, idx, component, main_label, num_ticks) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d[component, idx, ], q_d_50[component, idx, ], q_d_05[component, idx, ], q_d_95[component, idx, ])) * 2
  ylims[2] <- min(1.3, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.3, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

  lines(idx, q_d_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d[component, idx, 1], col = "darkorange", lwd = 0.5, lty=2)  # Lower bound
  lines(idx, q_d[component, idx, 3], col = "darkorange", lwd = 0.5, lty=2)  # Upper bound
  lines(idx, q_d[component, idx, 2], col = "darkorange", lwd = 1)            # Median line

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


In [ ]:
# Set up plotting window for a 2x3 matrix layout
par(mfrow = c(2, 1), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

# Index range and plotting
idx <- 1:TT
components <- c(1, 2, 4, 6, 8, 9)
component_labels <- c("Trend Component", "First Harmonic", "Second Harmonic", "(1/6.83) Harmonic", 
                      "Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")

for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component_all(q_d_NDLM, q_d_50, q_d_05, q_d_95, Y, idx, components[i], component_labels[i], num_ticks = 8)
}

# Add a common legend or note at the bottom
mtext("Legend: Orange: Mean (NDLM), Forest Green - 50_M, Dark Red - 05_M, Dark Blue - 95_M", side = 1, outer = TRUE, line = 2, cex = 0.8)

# Reset plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
compute_xb_corrected <- function(samp_theta, FF, sig.samp, pp) {
  # Determine dimensions
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  # Allocate space for the result: 3x12009x2000
  xb <- array(NA, dim = c(3, n_time, n_sim))
  
  # Loop over time points
  for (t in 1:n_time) {
    FF_t <- FF[,,t]  # 9x3 matrix for time t
    
    # Extract all simulations for time t across all components: 9x2000 matrix
    theta_t_s <- samp_theta[, t, ]
    
    # Perform matrix multiplication
    xb[, t, ] <- t(FF_t) %*% theta_t_s + t(sqrt(sig.samp))*qnorm(pp) 
  }
  
  return(xb)
}


In [ ]:

# # Apply the corrected function to each samp.theta matrix
# xb_M_50 <- compute_xb_corrected(samp.theta_M, FF, samp.sigma_M, 0.5)
# xb_M_05 <- compute_xb_corrected(samp.theta_M, FF, samp.sigma_M, 0.05)
# xb_M_95 <- compute_xb_corrected(samp.theta_M, FF, samp.sigma_M, 0.95)

# # Compute quantiles for xb_50 as an example (apply this logic to xb_05 and xb_95 as needed)
# quantiles_xb_M_50 <- apply(xb_M_50, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
# quantiles_xb_M_05 <- apply(xb_M_05, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
# quantiles_xb_M_95 <- apply(xb_M_95, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))




In [ ]:
compute_xb_corrected <- function(samp_theta, FF, sig.samp, pp) {
  # Determine dimensions
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  # Allocate space for the result: 3x12009x2000
  xb <- array(NA, dim = c(3, n_time, n_sim))
  
  # Loop over time points
  for (t in 1:n_time) {
    FF_t <- FF[,,t]  # 9x3 matrix for time t
    
    # Extract all simulations for time t across all components: 9x2000 matrix
    theta_t_s <- samp_theta[, t, ]
    
    # Perform matrix multiplication
    xb[, t, ] <- t(FF_t) %*% theta_t_s + t(sqrt(sig.samp))*qnorm(pp) 
  }
  
  return(xb)
}

# Apply the corrected function to each samp.theta matrix
xb_M_50 <- compute_xb_corrected(samp.theta__NDLM, FF, samp.sigma__NDLM, 0.5)
xb_M_05 <- compute_xb_corrected(samp.theta__NDLM, FF, samp.sigma__NDLM, 0.05)
xb_M_95 <- compute_xb_corrected(samp.theta__NDLM, FF, samp.sigma__NDLM, 0.95)

# Compute quantiles for xb_50 as an example (apply this logic to xb_05 and xb_95 as needed)
quantiles_xb_M_50 <- apply(xb_M_50, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_M_05 <- apply(xb_M_05, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_M_95 <- apply(xb_M_95, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))


In [ ]:
# idx <- 15810:16032
idx <- (TT-500):TT

# Function to create nice date labels
create_date_labels <- function(idx, num_labels = 10) {
    selected_dates <- dates_ts_usgs[idx]  # assuming dates_ts_usgs is an array of dates corresponding to idx
    tick_positions <- pretty(idx, num_labels)
    tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")
    return(list(tick_positions = tick_positions, tick_labels = tick_labels))
}

# Setting up the plotting window
par(mfrow = c(2, 1), mar = c(3.2, 4, 2, 1) + 0.1, oma = c(4, 0, 0, 0))

# Calculate date labels outside of the plotting function for consistency
date_info <- create_date_labels(idx)

# First plot
plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="NDLM", xaxt="n")

lines(idx, quantiles_xb_M_50[2, 1, idx], col="lightgreen", lwd=1) 
lines(idx, quantiles_xb_M_50[3, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_50[1, 1, idx], col="lightgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_M_05[2, 1, idx], col="pink", lwd=1) 
lines(idx, quantiles_xb_M_05[3, 1, idx], col="pink", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_05[1, 1, idx], col="pink", lwd=0.5, lty=2)
lines(idx, quantiles_xb_M_95[2, 1, idx], col="lightblue", lwd=1) 
lines(idx, quantiles_xb_M_95[3, 1, idx], col="lightblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_95[1, 1, idx], col="lightblue", lwd=0.5, lty=2) 
axis(1, at = date_info$tick_positions, labels = FALSE)

# Adding rotated text labels
text(x = date_info$tick_positions, y = min(par("usr")[3], -1.5), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Second plot
plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="exAL", xaxt="n")

lines(idx, quantiles_xb_50_corrected[2, 1, idx], col="forestgreen", lwd=1) 
lines(idx, quantiles_xb_50_corrected[1, 1, idx], col="forestgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_50_corrected[3, 1, idx], col="forestgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_05_corrected[2, 1, idx], col="darkred", lwd=1) 
lines(idx, quantiles_xb_05_corrected[1, 1, idx], col="darkred", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_05_corrected[3, 1, idx], col="darkred", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_95_corrected[2, 1, idx], col="darkblue", lwd=1) 
lines(idx, quantiles_xb_95_corrected[1, 1, idx], col="darkblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_95_corrected[3, 1, idx], col="darkblue", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_20_corrected[2, 1, idx], col="purple", lwd=0.5) 
lines(idx, quantiles_xb_35_corrected[2, 1, idx], col="purple", lwd=0.5) 
lines(idx, quantiles_xb_65_corrected[2, 1, idx], col="purple", lwd=0.5) 
lines(idx, quantiles_xb_80_corrected[2, 1, idx], col="purple", lwd=0.5) 

axis(1, at = date_info$tick_positions, labels = FALSE)

# Adding rotated text labels for the second plot
text(x = date_info$tick_positions, y = min(par("usr")[3], -1.5), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Reset the plotting parameters to default after plotting is done
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
idx <- (TT-1000):TT
# Function to create nice date labels
create_date_labels <- function(idx, num_labels = 8) {
    selected_dates <- dates_ts_usgs[idx]  # assuming dates_ts_usgs is an array of dates corresponding to idx
    tick_positions <- pretty(idx, num_labels)
    tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")
    return(list(tick_positions = tick_positions, tick_labels = tick_labels))
}

par(mfrow = c(3, 1), mar = c(3.2, 4, 2, 1) + 0.1, oma = c(4, 0, 0, 0))
date_info <- create_date_labels(idx)

plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="NDLM vs exAL: 95th", xaxt="n")

lines(idx, quantiles_xb_M_95[2, 1, idx], col="lightblue", lwd=1) 
lines(idx, quantiles_xb_M_95[3, 1, idx], col="lightblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_95[1, 1, idx], col="lightblue", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_95_corrected[2, 1, idx], col="darkblue", lwd=1) 
lines(idx, quantiles_xb_95_corrected[1, 1, idx], col="darkblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_95_corrected[3, 1, idx], col="darkblue", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="NDLM vs exAL: 05th", xaxt="n")

lines(idx, quantiles_xb_M_05[2, 1, idx], col="pink", lwd=1) 
lines(idx, quantiles_xb_M_05[3, 1, idx], col="pink", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_05[1, 1, idx], col="pink", lwd=0.5, lty=2)
lines(idx, quantiles_xb_05_corrected[2, 1, idx], col="darkred", lwd=1) 
lines(idx, quantiles_xb_05_corrected[1, 1, idx], col="darkred", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_05_corrected[3, 1, idx], col="darkred", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)


plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="NDLM vs exA: 50th", xaxt="n")

lines(idx, quantiles_xb_M_50[2, 1, idx], col="lightgreen", lwd=1) 
lines(idx, quantiles_xb_M_50[3, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_M_50[1, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_50_corrected[2, 1, idx], col="forestgreen", lwd=1) 
lines(idx, quantiles_xb_50_corrected[1, 1, idx], col="forestgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_50_corrected[3, 1, idx], col="forestgreen", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Reset the plotting parameters to default after plotting is done
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
# v <- xb_95_corrected
# boolean_array <- sweep(v[1,,], MARGIN = 1, STATS = Y[1,], FUN = ">")

# Metrics 

### cv metric?

## I. ELBO 
### $\text{ELBO}(\theta, \phi) = \mathbb{E}_{q_\phi(z)}[\log p_\theta(x, z) - \log q_\phi(z)]$

In [ ]:
elbo_values <- data.frame(
  Model = c("NDLM", "exAL-0.5", "exAL-0.05", "exAL-0.95"),
  ELBO = c(seq.elbo__NDLM[length(seq.elbo__NDLM)],
          seq.elbo_50_exAL[dim(seq.elbo_50_exAL)[2]],
           seq.elbo_5_exAL[dim(seq.elbo_5_exAL)[2]],
           seq.elbo_95_exAL[dim(seq.elbo_95_exAL)[2]])
)

print(elbo_values)


In [ ]:
post_mean_50_exAL <- apply(samp.post.pred_50_exAL, c(1, 2), mean)
post_mean_5_exAL <- apply(samp.post.pred_5_exAL, c(1, 2), mean)
post_mean_95_exAL <- apply(samp.post.pred_95_exAL, c(1, 2), mean)

In [ ]:
check_loss <- function(y, mu, tau) {
  if (!is.numeric(tau) || tau < 0 || tau > 1) {
    stop("tau must be a numeric value between 0 and 1.")
  }
  errors <- y - mu
  loss <- ifelse(errors >= 0, 
                 tau * errors,    
                 (1 - tau) * -errors)  
return(loss)
}

In [ ]:
# post_ndlm <- function(samp_theta, FF, sig.samp) {
#   n_time <- dim(samp_theta)[2]
#   n_sim <- dim(samp_theta)[3]
#   y_post <- array(NA, dim = c(3, n_time, n_sim))
  
#   for (t in 1:n_time) {
#     FF_t <- FF[,,t]  
#     theta_t_s <- samp_theta[, t, ]
#     y_post[, t, ] <- t(FF_t) %*% theta_t_s + t(sqrt(sig.samp))*matrix(rnorm(3*n_sim),3,n_sim)  
#   }
#   return(y_post)
# }

# y_M_post <- post_ndlm(samp.theta_M, FF, samp.sigma_M)

In [ ]:
y_post_mean_ndlm <- apply(samp.post.pred__NDLM, c(1, 2), mean)
y_post_qs_ndlm <- apply(samp.post.pred__NDLM, c(1, 2), function(x) quantile(x, probs = c(0.05, 0.5, 0.95)))

In [ ]:
y_post_qs_ndlm_95  <- y_post_mean_ndlm[1,] + mean(sqrt(samp.sigma__NDLM[1,]))*pnorm(0.95)
y_post_qs_ndlm_50  <- y_post_mean_ndlm[1,] + mean(sqrt(samp.sigma__NDLM[1,]))*pnorm(0)
y_post_qs_ndlm_05  <- y_post_mean_ndlm[1,] + mean(sqrt(samp.sigma__NDLM[1,]))*pnorm(0.05)

## II. Generalized PPLC 
###  $\sum_{t=1}^T \rho_{p_0}(y_{t}^{\text{usgs}} - \mathbb{E}_{q_\phi(z)}[y_{t, \text{new}}^{\text{usgs}}])$


In [ ]:
library(knitr)

# Data preparation and rolling mean calculation
data_50_M <- data.frame(Time = 1:length(Y[1,]), Loss = check_loss(Y[1,], post_mean_50_exAL[1,], 0.50))
data_05_M <- data.frame(Time = 1:length(Y[1,]), Loss = check_loss(Y[1,], post_mean_5_exAL[1,], 0.05))
data_95_M <- data.frame(Time = 1:length(Y[1,]), Loss = check_loss(Y[1,], post_mean_95_exAL[1,], 0.95))
data_ndlm <- data.frame(Time = 1:length(Y[1,]), Loss = check_loss(Y[1,], y_post_mean_ndlm[1,], 0.50))

data_50_M$RollingMean <- rollapply(data_50_M$Loss, 360, mean, partial = TRUE, fill = NA, align = "right")
data_05_M$RollingMean <- rollapply(data_05_M$Loss, 360, mean, partial = TRUE, fill = NA, align = "right")
data_95_M$RollingMean <- rollapply(data_95_M$Loss, 360, mean, partial = TRUE, fill = NA, align = "right")
data_ndlm$RollingMean <- rollapply(data_ndlm$Loss, 360, mean, partial = TRUE, fill = NA, align = "right")

data_50_M$model <- "exAL-0.5"
data_05_M$model <- "exAL-0.05"
data_95_M$model <- "exAL-0.95"
data_ndlm$model <- "NDLM"

# Combine all data into one data frame
all_data <- rbind(data_ndlm, data_50_M, data_05_M, data_95_M)

# Plotting with improved colors and cleaned x-axis
p <- ggplot(all_data, aes(x = Time)) +
  geom_line(aes(y = Loss, colour = "Actual Loss"), linewidth = 1.2) +
  geom_line(aes(y = RollingMean, colour = "Rolling Mean (360)"), linewidth = 0.8) +
  facet_wrap(~ model, nrow = 1, scales = "free_x") +  # One row of plots with free scales
  labs(title = "Check Loss Function Across Models",
       x = "",
       y = "Loss",
       colour = "Legend") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),  # Hide x-axis text
    axis.ticks.x = element_blank()  # Hide x-axis ticks
  ) +
  scale_colour_manual(values = c("Actual Loss" = "#1f77b4", "Rolling Mean (360)" = "#ff7f0e"))

# Display the plot
print(p)


mean_losses <- data.frame(
  Model = c("NDLM", "exAL-0.5", "exAL-0.05", "exAL-0.95"),
  MeanLoss = c(mean(data_ndlm$Loss), mean(data_50_M$Loss), mean(data_05_M$Loss), mean(data_95_M$Loss))
)

# Format the mean losses into a clean table
kable(mean_losses, format = "markdown", caption = "Mean Check Losses Across Models")

In [ ]:
score <- subset(all_data, model==unique(all_data[,4])[1])[,2]
plot.ts(cumsum(score)/TT, col='darkorange', ylim = c(0,0.35))
score <- subset(all_data, model==unique(all_data[,4])[2])[,2]
lines(cumsum(score)/TT, col='darkgreen')
score <- subset(all_data, model==unique(all_data[,4])[3])[,2]
lines(cumsum(score)/TT, col='darkred')
score <- subset(all_data, model==unique(all_data[,4])[4])[,2]
lines(cumsum(score)/TT, col='darkblue')

## III. Generalized MSE
###  $\sum_{t=1}^T \mathbb{E}_{q_\phi(z)}[\rho_{p_0}(y_{t}^{\text{usgs}} - y_{t, \text{new}}^{\text{usgs}})]$

In [ ]:
# Function to process each model's predictive performance
process_model <- function(y_vector, post_pred, tau, model_name) {
  n_samples <- dim(post_pred)[3]  # Number of samples from the third dimension
  y_matrix <- matrix(rep(y_vector, each = n_samples), nrow = length(y_vector), ncol = n_samples)
  
  # Compute differences and apply loss function
  diff_matrix <- y_matrix - post_pred[1, , ]
  loss_matrix <- ifelse(diff_matrix >= 0, tau * diff_matrix, (1 - tau) * -diff_matrix)
  mean_per_t <- rowMeans(loss_matrix)
  
  data_frame <- data.frame(
    Time = 1:length(mean_per_t),
    Loss = mean_per_t,
    RollingMean = rollapply(mean_per_t, 360, mean, partial = TRUE, fill = NA, align = "right"),
    Model = model_name
  )
  
  overall_mean_loss = mean(mean_per_t)
  return(list(data_frame = data_frame, mean_loss = overall_mean_loss))
}

# List of models with parameters and specific names
models <- list(
  list(post_pred = samp.post.pred__NDLM, tau = 0.50, name = "NDLM"),
  list(post_pred = samp.post.pred_50_exAL, tau = 0.5, name = "exAL-0.5"),
  list(post_pred = samp.post.pred_5_exAL, tau = 0.05, name = "exAL-0.05"),
  list(post_pred = samp.post.pred_95_exAL, tau = 0.95, name = "exAL-0.95")
)

results <- lapply(models, function(m) process_model(Y[1, ], m$post_pred, m$tau, m$name))

# Combine results into a single data frame for plotting
all_data <- do.call(rbind, lapply(results, `[[`, "data_frame"))

# Plotting using ggplot2 with the specific formatting as requested
p <- ggplot(all_data, aes(x = Time)) +
  geom_line(aes(y = Loss, colour = "Actual Loss"), linewidth = 1.2) +
  geom_line(aes(y = RollingMean, colour = "Rolling Mean (360)"), linewidth = 0.8) +
  facet_wrap(~ Model, nrow = 1, scales = "free_x") +
  labs(title = "Predictive Performance Across Models",
       x = "",
       y = "Loss",
       colour = "Legend") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),  # Hide x-axis text
    axis.ticks.x = element_blank()  # Hide x-axis ticks
  ) +
  scale_colour_manual(values = c("Actual Loss" = "#1f77b4", "Rolling Mean (360)" = "#ff7f0e"))

# Display the plot
print(p)

# Displaying the table of mean losses
# Assuming results are generated as follows:
results <- lapply(models, function(m) process_model(Y[1, ], m$post_pred, m$tau, m$name))
# Extracting mean losses to a data frame
mean_losses <- do.call(rbind, lapply(results, function(x) {
  data.frame(Model = x$data_frame$Model[1], MeanLoss = x$mean_loss, stringsAsFactors = FALSE)
}))
# Formatting and printing the table using knitr::kable
kable(mean_losses, format = "markdown", caption = "Mean Losses Across Models")


In [ ]:
score <- subset(all_data, Model==unique(all_data[,4])[1])[,2]
plot.ts(cumsum(score)/TT, col='darkorange', ylim = c(0,0.6))
score <- subset(all_data, Model==unique(all_data[,4])[2])[,2]
lines(cumsum(score)/TT, col='darkgreen')
score <- subset(all_data, Model==unique(all_data[,4])[3])[,2]
lines(cumsum(score)/TT, col='darkred')
score <- subset(all_data, Model==unique(all_data[,4])[4])[,2]
lines(cumsum(score)/TT, col='darkblue')

## IV. KL divergence
### $KL(h, \phi) = \int_{-\infty}^{\infty} h(x) \log \left(\frac{h(x)}{\phi(x)}\right) dx$


where:
- $ h(x) $ "standardize" one-step-ahead forecast.
- $ \phi(x) $ is the standard normal density.




In [ ]:
# Function to process and plot KL Divergence and forecast errors
process_errors <- function(errors, model_name) {
  s <-0
  n <- 100
  for(k in 1:n){
  T_e <- length(errors)
  ref <- stats::rnorm(T_e)  # Reference normal distribution
  kl_divergence <- mean(FNN::KL.divergence(ref, errors))
  s <- s + kl_divergence/n
  }
  # Creating a dataframe for ggplot
  data_frame <- data.frame(
    Time = 1:T_e,
    NormalizedErrors = stats::pnorm(errors),
    RollingMean = rollapply(stats::pnorm(errors), 90, mean, partial = TRUE, fill = NA, align = "right"),
    Model = model_name
  )

  return(list(data_frame = data_frame, kl_divergence = kl_divergence))
}

# List of models with parameters and specific names
models <- list(
  list(errors = new.theta.out__NDLM$standard_forecast_errors[1,], name = "NDLM Errors"),
  list(errors = new.theta.out_50_exAL$standard_forecast_errors[1,], name = "exAL-0.5 errors"),
  list(errors = new.theta.out_5_exAL$standard_forecast_errors[1,], name = "exAL-0.05 errors"),
  list(errors = new.theta.out_95_exAL$standard_forecast_errors[1,], name = "exAL-0.95 errors")
)


results <- lapply(models, function(m) process_errors(m$errors, m$name))

# Combine results into a single data frame for plotting
all_data <- do.call(rbind, lapply(results, `[[`, "data_frame"))

# Plotting using ggplot2 with the specific formatting as requested
p <- ggplot(all_data, aes(x = Time)) +
  geom_line(aes(y = NormalizedErrors, colour = "Actual Errors"), linewidth = 1.2) +
  geom_line(aes(y = RollingMean, colour = "Rolling Mean (90)"), linewidth = 0.8) +
  facet_wrap(~ Model, nrow = 1, scales = "free_x") +
  labs(title = "KL Divergence and Forecast Errors Across Models",
       x = "",
       y = "Normalized Errors",
       colour = "Legend") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),  # Hide x-axis text
    axis.ticks.x = element_blank()  # Hide x-axis ticks
  ) +
  scale_colour_manual(values = c("Actual Errors" = "#1f77b4", "Rolling Mean (90)" = "#ff7f0e"))

# Display the plot
print(p)

# Displaying KL Divergences

# Assuming results were generated as follows:
results <- lapply(models, function(m) process_errors(m$errors, m$name))
kl_divergences <- do.call(rbind, lapply(results, function(x) data.frame(Model = x$data_frame$Model[1], KL_Divergence = x$kl_divergence)))

# Format the KL Divergence data into a clean table
kable(kl_divergences, format = "markdown", caption = "Standardize Forecast Error Models")


In [ ]:
score <- subset(all_data, Model==unique(all_data[,4])[1])[,2]
plot.ts(cumsum(score)/TT, col='darkorange', ylim = c(0,0.6))
score <- subset(all_data, Model==unique(all_data[,4])[2])[,2]
lines(cumsum(score)/TT, col='darkgreen')
score <- subset(all_data, Model==unique(all_data[,4])[3])[,2]
lines(cumsum(score)/TT, col='darkred')
score <- subset(all_data, Model==unique(all_data[,4])[4])[,2]
lines(cumsum(score)/TT, col='darkblue')

Why not the "smoothed errors"??

## V. Quantile Check: 
### $\frac{1}{T} \sum_{t=1}^{T}I(y^{usgs}_t \leq F_t'\theta^*_t) \approx p_0$


In [ ]:
# Define a function to process each model and calculate statistics
process_model_stats <- function(v, y, target_prob) {
  boolean_array <- sweep(v[1, , ], MARGIN = 1, STATS = y, FUN = ">")
  boolean_means <- apply(boolean_array, 2, mean)
  c(SD = sd(boolean_means), MAD = mean(abs(boolean_means - target_prob)))
}

# Assume 'xb_50_corrected', 'xb_05_corrected', 'xb_95_corrected',
# 'xb_M_50', 'xb_M_05', 'xb_M_95' are loaded and available

# Apply the function to each model
stats_50_corrected <- process_model_stats(xb_50_corrected, Y[1,], 0.5)
stats_05_corrected <- process_model_stats(xb_05_corrected, Y[1,], 0.05)
stats_95_corrected <- process_model_stats(xb_95_corrected, Y[1,], 0.95)

stats_M_50 <- process_model_stats(xb_M_50, Y[1,], 0.5)
stats_M_05 <- process_model_stats(xb_M_05, Y[1,], 0.05)
stats_M_95 <- process_model_stats(xb_M_95, Y[1,], 0.95)

# Combine all stats into a data frame, arranging to compare similar models
results <- data.frame(
  Model = c("exAL-0.5", "NDLM-0.5", "exAL-0.05", "NDLM-0.05",
            "exAL-0.95", "NDLM-0.95"),
  SD = c(stats_50_corrected["SD"], stats_M_50["SD"], stats_05_corrected["SD"], stats_M_05["SD"],
         stats_95_corrected["SD"], stats_M_95["SD"]),
  MAD = c(stats_50_corrected["MAD"], stats_M_50["MAD"], stats_05_corrected["MAD"], stats_M_05["MAD"],
          stats_95_corrected["MAD"], stats_M_95["MAD"])
)


# Print the table using kable
kable(results, format = "markdown", caption = "Statistical Measures for exAL and NDLM Models, Paired for Comparison")



In [ ]:
# Function to analyze and plot histograms with better visibility and aesthetics
analyze_and_plot <- function(v, dataset_name) {
  # Perform the boolean comparison with the global Y[1,]
  boolean_array <- sweep(v[1,,], MARGIN = 1, STATS = Y[1,], FUN = ">")
  boolean_means <- apply(boolean_array, 2, mean)
  
  # Plotting histogram with adjusted breaks and color
  hist(boolean_means, 
       main = paste("", dataset_name), 
       xlab = "Proportion", 
       breaks = 30, 
       col = rgb(0.1, 0.2, 0.5, 0.8),  # Adjusted alpha for better visibility
       border = "darkblue", 
       las = 1,
       ylim = c(0, max(table(cut(boolean_means, breaks = 50))) + 5))  # Adjusted ylim for clarity
  
  # Calculating percentiles
  p2.5 <- quantile(boolean_means, 0.025)
  p97.5 <- quantile(boolean_means, 0.975)

  # Adding vertical lines for percentiles
  abline(v = p2.5, col = "darkred", lwd = 0.5, lty = 2)
  abline(v = p97.5, col = "darkred", lwd = 0.5, lty = 2)
  
  # Adding annotations for percentiles
  text(x = p2.5, y = par("usr")[4] * 0.95, labels = paste("", round(p2.5, 4)), 
       srt = 0, col = "darkred", cex = 0.8, adj = 1)
  text(x = p97.5, y = par("usr")[4] * 0.95, labels = paste("", round(p97.5, 4)), 
       srt = 0, col = "darkred", cex = 0.8, adj = 1)
}

# Assuming Y and datasets are properly defined and loaded
# Setup the plot layout
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1))  # Adjusted layout to accommodate 6 plots

# Analyze and plot for each dataset
analyze_and_plot(xb_05_corrected, "exAL-0.05")
analyze_and_plot(xb_50_corrected, "exAL-0.5")
analyze_and_plot(xb_95_corrected, "exAL-0.95")
analyze_and_plot(xb_M_05, "NDLM-0.05")
analyze_and_plot(xb_M_50, "NDLM-0.5")
analyze_and_plot(xb_M_95, "NDLM-0.95")


In [ ]:
# Function to analyze and plot rolling statistics with enhanced aesthetics
analyze_and_plot_rolling <- function(v, dataset_name, roll_m, y_limits, h_line) {
  # Perform the boolean comparison with the global Y[1,]
  boolean_array <- sweep(v[1,,], MARGIN = 1, STATS = Y[1,], FUN = ">")
  
  # Convert boolean results to a time series object
  ts_data <- zoo(apply(boolean_array, 2, mean), order.by = 1:dim(boolean_array)[2])
  
  # Calculate rolling statistics
  roll_mean <- rollapply(ts_data, roll_m, mean, fill = NA, align = "right")
  roll_p2.5 <- rollapply(ts_data, roll_m, quantile, probs = 0.025, fill = NA, align = "right")
  roll_p97.5 <- rollapply(ts_data, roll_m, quantile, probs = 0.975, fill = NA, align = "right")
  
  # Plotting the time series with refined aesthetics
  plot(roll_mean, type = "l", col = "darkblue", xlab = "Time", ylab = "Probability", main = paste("Rolling Mean -", dataset_name),
       ylim = y_limits, lwd = 2)
  lines(roll_p2.5, col = "darkred", lty = 1, lwd = 1)
  lines(roll_p97.5, col = "darkred", lty = 1, lwd = 1)
  
  # Add horizontal line at the specified position
  abline(h = h_line, col = "darkorange", lwd = 2, lty = 2)
  
  # Legend for clarity
  legend("topright", legend = c("Mean", "95% Bands"), 
         col = c("darkblue", "darkred"), lty = 1, lwd = 2, cex = 0.8)
}

# Adjust layout to accommodate 6 plots
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))  # Adjust outer margin for overall plot titles or notes

# Analyze and plot for each dataset with updated aesthetic settings
analyze_and_plot_rolling(xb_95_corrected, "xb_95_corrected", 90, c(0.9, 1), 0.95)
analyze_and_plot_rolling(xb_50_corrected, "xb_50_corrected", 90, c(0.4, 0.6), 0.5)
analyze_and_plot_rolling(xb_05_corrected, "xb_05_corrected", 90, c(0.0, 1), 0.05)

analyze_and_plot_rolling(xb_M_95, "xb_95_ndlm", 90, c(0.9, 1), 0.95)
analyze_and_plot_rolling(xb_M_50, "xb_50_ndlm", 90, c(0.4, 0.6), 0.5)
analyze_and_plot_rolling(xb_M_05, "xb_05_ndlm", 90, c(0.0, 0.1), 0.05)


## VI. Posterior Predictive Quantile Check: 
### $\frac{1}{T} \sum_{t=1}^T I(y_{t}^* \leq F_t'\theta^*_t) \approx p_0$ 
### $ \mathbb{E}[I(y_{t}^{new} \leq F_t'\theta_t) | y^{usgs}_{1:T}] \approx p_0$ 


In [ ]:
# Define a function to process each model comparison and calculate statistics
process_comparison_stats <- function(predicted, corrected, target_prob) {
  # Compute the difference and the boolean array where differences are less than zero
  boolean_array <- (predicted - corrected) < 0
  boolean_means <- apply(boolean_array, 2, mean) - target_prob
  
  # Calculate SD and MAD
  SD = sd(boolean_means)
  MAD = mean(abs(boolean_means))
  
  return(c(SD = SD, MAD = MAD))
}

# Assume 'samp.post.pred_xx_M', 'xb_xx_corrected', 'y_M_post', 'xb_M_xx' are loaded and available

# Apply the function to each model and compute required statistics
stats_50_corr <- process_comparison_stats(samp.post.pred_50_exAL[1,,], xb_50_corrected[1,,], 0.5)
stats_05_corr <- process_comparison_stats(samp.post.pred_5_exAL[1,,], xb_05_corrected[1,,], 0.05)
stats_95_corr <- process_comparison_stats(samp.post.pred_95_exAL[1,,], xb_95_corrected[1,,], 0.95)

stats_M_50 <- process_comparison_stats(samp.post.pred__NDLM[1,,], xb_M_50[1,,], 0.5)
stats_M_05 <- process_comparison_stats(samp.post.pred__NDLM[1,,], xb_M_05[1,,], 0.05)
stats_M_95 <- process_comparison_stats(samp.post.pred__NDLM[1,,], xb_M_95[1,,], 0.95)

# Combine all stats into a data frame
results <- data.frame(
  Model = c("exAL-0.5", "NDLM-0.5", "exAL-0.05", "NDLM-0.05", "exAL-0.95", "NDLM-0.95"),
  SD = c(stats_50_corr["SD"], stats_M_50["SD"], stats_05_corr["SD"], stats_M_05["SD"], stats_95_corr["SD"], stats_M_95["SD"]),
  MAD = c(stats_50_corr["MAD"], stats_M_50["MAD"], stats_05_corr["MAD"], stats_M_05["MAD"], stats_95_corr["MAD"], stats_M_95["MAD"])
)

kable(results, format = "markdown", caption = "Statistical Measures for Predicted vs. Corrected Models")


In [ ]:
# Function to analyze and plot histograms using posterior predictive samples
analyze_and_plot <- function(post_pred, corrected, dataset_name) {
  # Perform the boolean comparison directly with the posterior predictive samples
  boolean_array <- (post_pred[1,,] - corrected[1,,]) < 0
  boolean_means <- apply(boolean_array, 2, mean)
  
  # Plotting histogram with adjusted aesthetics
  hist(boolean_means,
       main = paste("", dataset_name),
       xlab = "Proportion",
       breaks = 30,
       col = rgb(0.1, 0.2, 0.5, 0.8),
       border = "darkblue",
       las = 1,
       ylim = c(0, max(table(cut(boolean_means, breaks = 50))) + 5))
  
  # Calculating percentiles
  p2.5 <- quantile(boolean_means, 0.025)
  p97.5 <- quantile(boolean_means, 0.975)

  # Adding vertical lines for percentiles
  abline(v = p2.5, col = "darkred", lwd = 0.5, lty = 2)
  abline(v = p97.5, col = "darkred", lwd = 0.5, lty = 2)
  
  # Adding annotations for percentiles
  text(x = p2.5, y = par("usr")[4] * 0.95, labels = paste(round(p2.5, 4)), 
       srt = 0, col = "darkred", cex = 0.8, adj = 1)
  text(x = p97.5, y = par("usr")[4] * 0.95, labels = paste(round(p97.5, 4)), 
       srt = 0, col = "darkred", cex = 0.8, adj = 1)
}

# Setup the plot layout
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1))  # Adjusted layout to accommodate 6 plots

# Assuming that post_pred and corrected datasets are properly defined and loaded
# Analyze and plot for each dataset
analyze_and_plot(samp.post.pred_5_exAL, xb_05_corrected, "exAL-0.05")
analyze_and_plot(samp.post.pred_50_exAL, xb_50_corrected, "exAL-0.50")
analyze_and_plot(samp.post.pred_95_exAL, xb_95_corrected, "exAL-0.95")
analyze_and_plot(samp.post.pred__NDLM, xb_M_05, "NDLM-0.05")
analyze_and_plot(samp.post.pred__NDLM, xb_M_50, "NDLM-0.5")
analyze_and_plot(samp.post.pred__NDLM, xb_M_95, "NDLM-0.95")

In [ ]:
# Function to analyze and plot rolling statistics using posterior predictive samples
analyze_and_plot_rolling <- function(post_pred, corrected, dataset_name, roll_m, y_limits, h_line) {
  # Perform the boolean comparison directly with the posterior predictive samples
  boolean_array <- (post_pred[1,,] - corrected[1,,]) < 0
  
  # Convert boolean results to a time series object
  ts_data <- zoo(apply(boolean_array, 2, mean), order.by = 1:dim(boolean_array)[2])
  
  # Calculate rolling statistics
  roll_mean <- rollapply(ts_data, roll_m, mean, fill = NA, align = "right")
  roll_p2.5 <- rollapply(ts_data, roll_m, quantile, probs = 0.025, fill = NA, align = "right")
  roll_p97.5 <- rollapply(ts_data, roll_m, quantile, probs = 0.975, fill = NA, align = "right")
  
  # Plotting the time series with refined aesthetics
  plot(roll_mean, type = "l", col = "darkblue", xlab = "Time", ylab = "Probability", 
       main = paste("Rolling Mean -", dataset_name), ylim = y_limits, lwd = 2)
  lines(roll_p2.5, col = "darkred", lty = 1, lwd = 1)
  lines(roll_p97.5, col = "darkred", lty = 1, lwd = 1)
  
  # Add horizontal line at the specified position
  abline(h = h_line, col = "darkorange", lwd = 2, lty = 2)
  
  # Legend for clarity
  legend("topright", legend = c("Mean", "95% Band"), 
         col = c("darkblue", "darkred"), lty = 1, lwd = 1, cex = 0.8)
}

# Adjust layout to accommodate 6 plots
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))

# Analyze and plot for each dataset with updated aesthetic settings
analyze_and_plot_rolling(samp.post.pred_50_exAL, xb_50_corrected, "exAL-0.5", 90, c(0.45, 0.55), 0.5)
analyze_and_plot_rolling(samp.post.pred_5_exAL, xb_05_corrected, "exAL-0.05", 90, c(0, 0.1), 0.05)
analyze_and_plot_rolling(samp.post.pred_95_exAL, xb_95_corrected, "exAL-0.95", 90, c(0.9, 1), 0.95)

analyze_and_plot_rolling(samp.post.pred__NDLM, xb_M_50, "NDLM-0.5", 90, c(0.45, 0.55), 0.5)
analyze_and_plot_rolling(samp.post.pred__NDLM, xb_M_05, "NDLM-0.05", 90, c(0, 0.1), 0.05)
analyze_and_plot_rolling(samp.post.pred__NDLM, xb_M_95, "NDLM-0.95", 90, c(0.9, 1), 0.95)


## NWS Stamdardize log(Forecasts+1)

CHECK IF IM COMPUTING RIGHT ELBOS!

In [ ]:
idx <- (TT-500):TT
date_info <- create_date_labels(idx)
par(mfrow = c(2, 2), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))

new.cpp <- samp.post.pred_50_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-50"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.975), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkorange')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.025), col='darkblue')

new.cpp <- samp.post.pred_5_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-05"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.975), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.025), col='darkorange')

new.cpp <- samp.post.pred_95_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-95"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.975), col='darkorange')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.025), col='darkblue')

new.cpp <- samp.post.pred__NDLM
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("NDLM"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.975), col='darkblue')
lines(apply(new.cpp[1, idx, ], 1, mean), col='darkorange')
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.025), col='darkblue')


In [ ]:
new.cpp1 <- samp.post.pred_5_exAL
new.cpp2 <- samp.post.pred_20_exAL
new.cpp3 <- samp.post.pred_35_exAL
new.cpp4 <- samp.post.pred_50_exAL
new.cpp5 <- samp.post.pred_65_exAL
new.cpp6 <- samp.post.pred_80_exAL
new.cpp7 <- samp.post.pred_95_exAL

par(mfrow = c(1, 1), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))
plot.ts(Y[1,idx], col='black', ylim = c(-3,4), main = paste("Quantiles of the Posterior Predictive"), xaxt="n")
# lines(apply(new.cpp1[1, idx, ], 1, quantile, probs = 0.05), col='darkred')
# lines(apply(new.cpp2[1, idx, ], 1, quantile, probs = 0.20), col='purple')
# lines(apply(new.cpp3[1, idx, ], 1, quantile, probs = 0.35), col='purple')
# lines(apply(new.cpp4[1, idx, ], 1, quantile, probs = 0.50), col='forestgreen')
# lines(apply(new.cpp5[1, idx, ], 1, quantile, probs = 0.65), col='purple')
# lines(apply(new.cpp6[1, idx, ], 1, quantile, probs = 0.80), col='purple')
# lines(apply(new.cpp7[1, idx, ], 1, quantile, probs = 0.95), col='darkblue')

x1 <- apply(new.cpp1[1, idx, ], 1, quantile, probs = 0.05)
x2 <- apply(new.cpp2[1, idx, ], 1, quantile, probs = 0.20)
x3 <- apply(new.cpp3[1, idx, ], 1, quantile, probs = 0.35)
x4 <- apply(new.cpp4[1, idx, ], 1, quantile, probs = 0.50)
x5 <- apply(new.cpp5[1, idx, ], 1, quantile, probs = 0.65)
x6 <- apply(new.cpp6[1, idx, ], 1, quantile, probs = 0.80)
x7 <- apply(new.cpp7[1, idx, ], 1, quantile, probs = 0.95)
lines( (x1+x2+x3+x4+x5+x6+x7)/7, col='pink')
# lines( (x1+x4+x7)/7, col='gold')
new.cpp0 <- samp.post.pred__NDLM
x0 <- apply(new.cpp0[1, idx, ], 1, quantile, probs = 0.5)
lines( x0, col='purple')

x1 <- apply(new.cpp1[1, idx, ], 1, quantile, probs = 0.5)
x2 <- apply(new.cpp2[1, idx, ], 1, quantile, probs = 0.5)
x3 <- apply(new.cpp3[1, idx, ], 1, quantile, probs = 0.5)
x4 <- apply(new.cpp4[1, idx, ], 1, quantile, probs = 0.5)
x5 <- apply(new.cpp5[1, idx, ], 1, quantile, probs = 0.5)
x6 <- apply(new.cpp6[1, idx, ], 1, quantile, probs = 0.5)
x7 <- apply(new.cpp7[1, idx, ], 1, quantile, probs = 0.5)
lines( (x1+x2+x3+x4+x5+x6+x7)/7, col='gold')
# lines( (x1+x4+x7)/3, col='gold')


In [ ]:
# MEDIAN POST PREDICTIVE for each model
idx <- (TT-500):TT
date_info <- create_date_labels(idx)
par(mfrow = c(2, 2), mar = c(4, 4, 2, 1), oma = c(0, 0, 2, 0))

new.cpp <- samp.post.pred_50_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-50"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='forestgreen')

new.cpp <- samp.post.pred_5_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-05"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkred')

new.cpp <- samp.post.pred_95_exAL
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("exAL-95"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkblue')

new.cpp <- samp.post.pred__NDLM
plot.ts(Y[1,idx], col='black', 
        ylim = c(-3,4),
        main = paste("NDLM"), xaxt="n")
lines(apply(new.cpp[1, idx, ], 1, quantile, probs = 0.5), col='darkorange')

## AV and SL

In [ ]:
load_variables <- function(filename, dir_path) {
  file_path <- file.path(dir_path, filename)
  load(file_path)
  cat("Variables loaded from:", file_path, "\n")
}

file_path <- "/home/jaguir26/project1_ucsc_phd/variables_50_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_5_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_95_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_20_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_35_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_65_AV.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_80_AV.RData"
load(file_path)

file_path <- "/home/jaguir26/project1_ucsc_phd/variables_50_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_5_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_95_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_20_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_35_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_65_SL.RData"
load(file_path)
file_path <- "/home/jaguir26/project1_ucsc_phd/variables_80_SL.RData"
load(file_path)


In [ ]:
# Set up the plotting area for a 6x2 matrix layout
par(mfrow = c(2, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 3, 0))

# Plot each time series with the specified colors
y1 <- seq.sigma_50_SL
y2 <- seq.sigma_50_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Sigma 50th", xlab = "Iteration", ylab = "Sigma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Sigma 50th", xlab = "Iteration", ylab = "Sigma", col = 'purple')

y1 <- seq.sigma_5_SL
y2 <- seq.sigma_5_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Sigma 05th", xlab = "Iteration", ylab = "Sigma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Sigma 05th", xlab = "Iteration", ylab = "Sigma", col = 'purple')

y1 <- seq.sigma_95_SL
y2 <- seq.sigma_95_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Sigma 95th", xlab = "Iteration", ylab = "Sigma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Sigma 95th", xlab = "Iteration", ylab = "Sigma", col = 'purple')

# Plot each time series with the specified colors
y1 <- seq.gamma_50_SL
y2 <- seq.gamma_50_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Gamma 50th", xlab = "Iteration", ylab = "Gamma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Gamma 50th", xlab = "Iteration", ylab = "Gamma", col = 'purple')

y1 <- seq.gamma_5_SL
y2 <- seq.gamma_5_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Gamma 05th", xlab = "Iteration", ylab = "Gamma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Gamma 05th", xlab = "Iteration", ylab = "Gamma", col = 'purple')

y1 <- seq.gamma_95_SL
y2 <- seq.gamma_95_AV
YLIM <- c(min(y1,y2), max(y1,y2))
plot.ts(t(y1), main = "Gamma 95th", xlab = "Iteration", ylab = "Gamma", col = 'green', ylim = YLIM )
lines(t(y2), main = "Gamma 95th", xlab = "Iteration", ylab = "Gamma", col = 'purple')

# Resetting the plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
# Set up the plotting area for a 6x2 matrix layout
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1), oma = c(0, 0, 3, 0))

na_lim <- 5

y1 <- seq.elbo_50_SL
y2 <- seq.elbo_50_AV
y1[1:na_lim]=NaN
y2[1:na_lim]=NaN
YLIM <- c(min(y1,y2, na.rm = TRUE), max(y1,y2, na.rm = TRUE))
plot.ts(t(y1), main = "ELBO 50th", xlab = "Iteration", ylab = "ELBO", col = 'green', ylim = YLIM )
lines(t(y2), main = "ELBO 50th", xlab = "Iteration", ylab = "ELBO", col = 'purple')

y1 <- seq.elbo_5_SL
y2 <- seq.elbo_5_AV
y1[1:na_lim]=NaN
y2[1:na_lim]=NaN
YLIM <- c(min(y1,y2, na.rm = TRUE), max(y1,y2, na.rm = TRUE))
plot.ts(t(y1), main = "ELBO 05th", xlab = "Iteration", ylab = "ELBO", col = 'green', ylim = YLIM )
lines(t(y2), main = "ELBO 05th", xlab = "Iteration", ylab = "ELBO", col = 'purple')

y1 <- seq.elbo_95_SL
y2 <- seq.elbo_95_AV
y1[1:na_lim]=NaN
y2[1:na_lim]=NaN
YLIM <- c(min(y1,y2, na.rm = TRUE), max(y1,y2, na.rm = TRUE))
plot.ts(t(y1), main = "ELBO 95th", xlab = "Iteration", ylab = "ELBO", col = 'green', ylim = YLIM )
lines(t(y2), main = "ELBO 95th", xlab = "Iteration", ylab = "ELBO", col = 'purple')

par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
n <- dim(FF)[1]-J

In [ ]:
compute_xb_corrected <- function(samp_theta, FF) {
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  xb <- array(NA, dim = c(J+1, n_time, n_sim))
  
  for (t in 1:n_time) {
    FF_t <- FF[,,t]
    theta_t_s <- samp_theta[, t, ]
    xb[, t, ] <- t(FF_t) %*% theta_t_s
  }
  return(xb)
}

# Apply the corrected function to each samp.theta matrix
xb_50_AV_corrected <- compute_xb_corrected(samp.theta_50_AV, FF[1:n,,])
xb_05_AV_corrected <- compute_xb_corrected(samp.theta_5_AV, FF[1:n,,])
xb_95_AV_corrected <- compute_xb_corrected(samp.theta_95_AV, FF[1:n,,])
xb_20_AV_corrected <- compute_xb_corrected(samp.theta_20_AV, FF[1:n,,])
xb_35_AV_corrected <- compute_xb_corrected(samp.theta_35_AV, FF[1:n,,])
xb_65_AV_corrected <- compute_xb_corrected(samp.theta_65_AV, FF[1:n,,])
xb_80_AV_corrected <- compute_xb_corrected(samp.theta_80_AV, FF[1:n,,])

xb_50_SL_corrected <- compute_xb_corrected(samp.theta_50_AV, FF[1:n,,])
xb_05_SL_corrected <- compute_xb_corrected(samp.theta_5_AV, FF[1:n,,])
xb_95_SL_corrected <- compute_xb_corrected(samp.theta_95_AV, FF[1:n,,])
xb_20_SL_corrected <- compute_xb_corrected(samp.theta_20_AV, FF[1:n,,])
# xb_35_SL_corrected <- compute_xb_corrected(samp.theta_35_AV, FF[1:n,,])
xb_65_SL_corrected <- compute_xb_corrected(samp.theta_65_AV, FF[1:n,,])
xb_80_SL_corrected <- compute_xb_corrected(samp.theta_80_AV, FF[1:n,,])


In [ ]:
# Compute quantiles
compute_quantiles <- function(xb) {
  apply(xb, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
}

quantiles_xb_AV_50_corrected <- compute_quantiles(xb_50_AV_corrected)
quantiles_xb_AV_05_corrected <- compute_quantiles(xb_05_AV_corrected)
quantiles_xb_AV_95_corrected <- compute_quantiles(xb_95_AV_corrected)
quantiles_xb_AV_20_corrected <- compute_quantiles(xb_20_AV_corrected)
quantiles_xb_AV_35_corrected <- compute_quantiles(xb_35_AV_corrected)
quantiles_xb_AV_65_corrected <- compute_quantiles(xb_65_AV_corrected)
quantiles_xb_AV_80_corrected <- compute_quantiles(xb_80_AV_corrected)


quantiles_xb_SL_50_corrected <- compute_quantiles(xb_50_SL_corrected)
quantiles_xb_SL_05_corrected <- compute_quantiles(xb_05_SL_corrected)
quantiles_xb_SL_95_corrected <- compute_quantiles(xb_95_SL_corrected)
quantiles_xb_SL_20_corrected <- compute_quantiles(xb_20_SL_corrected)
# quantiles_xb_SL_35_corrected <- compute_quantiles(xb_35_SL_corrected)
quantiles_xb_SL_65_corrected <- compute_quantiles(xb_65_SL_corrected)
quantiles_xb_SL_80_corrected <- compute_quantiles(xb_80_SL_corrected)

In [ ]:
# Define data list
data_list <- list(
  gamma_50_AV = list(data = samp.gamma_50_AV, quantile = "50th", variable = "Gamma", source = "AV"),
  gamma_95_AV = list(data = samp.gamma_95_AV, quantile = "95th", variable = "Gamma", source = "AV"),
  gamma_05_AV = list(data = samp.gamma_5_AV, quantile = "05th", variable = "Gamma", source = "AV"),
  sigma_50_AV = list(data = samp.sigma_50_AV, quantile = "50th", variable = "Sigma", source = "AV"),
  sigma_95_AV = list(data = samp.sigma_95_AV, quantile = "95th", variable = "Sigma", source = "AV"),
  sigma_05_AV = list(data = samp.sigma_5_AV, quantile = "05th", variable = "Sigma", source = "AV"),
  gamma_50_SL = list(data = samp.gamma_50_SL, quantile = "50th", variable = "Gamma", source = "SL"),
  gamma_95_SL = list(data = samp.gamma_95_SL, quantile = "95th", variable = "Gamma", source = "SL"),
  gamma_05_SL = list(data = samp.gamma_5_SL, quantile = "05th", variable = "Gamma", source = "SL"),
  sigma_50_SL = list(data = samp.sigma_50_SL, quantile = "50th", variable = "Sigma", source = "SL"),
  sigma_95_SL = list(data = samp.sigma_95_SL, quantile = "95th", variable = "Sigma", source = "SL"),
  sigma_05_SL = list(data = samp.sigma_5_SL, quantile = "05th", variable = "Sigma", source = "SL")
)

# Function to calculate quantiles
calculate_quantiles <- function(data, variable_name, quantile_name, source_name) {
  quantile_values <- quantile(data, probs = c(0.025, 0.5, 0.975))
  tibble(
    variable = variable_name,
    source = source_name,
    quantile = quantile_name,
    quantile_025 = quantile_values["2.5%"],
    median = quantile_values["50%"],
    quantile_975 = quantile_values["97.5%"]
  )
}

# Calculate quantiles for each dataset
all_quantiles <- bind_rows(
  lapply(data_list, function(item) {
    calculate_quantiles(item$data, item$variable, item$quantile, item$source)
  })
)

# Print the complete table of quantiles
print(all_quantiles)


In [ ]:
prepare_quantile_data <- function(v_d) {
  v_d_transposed <- aperm(v_d, c(3, 1, 2))
  q_d_transposed <- apply(v_d_transposed, 2:3, function(x) quantile(x, probs = c(0.975, 0.5, 0.025)))
  q_d <- aperm(q_d_transposed, c(2, 3, 1))
  return(q_d)
}

# Apply the function to each dataset
q_d_50_AV <- prepare_quantile_data(samp.theta_50_AV)
q_d_05_AV <- prepare_quantile_data(samp.theta_5_AV)
q_d_95_AV <- prepare_quantile_data(samp.theta_95_AV)
q_d_50_SL <- prepare_quantile_data(samp.theta_50_SL)
q_d_05_SL <- prepare_quantile_data(samp.theta_5_SL)
q_d_95_SL <- prepare_quantile_data(samp.theta_95_SL)

In [ ]:
dates_ts_usgs <- timestamps

# Function to plot with quantiles and dates on x-axis
plot_quantile_component <- function(q_d_50, q_d_05, q_d_95, Y, idx, component, main_label, num_ticks) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d_50[component, idx, ], q_d_05[component, idx, ], q_d_95[component, idx, ])) * 2
  ylims[2] <- min(1.3, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.3, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

  lines(idx, q_d_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


# Function to plot with quantiles and dates on x-axis
plot_quantile_component_all <- function(q_d1_50, q_d1_05, q_d1_95, q_d2_50, q_d2_05, q_d2_95, Y, idx, component, main_label, num_ticks) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Ensure consistent margins in the function
  
  selected_dates <- dates_ts_usgs[idx]  # Retrieve dates corresponding to the indices

  # Ensure that there are exactly num_ticks ticks evenly distributed
  num_ticks <- 10
  tick_positions <- pretty(idx, num_ticks)  # Using pretty() to generate nice breakpoints

  # Adjust if pretty() provides more ticks than needed
  if (length(tick_positions) > num_ticks) {
    tick_positions <- tick_positions[seq(1, length(tick_positions), length.out = num_ticks)]
  }
  
  tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")

  ylims <- range(c(q_d1_50[component, idx, ], q_d1_05[component, idx, ], q_d1_95[component, idx, ])) * 2
  ylims[2] <- min(1.3, ylims[2])
  ylims[2] <- max(0.5, ylims[2])
  ylims[1] <- max(-1.3, ylims[1])
  ylims[1] <- min(-0.5, ylims[1])

  if (!main_label %in% c("Discrepancy USGS-GLOFAS", "Discrepancy USGS-NWS")) {
    plot(idx, Y[1, idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  } else {
    plot(idx,  -Y[1, idx]+Y[ifelse(main_label == "Discrepancy USGS-GLOFAS", 2, 3), idx], type = "l", col = "gray", lwd = 2, ylim = ylims, xlab = " ", ylab = "log-flow", main = main_label, xaxt = "n")
  }

  lines(idx, q_d1_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d1_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d1_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d1_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d1_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d1_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d1_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d1_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d1_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)

  lines(idx, q_d2_50[component, idx, 2], col = "forestgreen", lwd = 1)
  lines(idx, q_d2_50[component, idx, 1], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d2_50[component, idx, 3], col = "forestgreen", lwd = 0.5, lty = 2)
  lines(idx, q_d2_05[component, idx, 2], col = "darkred", lwd = 1)
  lines(idx, q_d2_05[component, idx, 1], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d2_05[component, idx, 3], col = "darkred", lwd = 0.5, lty = 2)
  lines(idx, q_d2_95[component, idx, 2], col = "darkblue", lwd = 1)
  lines(idx, q_d2_95[component, idx, 1], col = "darkblue", lwd = 0.5, lty = 2)
  lines(idx, q_d2_95[component, idx, 3], col = "darkblue", lwd = 0.5, lty = 2)

  abline(h=0, col='black')

  axis(1, at = tick_positions, labels = FALSE) 
  text(x = tick_positions, y = par("usr")[3] - 0.1 * diff(par("usr")[3:4]), labels = tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)
}


In [ ]:
# Set up plotting window for a 2x3 matrix layout
par(mfrow = c(2, 1), mar = c(4, 4, 2, 1), oma = c(4, 0, 0, 0))

# Index range and plotting
idx <- 1:TT
components <- c(1, 2, 4, 6)
component_labels <- c("SL - Trend Component", "SL - First Harmonic", "SL - Second Harmonic", "SL - (1/6.83) Harmonic")
for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component(q_d_50_SL,q_d_05_SL, q_d_95_SL, Y, idx, components[i], component_labels[i], num_ticks = 8)
}
# Add a common legend or note at the bottom
mtext("Legend: Forest Green - 50th, Dark Red - 05th, Dark Blue - 95th", side = 1, outer = TRUE, line = 2, cex = 0.8)

component_labels <- c("AV - Trend Component", "AV - First Harmonic", "AV - Second Harmonic", "AV - (1/6.83) Harmonic")
for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component(q_d_50_AV,q_d_05_AV, q_d_95_AV, Y, idx, components[i], component_labels[i], num_ticks = 8)
}
# Add a common legend or note at the bottom
mtext("Legend: Forest Green - 50th, Dark Red - 05th, Dark Blue - 95th", side = 1, outer = TRUE, line = 2, cex = 0.8)

component_labels <- c("SL vs AV - Trend Component", "SL vs AV - First Harmonic", "SL vs AV - Second Harmonic", "SL vs AV - (1/6.83) Harmonic")
for (i in 1:length(components)) {
  par(mar = c(4, 4, 2, 1) + 0.1)  # Reset margins at each iteration
  plot_quantile_component_all(q_d_50_SL,q_d_05_SL, q_d_95_SL, q_d_50_AV,q_d_05_AV, q_d_95_AV, Y, idx, components[i], component_labels[i], num_ticks = 8)
}
# Add a common legend or note at the bottom
mtext("Legend: Forest Green - 50th, Dark Red - 05th, Dark Blue - 95th", side = 1, outer = TRUE, line = 2, cex = 0.8)

# Reset plotting layout to default
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))


In [ ]:
# idx <- 15810:16032
idx <- (TT-500):TT

# Function to create nice date labels
create_date_labels <- function(idx, num_labels = 10) {
    selected_dates <- dates_ts_usgs[idx]  # assuming dates_ts_usgs is an array of dates corresponding to idx
    tick_positions <- pretty(idx, num_labels)
    tick_labels <- format(selected_dates[match(tick_positions, idx)], "%Y-%m-%d")
    return(list(tick_positions = tick_positions, tick_labels = tick_labels))
}

In [ ]:
# Setting up the plotting window
par(mfrow = c(2, 1), mar = c(3.2, 4, 2, 1) + 0.1, oma = c(4, 0, 0, 0))

# Calculate date labels outside of the plotting function for consistency
date_info <- create_date_labels(idx)

# First plot
plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="AV", xaxt="n")


lines(idx, quantiles_xb_AV_50_corrected[2, 1, idx], col="lightgreen", lwd=1) 
lines(idx, quantiles_xb_AV_50_corrected[3, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_50_corrected[1, 1, idx], col="lightgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_AV_05_corrected[2, 1, idx], col="pink", lwd=1) 
lines(idx, quantiles_xb_AV_05_corrected[3, 1, idx], col="pink", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_05_corrected[1, 1, idx], col="pink", lwd=0.5, lty=2)
lines(idx, quantiles_xb_AV_95_corrected[2, 1, idx], col="lightblue", lwd=1) 
lines(idx, quantiles_xb_AV_95_corrected[3, 1, idx], col="lightblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_95_corrected[1, 1, idx], col="lightblue", lwd=0.5, lty=2) 
axis(1, at = date_info$tick_positions, labels = FALSE)

# Adding rotated text labels
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Second plot
plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="SL", xaxt="n")

lines(idx, quantiles_xb_SL_50_corrected[2, 1, idx], col="darkgreen", lwd=1) 
lines(idx, quantiles_xb_SL_50_corrected[3, 1, idx], col="darkgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_50_corrected[1, 1, idx], col="darkgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_SL_05_corrected[2, 1, idx], col="darkred", lwd=1) 
lines(idx, quantiles_xb_SL_05_corrected[3, 1, idx], col="darkred", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_05_corrected[1, 1, idx], col="darkred", lwd=0.5, lty=2)
lines(idx, quantiles_xb_SL_95_corrected[2, 1, idx], col="darkblue", lwd=1) 
lines(idx, quantiles_xb_SL_95_corrected[3, 1, idx], col="darkblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_95_corrected[1, 1, idx], col="darkblue", lwd=0.5, lty=2) 
axis(1, at = date_info$tick_positions, labels = FALSE)

# Adding rotated text labels for the second plot
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)



In [ ]:

date_info <- create_date_labels(idx)

plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="AV vs SL: 95th", xaxt="n")

lines(idx, quantiles_xb_AV_95_corrected[2, 1, idx], col="lightblue", lwd=1) 
lines(idx, quantiles_xb_AV_95_corrected[3, 1, idx], col="lightblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_95_corrected[1, 1, idx], col="lightblue", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_SL_95_corrected[2, 1, idx], col="darkblue", lwd=1) 
lines(idx, quantiles_xb_SL_95_corrected[3, 1, idx], col="darkblue", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_95_corrected[1, 1, idx], col="darkblue", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)


plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="AV vs SL: 05th", xaxt="n")

lines(idx, quantiles_xb_AV_05_corrected[2, 1, idx], col="pink", lwd=1) 
lines(idx, quantiles_xb_AV_05_corrected[3, 1, idx], col="pink", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_05_corrected[1, 1, idx], col="pink", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_SL_05_corrected[2, 1, idx], col="darkred", lwd=1) 
lines(idx, quantiles_xb_SL_05_corrected[3, 1, idx], col="darkred", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_05_corrected[1, 1, idx], col="darkred", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)


plot(idx, Y[1, idx], type="l", col="gray", lwd=2, 
     ylim=range(c(-2,5)), xlab=" ", ylab="log-flow", main="AV vs SL: 50th", xaxt="n")

lines(idx, quantiles_xb_AV_50_corrected[2, 1, idx], col="lightgreen", lwd=1) 
lines(idx, quantiles_xb_AV_50_corrected[3, 1, idx], col="lightgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_AV_50_corrected[1, 1, idx], col="lightgreen", lwd=0.5, lty=2) 
lines(idx, quantiles_xb_SL_50_corrected[2, 1, idx], col="darkgreen", lwd=1) 
lines(idx, quantiles_xb_SL_50_corrected[3, 1, idx], col="darkgreen", lwd=0.5, lty=2)  
lines(idx, quantiles_xb_SL_50_corrected[1, 1, idx], col="darkgreen", lwd=0.5, lty=2) 

axis(1, at = date_info$tick_positions, labels = FALSE)
text(x = date_info$tick_positions, y = min(par("usr")[3], -2.8), labels = date_info$tick_labels, srt = 45, adj = 1, xpd = TRUE, cex = 0.8)

# Reset the plotting parameters to default after plotting is done
par(mfrow = c(1, 1), mar = c(5, 4, 4, 2) + 0.1, oma = c(0, 0, 0, 0))



## Quantile Forecasts

In [ ]:
# df_t    <- 0.9991048
# df_s    <- 0.9982188
# df_s67  <- 0.9993456
# df_discrep <- rep(0.95, 3-1)
# data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned.csv"
# streamflow_data <- read_csv(data_path, show_col_types = FALSE)
# timestamps <- as.Date(streamflow_data$Date)
# time_series_matrix <- as.matrix(streamflow_data[, c('USGS')])
# Y <- t(time_series_matrix)
# m_yy <- mean(Y, na.rm = TRUE)
# s_yy <- sd(Y, na.rm = TRUE)  
# k <- 0.1*s_yy
# trend.comp = polytrendMod(1, m0 = m_yy, C0 = k)
# harm = c(1, 2, 1/6.8333333)   
# seas.comp = seasMod(p = 363.5854, h = harm , C0 = 0.5*k*diag(2*length(harm)))
# model = combineMods(trend.comp, seas.comp)
# y = Y;
# if(is.null(nrow(y))){ 
#     JJJ <- 1
#     y = array(y, c(JJJ,length(y)))
#  }else{
#     JJJ <- nrow(Y)
#     y = array(y, c(JJJ,ncol(y)))
#  }
# df = c(df_t,df_s, df_s67); dim.df = c(1, 2*length(harm)-2, 2); 
# n.samp = 2000; 
# verbose = TRUE; k = 5;
#   ########################
#   TT = dim(y)[2] 
#   J = dim(y)[1]-1 
#   p = length(model$m0) 
#   ########################
#   m0 = c(model$m0,rep(0,J))
#   C0 = bdiag(model$C0,diag(J))
#   ########################
#   df.mat = make_df_mat(df, dim.df, p)
#   df.mat.k = make_df_mat_k(df, dim.df, p, k)
#   ########################
#   if(J<=0){ # Chanhe to ==0
#   ex.df.mat <- df.mat
#   ex.df.mat.k <- df.mat.k 
#   }else{
#   extra_df.mat <- make_df_mat(df_discrep,rep(1,J),J)
#   extra_df.mat.k <- make_df_mat_k(df_discrep,rep(1,J),J,k)
#   ex.df.mat <- bdiag(df.mat, extra_df.mat)
#   ex.df.mat.k <- bdiag(df.mat.k, extra_df.mat.k)
#   }
#   ########################
#   GG = array( bdiag(model$GG,diag(J)), c(p+J, p+J, TT) )
#   F1 <- matrix(model$FF,p,J+1)
#   F2 <- cbind(rep(0,J),diag(J))
#   FF = array(rbind(F1,F2), c(p+J, 1+J, TT))

## NEEDS REVISION

In [ ]:
# Define parameters and initialize variables
tot_forecast <- 180
# p <- nrow(GG)  # Assuming GG is a square matrix
# J <- nrow(samp.theta_50_SL) - p  # Assuming the number of rows in samp.theta_50_SL is p + J
# n.samp <- dim(samp.theta_50_SL)[3]
# TT <- dim(GG)[3]  # Assuming TT is the last time point index

# Initialize forecast arrays
forecast_50_SL <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))
forecast_05_SL <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))
forecast_95_SL <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))

forecast_50_AV <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))
forecast_05_AV <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))
forecast_95_AV <- array(NA_real_, c(p + ppx + J, tot_forecast, n.samp))

# Forecast function
forecast_quantile <- function(delta, new_theta_out, samp_theta, forecast_array) {
  # df_s67 <- 0.9999
  # df <- c(delta, df_s67)

  df <- c(delta)
  for (k in 1:tot_forecast) {
    M_k <- make_df_mat_k(df, dim.df, p, 1)
    P <- GG[,,TT] %*% new_theta_out$sC[,,TT] %*% t(GG[,,TT])
    W <- P * M_k
    W_sqrt <- sqrtm(W)
    
    for (i in 1:n.samp) {
      e <- matrix(rnorm(p + J), p + J, 1)
      if (k == 1) {
        Q <- GG[,,TT] %*% samp_theta[,TT,i]
      } else {
        Q <- GG[,,TT] %*% forecast_array[,k - 1,i]
      }
      forecast_array[,k,i] <- Q + W_sqrt %*% e
    }
  }
  
  return(forecast_array)
}

# Perform forecasting for each quantile for both models
forecast_50_SL <- forecast_quantile(delta_50_SL, new.theta.out_50_SL, samp.theta_50_SL, forecast_50_SL)
forecast_05_SL <- forecast_quantile(delta_50_SL, new.theta.out_05_SL, samp.theta_05_SL, forecast_05_SL)
forecast_95_SL <- forecast_quantile(delta_95_SL, new.theta.out_95_SL, samp.theta_95_SL, forecast_95_SL)

forecast_50_AV <- forecast_quantile(delta_50_AV, new.theta.out_50_AV, samp.theta_50_AV, forecast_50_AV)
forecast_05_AV <- forecast_quantile(delta_50_AV, new.theta.out_05_AV, samp.theta_05_AV, forecast_05_AV)
forecast_95_AV <- forecast_quantile(delta_95_AV, new.theta.out_95_AV, samp.theta_95_AV, forecast_95_AV)

# Compute xb forecast
compute_xb_forecast <- function(samp_theta, F) {
  # Determine dimensions
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  xb <- array(NA, dim = c(J + 1, n_time, n_sim))
  
  for (t in 1:n_time) {
    theta_t_s <- samp_theta[, t, ]
    xb[, t, ] <- t(F) %*% theta_t_s  # Results in a 3x2000 matrix
  }
  
  return(xb)
}

# Compute xb forecast for each quantile for both models
xb_50_SL_forecast <- compute_xb_forecast(forecast_50_SL, FF[,,TT])
xb_95_SL_forecast <- compute_xb_forecast(forecast_95_SL, FF[,,TT])
xb_05_SL_forecast <- compute_xb_forecast(forecast_05_SL, FF[,,TT])

xb_50_AV_forecast <- compute_xb_forecast(forecast_50_AV, FF[,,TT])
xb_95_AV_forecast <- compute_xb_forecast(forecast_95_AV, FF[,,TT])
xb_05_AV_forecast <- compute_xb_forecast(forecast_05_AV, FF[,,TT])

# Calculate quantiles of the forecasted values
quantiles_xb_SL_50_forecast <- apply(xb_50_SL_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_SL_95_forecast <- apply(xb_95_SL_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_SL_05_forecast <- apply(xb_05_SL_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))

quantiles_xb_AV_50_forecast <- apply(xb_50_AV_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_AV_95_forecast <- apply(xb_95_AV_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_AV_05_forecast <- apply(xb_05_AV_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))


In [ ]:
# Load and process data
init <- TT-500
idx_f <- (TT + 1):(TT + tot_forecast)
yy <- c(Y[1, init:TT], rep(NA_real_, tot_forecast))
idx <- init:(TT + tot_forecast)

data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned_extra.csv"
all_streamflow_data <- read_csv(data_path, show_col_types = FALSE)
truth <- all_streamflow_data[(TT + 1):(TT + tot_forecast), 2]

# Plotting function
plot_forecast <- function(idx, yy, init, TT, new_theta_out_50, new_theta_out_05, new_theta_out_95, idx_f, quantiles_50, quantiles_05, quantiles_95, truth, title) {
  plot.ts(idx, yy, type = 'l', main = title, ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data
  lines(init:TT, new_theta_out_50$exps[1, init:TT], col = 'forestgreen')
  lines(init:TT, new_theta_out_05$exps[1, init:TT], col = 'darkred')
  lines(init:TT, new_theta_out_95$exps[1, init:TT], col = 'darkblue')
  
  # Add lines for forecast data
  lines(idx_f, quantiles_50[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_05[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_95[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[3,1,], col = 'lightblue', lty = 2)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth", "50th qntl", "5th qntl", "95th qntl", "Frcst 50th qntl", "Frcst 5th qntl", "Frcst 95th qntl"),
         col = c("black", "orange", "forestgreen", "darkred", "darkblue", "lightgreen", "pink", "lightblue"),
         lty = c(1, NA, 1, 1, 1, 2, 2, 2), pch = c(NA, 1, NA, NA, NA, NA, NA, NA), bty = "n")
}

# Plot forecasts for SL model
plot_forecast(idx, yy, init, TT, new.theta.out_50_SL, new.theta.out_05_SL, new.theta.out_95_SL, 
              idx_f, quantiles_xb_SL_50_forecast, quantiles_xb_SL_05_forecast, quantiles_xb_SL_95_forecast, truth, "SL Model Forecast")

# Plot forecasts for Averaged model
plot_forecast(idx, yy, init, TT, new.theta.out_50_AV, new.theta.out_05_AV, new.theta.out_95_AV, 
              idx_f, quantiles_xb_AV_50_forecast, quantiles_xb_AV_05_forecast, quantiles_xb_AV_95_forecast, truth, "Averaged Model Forecast")

# Combined Plotting function
plot_combined_forecast <- function(idx, yy, init, TT, new_theta_out_50_SL, new_theta_out_05_SL, new_theta_out_95_SL, quantiles_50_SL, quantiles_05_SL, quantiles_95_SL, new_theta_out_50_AV, new_theta_out_05_AV, new_theta_out_95_AV, quantiles_50_AV, quantiles_05_AV, quantiles_95_AV, idx_f, truth) {
  plot.ts(idx, yy, type = 'l', main = "Combined SL and Averaged Model Forecast", ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data
  lines(init:TT, new_theta_out_50_SL$exps[1, init:TT], col = 'lightgreen')
  lines(init:TT, new_theta_out_05_SL$exps[1, init:TT], col = 'pink')
  lines(init:TT, new_theta_out_95_SL$exps[1, init:TT], col = 'lightblue')
  lines(init:TT, new_theta_out_50_AV$exps[1, init:TT], col = 'darkgreen')
  lines(init:TT, new_theta_out_05_AV$exps[1, init:TT], col = 'darkred')
  lines(init:TT, new_theta_out_95_AV$exps[1, init:TT], col = 'darkblue')
  
  # Add lines for forecast data
  lines(idx_f, quantiles_50_SL[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50_SL[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50_SL[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_05_SL[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05_SL[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05_SL[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_95_SL[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95_SL[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95_SL[3,1,], col = 'lightblue', lty = 2)
  
  lines(idx_f, quantiles_50_AV[1,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_50_AV[2,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_50_AV[3,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_05_AV[1,1,], col = 'darkred', lty = 3)
  lines(idx_f, quantiles_05_AV[2,1,], col = 'darkred', lty = 3)
  lines(idx_f, quantiles_05_AV[3,1,], col = 'darkred', lty = 3)
  lines(idx_f, quantiles_95_AV[1,1,], col = 'darkblue', lty = 3)
  lines(idx_f, quantiles_95_AV[2,1,], col = 'darkblue', lty = 3)
  lines(idx_f, quantiles_95_AV[3,1,], col = 'darkblue', lty = 3)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth",  "SL: 50th qntl", "SL: 5th qntl", "SL: 95th qntl", "SL: Frcst 50th qntl", "SL: Frcst 5th qntl", "SL: Frcst 95th qntl", "AV: 50th qntl", "AV: 5th qntl", "AV: 95th qntl", "AV: Frcst 50th qntl", "AV: Frcst 5th qntl", "AV: Frcst 95th qntl"),
         col = c("black", "orange", "darkgreen", "darkred", "darkblue", "lightgreen", "pink", "lightblue", "lightgreen", "pink", "lightblue", "darkgreen", "darkred", "darkblue"),
         lty = c(1, NA, 1, 1, 1, 2, 2, 2, 1, 1, 1, 3, 3, 3), pch = c(NA, 1, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA), bty = "n")
}

# Combined plot for both models
plot_combined_forecast(idx, yy, init, TT, new.theta.out_50_SL, new.theta.out_05_SL, new.theta.out_95_SL, quantiles_xb_SL_50_forecast, quantiles_xb_SL_05_forecast, quantiles_xb_SL_95_forecast, new.theta.out_50_AV, new.theta.out_05_AV, new.theta.out_95_AV, quantiles_xb_AV_50_forecast, quantiles_xb_AV_05_forecast, quantiles_xb_AV_95_forecast, idx_f, truth)


In [ ]:
# df_t    <- 0.9991048
# df_s    <- 0.9982188
# df_s67  <- 0.9993456
# df_discrep <- rep(0.95, 3-1)
# data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned.csv"
# streamflow_data <- read_csv(data_path, show_col_types = FALSE)
# timestamps <- as.Date(streamflow_data$Date)
# time_series_matrix <- as.matrix(streamflow_data[, c('USGS')])
# Y <- t(time_series_matrix)
# m_yy <- mean(Y, na.rm = TRUE)
# s_yy <- sd(Y, na.rm = TRUE)  
# k <- 0.1*s_yy
# trend.comp = polytrendMod(1, m0 = m_yy, C0 = k)
# harm = c(1, 2, 1/6.8333333)   
# seas.comp = seasMod(p = 363.5854, h = harm , C0 = 0.5*k*diag(2*length(harm)))
# model = combineMods(trend.comp, seas.comp)
# y = Y;
# if(is.null(nrow(y))){ 
#     JJJ <- 1
#     y = array(y, c(JJJ,length(y)))
#  }else{
#     JJJ <- nrow(Y)
#     y = array(y, c(JJJ,ncol(y)))
#  }
# df = c(df_t,df_s, df_s67); dim.df = c(1, 2*length(harm)-2, 2); 
# n.samp = 2000; 
# verbose = TRUE; k = 5;
#   ########################
#   TT = dim(y)[2] 
#   J = dim(y)[1]-1 
#   p = length(model$m0) 
#   ########################
#   m0 = c(model$m0,rep(0,J))
#   C0 = bdiag(model$C0,diag(J))
#   ########################
#   df.mat = make_df_mat(df, dim.df, p)
#   df.mat.k = make_df_mat_k(df, dim.df, p, k)
#   ########################
#   if(J<=0){ # Chanhe to ==0
#   ex.df.mat <- df.mat
#   ex.df.mat.k <- df.mat.k 
#   }else{
#   extra_df.mat <- make_df_mat(df_discrep,rep(1,J),J)
#   extra_df.mat.k <- make_df_mat_k(df_discrep,rep(1,J),J,k)
#   ex.df.mat <- bdiag(df.mat, extra_df.mat)
#   ex.df.mat.k <- bdiag(df.mat.k, extra_df.mat.k)
#   }
#   ########################
#   GG = array( bdiag(model$GG,diag(J)), c(p+J, p+J, TT) )
#   F1 <- matrix(model$FF,p,J+1)
#   F2 <- cbind(rep(0,J),diag(J))
#   FF = array(rbind(F1,F2), c(p+J, 1+J, TT))

In [ ]:
# # Parameters and initial setup
# df_t    <- 0.9991048
# df_s    <- 0.9982188
# df_s67  <- 0.9993456
# df_discrep <- rep(0.95, 3-1)
# ###########################################################################################
# data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned.csv"
# streamflow_data <- read_csv(data_path, show_col_types = FALSE)
# timestamps <- as.Date(streamflow_data$Date, show_col_types = FALSE)
# time_series_matrix <- as.matrix(streamflow_data[, c('USGS', 'NWS3.0',  'GloFAS')])
# Y <- t(time_series_matrix)
# ###########################################################################################
# TT <- ncol(Y)  
# y = Y;
# ###########################################################################################
# m_yy <- mean(Y, na.rm = TRUE)
# s_yy <- sd(Y, na.rm = TRUE)  
# k <- 0.1*s_yy
# trend.comp = polytrendMod(1, m0 = m_yy, C0 = k)
# harm = harmonics
# seas.comp = seasMod(p = 363.5854, h = harm , C0 = 0.5*k*diag(2*length(harm)))
# model = combineMods(trend.comp, seas.comp)
# if(is.null(nrow(y))){ 
#     JJJ <- 1
#     y = array(y, c(JJJ,length(y)))
#  }else{
#     JJJ <- nrow(Y)
#     y = array(y, c(JJJ,ncol(y)))
#  }
# df = c(df_t,df_s, df_s67); 
# dim.df = c(1, 2*length(harm)-2, 2); 
# gam.init = array(rep(0.01,JJJ), c(JJJ,1)); 
# sig.init = array(rep(0.1,JJJ), c(JJJ,1));
# n.samp = 2000; 
# PriorSigma = array(NA_real_, c(JJJ,2)); 
# PriorGamma = array(NA_real_, c(JJJ,3)); 
# verbose = TRUE; k = 5;
# ###########################################################################################
#   TT = dim(y)[2] 
#   J = dim(y)[1]-1 
#   p = length(model$m0) 
# ###########################################################################################
#   m0 = c(model$m0,rep(0,J))
#   C0 = bdiag(model$C0,diag(J))
# ###########################################################################################
#   model_simp <- model
#   df_simp <- df
#   dim.df_simp <- dim.df
#   model_simp$GG = array(model_simp$GG, c(p, p, TT))
#   model_simp$FF = array(model_simp$FF, c(p, 1, TT))
#   df.mat = make_df_mat(df, dim.df, p)
#   df.mat.k = make_df_mat_k(df, dim.df, p, k)
# ###########################################################################################
#   if(J<=0){
#   ex.df.mat <- df.mat
#   ex.df.mat.k <- df.mat.k 
#   }else{
#   extra_df.mat <- make_df_mat(df_discrep,rep(1,J),J)
#   extra_df.mat.k <- make_df_mat_k(df_discrep,rep(1,J),J,k)
#   ex.df.mat <- bdiag(df.mat, extra_df.mat)
#   ex.df.mat.k <- bdiag(df.mat.k, extra_df.mat.k)
#   }
# ###########################################################################################
#   GG = array( bdiag(model$GG,diag(J)), c(p+J, p+J, TT) )
#   model$GG = GG
#   F1 <- matrix(model$FF,p,J+1)
#   F2 <- cbind(rep(0,J),diag(J))
#   FF = array(rbind(F1,F2), c(p+J, 1+J, TT))
#   model$FF = FF


In [ ]:
# Define parameters and initialize variables
n.samp <- dim(samp.theta_50_M)[3]

# Initialize forecast arrays
forecast_50_M <- array(NA_real_, c(p + J, tot_forecast, n.samp))
forecast_05_M <- array(NA_real_, c(p + J, tot_forecast, n.samp))
forecast_95_M <- array(NA_real_, c(p + J, tot_forecast, n.samp))

# Function to compute matrix square root
sqrtm <- function(mat) {
  e <- eigen(mat)
  e$vectors %*% diag(sqrt(e$values)) %*% t(e$vectors)
}

# Forecast function
forecast_quantile <- function(delta, new_theta_out, samp_theta, forecast_array) {
  df <- delta[1:3]
  df_discrep <- rep(delta[4], J)

  for (k in 1:tot_forecast) {
    
    df.mat.k = make_df_mat_k(df, dim.df, p, 1)
    extra_df.mat.k <- make_df_mat_k(df_discrep,rep(1,J),J,k)
    M_k <- bdiag(df.mat.k, extra_df.mat.k)

    P <- GG[,,TT] %*% new_theta_out$sC[,,TT] %*% t(GG[,,TT])
    W <- P * M_k
    W_sqrt <- sqrtm(W)
    
    for (i in 1:n.samp) {
      e <- matrix(rnorm(p + J), p + J, 1)
      if (k == 1) {
        Q <- GG[,,TT] %*% samp_theta[,TT,i]
      } else {
        Q <- GG[,,TT] %*% forecast_array[,k - 1,i]
      }
      forecast_array[,k,i] <- Q + W_sqrt %*% e
    }
  }
  
  return(forecast_array)
}

# Perform forecasting for each quantile for the exAL model
forecast_50_M <- forecast_quantile(delta_50_M, new.theta.out_50_M, samp.theta_50_M, forecast_50_M)
forecast_05_M <- forecast_quantile(delta_50_M, new.theta.out_5_M, samp.theta_5_M, forecast_05_M)
forecast_95_M <- forecast_quantile(delta_95_M, new.theta.out_95_M, samp.theta_95_M, forecast_95_M)

# Compute xb forecast
compute_xb_forecast <- function(samp_theta, F) {
  n_time <- dim(samp_theta)[2]
  n_sim <- dim(samp_theta)[3]
  
  xb <- array(NA, dim = c(J + 1, n_time, n_sim))
  
  for (t in 1:n_time) {
    theta_t_s <- samp_theta[, t, ]
    xb[, t, ] <- t(F) %*% theta_t_s  # Results in a 3x2000 matrix
  }
  
  return(xb)
}

# Compute xb forecast for each quantile for the exAL model
xb_50_M_forecast <- compute_xb_forecast(forecast_50_M, FF[,,TT])
xb_95_M_forecast <- compute_xb_forecast(forecast_95_M, FF[,,TT])
xb_05_M_forecast <- compute_xb_forecast(forecast_05_M, FF[,,TT])

# Calculate quantiles of the forecasted values
quantiles_xb_M_50_forecast <- apply(xb_50_M_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_M_95_forecast <- apply(xb_95_M_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_M_05_forecast <- apply(xb_05_M_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))

# Load and process data
init <- 15900
idx_f <- (TT + 1):(TT + tot_forecast)
yy <- c(Y[1, init:TT], rep(NA_real_, tot_forecast))
idx <- init:(TT + tot_forecast)

data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned_extra.csv"
all_streamflow_data <- read_csv(data_path, show_col_types = FALSE)
truth <- all_streamflow_data[(TT + 1):(TT + tot_forecast), 2]

# Plotting function
plot_forecast <- function(idx, yy, init, TT, new_theta_out_50, new_theta_out_05, new_theta_out_95, idx_f, quantiles_50, quantiles_05, quantiles_95, truth, title) {
  plot.ts(idx, yy, type = 'l', main = title, ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data
  lines(init:TT, new_theta_out_50$exps[1, init:TT], col = 'forestgreen')
  lines(init:TT, new_theta_out_05$exps[1, init:TT], col = 'darkred')
  lines(init:TT, new_theta_out_95$exps[1, init:TT], col = 'darkblue')
  
  # Add lines for forecast data
  lines(idx_f, quantiles_50[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_05[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_95[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[3,1,], col = 'lightblue', lty = 2)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth", "50th qntl", "5th qntl", "95th qntl", "Frcst 50th qntl", "Frcst 5th qntl", "Frcst 95th qntl"),
         col = c("black", "orange", "forestgreen", "darkred", "darkblue", "lightgreen", "pink", "lightblue"),
         lty = c(1, NA, 1, 1, 1, 2, 2, 2), pch = c(NA, 1, NA, NA, NA, NA, NA, NA), bty = "n")
}

# Plot forecasts for the exAL model
plot_forecast(idx, yy, init, TT, new.theta.out_50_M, new.theta.out_5_M, new.theta.out_95_M, 
              idx_f, quantiles_xb_M_50_forecast, quantiles_xb_M_05_forecast, quantiles_xb_M_95_forecast, truth, "exAL Model Forecast")


In [ ]:

# Forecast function for NDLM
forecast_quantile_ndlm <- function(delta, new_theta_out, samp_theta, forecast_array, sig_samp, pp) {
  df <- delta[1:3]
  df_discrep <- rep(delta[4], J)
  
  for (k in 1:tot_forecast) {
    df.mat.k <- make_df_mat_k(df, dim.df, p, k)
    extra_df.mat.k <- make_df_mat_k(df_discrep, rep(1, J), J, k)
    M_k <- bdiag(df.mat.k, extra_df.mat.k)
    
    P <- GG[,,TT] %*% new_theta_out$sC[,,TT] %*% t(GG[,,TT])
    W <- P * M_k
    W_sqrt <- sqrtm(W)
    
    for (i in 1:n.samp) {
      e <- matrix(rnorm(p + J), p + J, 1)
      if (k == 1) {
        Q <- GG[,,TT] %*% samp_theta[,TT,i]
      } else {
        Q <- GG[,,TT] %*% forecast_array[,k - 1,i]
      }
      forecast_array[,k,i] <- Q + W_sqrt %*% e
    }
  }
  
  # Add quantile adjustment
  for (i in 1:n.samp) {
    forecast_array[,,i] <- forecast_array[,,i] + sqrt(sig_samp[i,1]) * qnorm(pp)
  }
  
  return(forecast_array)
}

# Parameters for NDLM model
df_t    <- 0.9992
df_s    <- 0.997
df_s67  <- 0.9992
df_discrep <- rep(0.94, 3-1)
delta_ndlm <- c(df_t, df_s, df_s67, df_discrep[1])

# Initialize forecast arrays for NDLM
forecast_50_NDLM <- array(NA_real_, c(p + J, tot_forecast, n.samp))
xb_05_NDLM_forecast <- array(NA_real_, c(1, tot_forecast, n.samp))
xb_95_NDLM_forecast <- array(NA_real_, c(1, tot_forecast, n.samp))

forecast_50_NDLM <- forecast_quantile_ndlm(delta_ndlm, new.theta.out_M, samp.theta_M, forecast_50_NDLM, samp.sigma_M, 0.5)
xb_50_NDLM_forecast <- compute_xb_forecast(forecast_50_NDLM, FF[,,TT])

for (i in 1:n.samp) {
  xb_05_NDLM_forecast[,,i] <- xb_50_NDLM_forecast[1,,i] + sqrt(samp.sigma_M[i,1]) * qnorm(0.05)
  xb_95_NDLM_forecast[,,i] <- xb_50_NDLM_forecast[1,,i] + sqrt(samp.sigma_M[i,1]) * qnorm(0.95)
}

quantiles_xb_NDLM_50_forecast <- apply(xb_50_NDLM_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_NDLM_05_forecast <- apply(xb_05_NDLM_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))
quantiles_xb_NDLM_95_forecast <- apply(xb_95_NDLM_forecast, c(1, 2), function(x) quantile(x, probs = c(0.025, 0.5, 0.975)))

# Plotting function
plot_forecast_ndlm <- function(idx, yy, init, TT, xb_50, xb_05, xb_95, idx_f, quantiles_50, quantiles_05, quantiles_95, truth, title) {
  plot.ts(idx, yy, type = 'l', main = title, ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data
  lines(init:TT, rowMeans(xb_50[1,,])[init:TT], col = 'forestgreen')
  lines(init:TT, rowMeans(xb_05[1,,])[init:TT], col = 'darkred')
  lines(init:TT, rowMeans(xb_95[1,,])[init:TT], col = 'darkblue')
  
  # Add lines for forecast data
  lines(idx_f, quantiles_50[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_50[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_05[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_05[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_95[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_95[3,1,], col = 'lightblue', lty = 2)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth", "50th qntl", "5th qntl", "95th qntl", "Frcst 50th qntl", "Frcst 5th qntl", "Frcst 95th qntl"),
         col = c("black", "orange", "forestgreen", "darkred", "darkblue", "lightgreen", "pink", "lightblue"),
         lty = c(1, NA, 1, 1, 1, 2, 2, 2), pch = c(NA, 1, NA, NA, NA, NA, NA, NA), bty = "n")
}
# Plot forecasts for the NDLM model
plot_forecast_ndlm(idx, yy, init, TT, xb_M_50, xb_M_05, xb_M_95, 
              idx_f, quantiles_xb_NDLM_50_forecast, quantiles_xb_NDLM_05_forecast, quantiles_xb_NDLM_95_forecast, truth, "NDLM Model Forecast")

In [ ]:
# Combined Plotting function for exAL and NDLM models
plot_combined_forecast_ndlm_exal <- function(idx, yy, init, TT, new_theta_out_50_exal, new_theta_out_05_exal, new_theta_out_95_exal, 
                                             quantiles_exal_50, quantiles_exal_05, quantiles_exal_95, 
                                             xb_50_ndlm, xb_05_ndlm, xb_95_ndlm, 
                                             quantiles_ndlm_50, quantiles_ndlm_05, quantiles_ndlm_95, 
                                             idx_f, truth) {
  plot.ts(idx, yy, type = 'l', main = "Combined exAL and NDLM Model Forecast", ylab = "Value", xlab = "Time", col = 'black')
  
  # Add lines for historical data for exAL
  lines(init:TT, new_theta_out_50_exal$exps[1, init:TT], col = 'lightgreen')
  lines(init:TT, new_theta_out_05_exal$exps[1, init:TT], col = 'pink')
  lines(init:TT, new_theta_out_95_exal$exps[1, init:TT], col = 'lightblue')
  
  lines(init:TT, rowMeans(xb_50_ndlm[1,,])[init:TT], col = 'darkgreen')
  lines(init:TT, rowMeans(xb_05_ndlm[1,,])[init:TT], col = 'darkred')
  lines(init:TT, rowMeans(xb_95_ndlm[1,,])[init:TT], col = 'darkblue')
  

  # Add lines for forecast data for exAL
  lines(idx_f, quantiles_exal_50[1,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_exal_50[2,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_exal_50[3,1,], col = 'lightgreen', lty = 2)
  lines(idx_f, quantiles_exal_05[1,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_exal_05[2,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_exal_05[3,1,], col = 'pink', lty = 2)
  lines(idx_f, quantiles_exal_95[1,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_exal_95[2,1,], col = 'lightblue', lty = 2)
  lines(idx_f, quantiles_exal_95[3,1,], col = 'lightblue', lty = 2)
  
  # Add lines for forecast data for NDLM
  lines(idx_f, quantiles_ndlm_50[1,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_ndlm_50[2,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_ndlm_50[3,1,], col = 'darkgreen', lty = 3)
  lines(idx_f, quantiles_ndlm_05[1,1,], col = 'red', lty = 3)
  lines(idx_f, quantiles_ndlm_05[2,1,], col = 'red', lty = 3)
  lines(idx_f, quantiles_ndlm_05[3,1,], col = 'red', lty = 3)
  lines(idx_f, quantiles_ndlm_95[1,1,], col = 'blue', lty = 3)
  lines(idx_f, quantiles_ndlm_95[2,1,], col = 'blue', lty = 3)
  lines(idx_f, quantiles_ndlm_95[3,1,], col = 'blue', lty = 3)
  
  # Add points for truth data
  points(idx_f, truth$std_discharge_cms, col = 'orange')
  
  # Add legend
  legend("topleft", legend = c("Observed", "Truth", "exAL: 50th qntl", "exAL: 5th qntl", "exAL: 95th qntl", 
                               "NDLM: 50th qntl", "NDLM: 5th qntl", "NDLM: 95th qntl"),
         col = c("black", "orange", "lightgreen", "pink", "lightblue", "darkgreen", "darkred", "darkblue"),
         lty = c(1, NA, 1, 1, 1, 1, 1, 1), pch = c(NA, 1, NA, NA, NA, NA, NA, NA), bty = "n")
}


# Combined plot for exAL and NDLM models
plot_combined_forecast_ndlm_exal(idx, yy, init, TT, new.theta.out_50_M, new.theta.out_5_M, new.theta.out_95_M, 
                                 quantiles_xb_M_50_forecast, quantiles_xb_M_05_forecast, quantiles_xb_M_95_forecast, 
                                 xb_M_50, xb_M_05, xb_M_95, 
                                 quantiles_xb_NDLM_50_forecast, quantiles_xb_NDLM_05_forecast, quantiles_xb_NDLM_95_forecast, 
                                 idx_f, truth)


# Scores for forecasts

In [ ]:
CheckLossFn = function(p0,diff){diff*p0 - diff*as.numeric(diff<0)}

In [ ]:
a_CRPS <- function(y,q1,q2,q3,ps){
    d1 <- y-q1; d2 <- y-q2; d3 <- y-q3;
    score <- CheckLossFn(ps[1],d1)+CheckLossFn(ps[2],d2)+CheckLossFn(ps[3],d3)
    return(score)
}

IS_a <- function(l,u,a,y){
    score <- (u-l)+2/a*(l-y)*ifelse(y<l,1,0)+2/a*(y-u)*ifelse(y>=u,1,0)
    return(score)
}

WIS_a <- function(K,m,L,U,A,y){
    score <- 0.5*abs(y-m)
    for(k in 1:K){
        score <- score +  A[k]/2*IS_a(L,U,A[k],y)
    }
    score <- score/(K+0.5)
    return(score)
}

Disp_a <- function(l,u,a){
    score <- a/2*(u-l)
    return(score)
}

IC_a <- function(y,l,u,a){
    score <- (ifelse(l<y,1,0)*ifelse(y<u,1,0))
    return(score)
}

CD_a <- function(y,l,u,a){
    score <- 1-a - IC_a(y,l,u,a)
    return(score)
}

QS_a <- function(l,u){
    score <- (u-l)
    return(score)
}

QS_a <- function(l,u){
    score <- (u-l)
    return(score)
}

## TODO: QB_a



## In-Sample Scoring

### a-CRPS, IS, WIS, Disp, IC, CD

In [ ]:
IS_AV <- array(NA_real_, c(1,TT) )
IS_SL <- array(NA_real_, c(1,TT) )
IS_NDLM <- array(NA_real_, c(1,TT) )
IS_exAL <- array(NA_real_, c(1,TT) )

a_CRPS_AV <- array(NA_real_, c(1,TT) )
a_CRPS_SL <- array(NA_real_, c(1,TT) )
a_CRPS_NDLM <- array(NA_real_, c(1,TT) )
a_CRPS_exAL <- array(NA_real_, c(1,TT) )

WIS_AV <- array(NA_real_, c(1,TT) )
WIS_SL <- array(NA_real_, c(1,TT) )
WIS_NDLM <- array(NA_real_, c(1,TT) )
WIS_exAL <- array(NA_real_, c(1,TT) )

DISP_AV <- array(NA_real_, c(1,TT) )
DISP_SL <- array(NA_real_, c(1,TT) )
DISP_NDLM <- array(NA_real_, c(1,TT) )
DISP_exAL <- array(NA_real_, c(1,TT) )

IC_AV <- array(NA_real_, c(1,TT) )
IC_SL <- array(NA_real_, c(1,TT) )
IC_NDLM <- array(NA_real_, c(1,TT) )
IC_exAL <- array(NA_real_, c(1,TT) )

CD_SL <- array(NA_real_, c(1,TT) )
CD_AV <- array(NA_real_, c(1,TT) )
CD_NDLM <- array(NA_real_, c(1,TT) )
CD_exAL <- array(NA_real_, c(1,TT) )

QS_SL <- array(NA_real_, c(1,TT) )
QS_AV <- array(NA_real_, c(1,TT) )
QS_NDLM <- array(NA_real_, c(1,TT) )
QS_exAL <- array(NA_real_, c(1,TT) )

for(t in 1:TT){
    yt <- Y[1,t]
    a <- 0.1 
    A <- c(a)

    Ut <- xb_95_AV_corrected[1,t,]
    Lt <- xb_05_AV_corrected[1,t,]
    Mt <- xb_50_AV_corrected[1,t,]
    
    IS_AV[t] <- mean(IS_a(Lt,Ut,a,yt))
    a_CRPS_AV[t]<- mean(a_CRPS(yt,Lt,Mt,Ut,c(0.05,0.5,0.95)))
    WIS_AV[t] <- mean(WIS_a(1,Mt,Lt,Ut,A,yt))
    DISP_AV[t] <- mean(Disp_a(Lt,Ut,a))
    IC_AV[t] <- mean(IC_a(yt,Lt,Ut,a))
    CD_AV[t] <- mean(CD_a(yt,Lt,Ut,a))
    QS_AV[t] <- mean(QS_a(Lt,Ut))

    Ut <- xb_95_SL_corrected[1,t,]
    Lt <- xb_05_SL_corrected[1,t,]
    Mt <- xb_50_SL_corrected[1,t,]

    IS_SL[t] <- mean(IS_a(Lt,Ut,a,yt))
    a_CRPS_SL[t]<- mean(a_CRPS(yt,Lt,Mt,Ut,c(0.05,0.5,0.95)))
    WIS_SL[t] <- mean(WIS_a(1,Mt,Lt,Ut,A,yt))
    DISP_SL[t] <- mean(Disp_a(Lt,Ut,a))
    IC_SL[t] <- mean(IC_a(yt,Lt,Ut,a))
    CD_SL[t] <- mean(CD_a(yt,Lt,Ut,a))
    QS_SL[t] <- mean(QS_a(Lt,Ut))

    Ut <- xb_M_95[1,t,]
    Lt <- xb_M_05[1,t,]
    Mt <- xb_M_50[1,t,]

    IS_NDLM[t] <- mean(IS_a(Lt,Ut,a,yt))
    a_CRPS_NDLM[t]<- mean(a_CRPS(yt,Lt,Mt,Ut,c(0.05,0.5,0.95)))
    WIS_NDLM[t] <- mean(WIS_a(1,Mt,Lt,Ut,A,yt))
    DISP_NDLM[t] <- mean(Disp_a(Lt,Ut,a))
    IC_NDLM[t] <- mean(IC_a(yt,Lt,Ut,a))
    CD_NDLM[t] <- mean(CD_a(yt,Lt,Ut,a))
    QS_NDLM[t] <- mean(QS_a(Lt,Ut))

    Ut <- xb_95_corrected[1,t,]
    Lt <- xb_05_corrected[1,t,]
    Mt <- xb_50_corrected[1,t,]

    IS_exAL[t] <- mean(IS_a(Lt,Ut,a,yt))
    a_CRPS_exAL[t]<- mean(a_CRPS(yt,Lt,Mt,Ut,c(0.05,0.5,0.95)))
    WIS_exAL[t] <- mean(WIS_a(1,Mt,Lt,Ut,A,yt))
    DISP_exAL[t] <- mean(Disp_a(Lt,Ut,a))
    IC_exAL[t] <- mean(IC_a(yt,Lt,Ut,a))
    CD_exAL[t] <- mean(CD_a(yt,Lt,Ut,a))
    QS_exAL[t] <- mean(QS_a(Lt,Ut))

}

add windowed with rolling mean

In [ ]:
mean(IS_AV[1,])
mean(IS_SL[1,])
mean(IS_NDLM[1,])
mean(IS_exAL[1,])

plot.ts(IS_AV[1,])
lines(IS_SL[1,], col='darkred')
lines(IS_NDLM[1,], col='darkgreen')
lines(IS_exAL[1,], col='purple')

plot.ts(cumsum(IS_AV[1,])/TT)
lines(cumsum(IS_SL[1,])/TT, col='darkred')
lines(cumsum(IS_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(IS_exAL[1,])/TT, col='purple')

In [ ]:
mean(a_CRPS_AV[1,])
mean(a_CRPS_SL[1,])
mean(a_CRPS_NDLM[1,])
mean(a_CRPS_exAL[1,])

plot.ts(a_CRPS_AV[1,])
lines(a_CRPS_SL[1,], col='darkred')
lines(a_CRPS_NDLM[1,], col='darkgreen')
lines(a_CRPS_exAL[1,], col='purple')

plot.ts(cumsum(a_CRPS_AV[1,])/TT)
lines(cumsum(a_CRPS_SL[1,])/TT, col='darkred')
lines(cumsum(a_CRPS_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(a_CRPS_exAL[1,])/TT, col='purple')

In [ ]:
mean(WIS_AV[1,])
mean(WIS_SL[1,])
mean(WIS_NDLM[1,])
mean(WIS_exAL[1,])

plot.ts(WIS_AV[1,])
lines(WIS_SL[1,], col='darkred')
lines(WIS_NDLM[1,], col='darkgreen')
lines(WIS_exAL[1,], col='purple')

plot.ts(cumsum(WIS_AV[1,])/TT)
lines(cumsum(WIS_SL[1,])/TT, col='darkred')
lines(cumsum(WIS_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(WIS_exAL[1,])/TT, col='purple')

In [ ]:
mean(DISP_AV[1,])
mean(DISP_SL[1,])
mean(DISP_NDLM[1,])
mean(DISP_exAL[1,])

plot.ts(DISP_AV[1,])
lines(DISP_SL[1,], col='darkred')
lines(DISP_NDLM[1,], col='darkgreen')
lines(DISP_exAL[1,], col='purple')

plot.ts(cumsum(DISP_AV[1,])/TT, ylim=c(0,0.1))
lines(cumsum(DISP_SL[1,])/TT, col='darkred')
lines(cumsum(DISP_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(DISP_exAL[1,])/TT, col='purple')

In [ ]:
mean(IC_AV[1,])
mean(IC_SL[1,])
mean(IC_NDLM[1,])
mean(IC_exAL[1,])

# plot.ts(IC_AV[1,])
# lines(IC_SL[1,], col='darkred')
# lines(IC_NDLM[1,], col='darkgreen')

plot.ts(cumsum(IC_AV[1,])/TT, ylim = c(0,1))
lines(cumsum(IC_SL[1,])/TT, col='darkred')
lines(cumsum(IC_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(IC_exAL[1,])/TT, col='purple')
abline(h=0.9, col='orange')

In [ ]:
mean(CD_AV[1,])
mean(CD_SL[1,])
mean(CD_NDLM[1,])
mean(CD_exAL[1,])

# plot.ts(CD_AV[1,])
# lines(CD_SL[1,], col='darkred')
# lines(CD_NDLM[1,], col='darkgreen')

plot.ts(cumsum(CD_AV[1,])/TT,ylim = c(-0.1,0.3))
lines(cumsum(CD_SL[1,])/TT, col='darkred')
lines(cumsum(CD_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(CD_exAL[1,])/TT, col='purple')
abline(h=0, col='orange')

In [ ]:
mean(QS_AV[1,])
mean(QS_SL[1,])
mean(QS_NDLM[1,])
mean(QS_exAL[1,])

plot.ts(QS_AV[1,])
lines(QS_SL[1,], col='darkred')
lines(QS_NDLM[1,], col='darkgreen')
lines(QS_exAL[1,], col='purple')


plot.ts(cumsum(QS_AV[1,])/TT,ylim = c(0,3))
lines(cumsum(QS_SL[1,])/TT, col='darkred')
lines(cumsum(QS_NDLM[1,])/TT, col='darkgreen')
lines(cumsum(QS_exAL[1,])/TT, col='purple')

In [ ]:
load_variables <- function(filename, dir_path) {
  file_path <- file.path(dir_path, filename)
  load(file_path)
  cat("Variables loaded from:", file_path, "\n")
}

file_path <- "/home/jaguir26/project1_ucsc_phd/variables_50_exAL.RData"
load(file_path)

In [ ]:
# library(ks)
# library(MASS)

# # Function to estimate differential entropy using KDE for univariate data
# estimate_differential_entropy_kde_univariate <- function(data) {
#   kde_result <- kde(data)
#   estimates <- kde_result$estimate
#   estimates[estimates <= 0] <- .Machine$double.eps # Prevent log(0) issues
#   log_estimates <- log(estimates)
#   log_estimates[!is.finite(log_estimates)] <- 0 # Handle non-finite values
#   entropy_estimate <- -sum(estimates * log_estimates) * diff(kde_result$eval.points)[1]
#   return(entropy_estimate)
# }

# # Function to estimate differential entropy using KDE for multivariate data
# estimate_differential_entropy_kde_multivariate <- function(data) {
#   kde_result <- kde(data)
#   estimates <- kde_result$estimate
#   estimates[estimates <= 0] <- .Machine$double.eps # Prevent log(0) issues
#   log_estimates <- log(estimates)
#   log_estimates[!is.finite(log_estimates)] <- 0 # Handle non-finite values
#   entropy_estimate <- -sum(estimates * log_estimates) * prod(diff(kde_result$eval.points[[1]]))
#   return(entropy_estimate)
# }

# # Function to estimate the KL divergence D_KL(p || N(0, I)) for univariate data
# estimate_kl_divergence_univariate <- function(data) {
#   # Estimate the differential entropy H(p)
#   H_p <- estimate_differential_entropy_kde_univariate(data)
  
#   # Compute the expected value of the squared norm of the vectors
#   E_p_x2 <- mean(data^2)
  
#   # Dimensionality is 1 for univariate data
#   k <- 1
  
#   # Compute the KL divergence
#   kl_divergence <- -H_p + (k / 2) * log(2 * pi) + (1 / 2) * E_p_x2
  
#   return(kl_divergence)
# }

# # Function to estimate the KL divergence D_KL(p || N(0, I)) for multivariate data
# estimate_kl_divergence_multivariate <- function(data) {
#   # Estimate the differential entropy H(p)
#   H_p <- estimate_differential_entropy_kde_multivariate(data)
  
#   # Dimensionality of the vectors
#   k <- ncol(data)
  
#   # Compute the expected value of the squared norm of the vectors
#   E_p_xTx <- mean(rowSums(data^2))
  
#   # Compute the KL divergence
#   kl_divergence <- -H_p + (k / 2) * log(2 * pi) + (1 / 2) * E_p_xTx
  
#   return(kl_divergence)
# }

# # Wrapper function for any sample
# compute_kl_divergence <- function(sample) {
#   # Ensure the input sample is a matrix
#   sample <- as.matrix(sample)
  
#   # Determine if the sample is univariate or multivariate
#   if (ncol(sample) == 1) {
#     kl_divergence <- estimate_kl_divergence_univariate(sample)
#   } else {
#     kl_divergence <- estimate_kl_divergence_multivariate(sample)
#   }
  
#   return(kl_divergence)
# }

# # Example usage
# set.seed(123)
# sample_size <- 1000

# # Univariate case
# data_univariate <- rnorm(sample_size, mean = 2, sd = 2)
# kl_divergence_univariate <- compute_kl_divergence(data_univariate)
# print(kl_divergence_univariate)

# # Multivariate case
# data_multivariate <- MASS::mvrnorm(sample_size, mu = c(0, 0, 0), Sigma = diag(3))
# kl_divergence_multivariate <- compute_kl_divergence(data_multivariate)
# print(kl_divergence_multivariate)


In [ ]:
errors <- t(new.theta.out_50_exAL$standard_forecast_errors)
kl_divergence <- compute_kl_divergence(errors)
print(kl_divergence)

errors <- matrix(new.theta.out_50_exAL$standard_forecast_errors[2,], ncol = 1)
kl_divergence <- compute_kl_divergence(errors)
print(kl_divergence)

In [ ]:
plot.ts(new.theta.out_50_exAL$standard_forecast_errors[1,])

In [ ]:
# Function to compute descriptive statistics
descriptive_stats <- function(data) {
  stats <- data.frame(
    Mean = apply(data, 1, mean),
    Variance = apply(data, 1, var),
    Skewness = apply(data, 1, function(x) mean((x - mean(x))^3) / (sd(x)^3)),
    Kurtosis = apply(data, 1, function(x) mean((x - mean(x))^4) / (sd(x)^4) - 3)
  )
  return(stats)
}

# Compute and print descriptive statistics for the residuals
residual_stats <- descriptive_stats(errors)
print(residual_stats)


Compare with barata

In [ ]:
# Residual plots
par(mfrow = c(3, 1))  # Set up a 3-row plot layout

# Residual plots for each dimension
for (i in 1:3) {
  plot(errors[i, ], main = paste("Residual Plot for Dimension", i), ylab = "Residuals")
  abline(h = 0, col = "red")
}

par(mfrow = c(1, 1))  # Reset to default layout


In [ ]:
# ACF plots for each dimension
par(mfrow = c(3, 1))  # Set up a 3-row plot layout

# ACF plots for each dimension
for (i in 1:3) {
  acf(errors[i, ], main = paste("ACF of Residuals for Dimension", i))
}

par(mfrow = c(1, 1))  # Reset to default layout


In [ ]:
# Histograms and Q-Q plots for each dimension
par(mfrow = c(2, 3))  # Set up a 3x2 plot layout

# Histograms
for (i in 1:3) {
  hist(errors[i, ], main = paste("Histogram of Residuals (Dimension", i, ")"), xlab = "Residual Value", breaks = 50)
}

# Q-Q plots
for (i in 1:3) {
  qqnorm(errors[i, ], main = paste("Q-Q Plot of Residuals (Dimension", i, ")"))
  qqline(errors[i, ], col = "red")
}


par(mfrow = c(1, 1))  # Reset to default layout


In [ ]:
# Required Libraries
library(zoo)

# Check Loss Function
CheckLossFn <- function(p0, diff) {
  diff * p0 - diff * as.numeric(diff < 0)
}

# Approximated CRPS
a_CRPS <- function(y, quantiles, ps) {
  score <- 0
  for (i in 1:length(ps)) {
    diff <- y - quantiles[i]
    score <- score + CheckLossFn(ps[i], diff)
  }
  return(2 / length(ps) * score)
}

# Interval Score
IS_a <- function(l, u, a, y) {
  score <- (u - l) + 2/a * (l - y) * ifelse(y < l, 1, 0) + 2/a * (y - u) * ifelse(y >= u, 1, 0)
  return(score)
}

# Weighted Interval Score
WIS_a <- function(m, quantiles, ps, y, coverages) {
  K <- length(coverages)
  score <- 0.5 * abs(y - m)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    L <- quantiles[L_index]
    U <- quantiles[U_index]
    score <- score + alpha/2 * IS_a(L, U, alpha, y)
  }
  score <- score / (K + 0.5)
  return(score)
}

# Dispersion of WIS
Disp_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  score <- 0
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    score <- score + alpha/2 * (u - l)
  }
  return(score)
}

# Interval Coverage
IC_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
  }
  return(scores)
}

# Coverage Deviation
CD_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    IC_k <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
    scores[k] <- 1 - alpha - IC_k
  }
  return(scores)
}

# Quantile Sharpness
QS_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- (u - l)
  }
  return(mean(scores))
}

# Main function to compute all scores
compute_scores_general <- function(median_samples, quantile_samples, coverages, observed) {
  # Create the ps vector based on the provided quantile samples
  ps <- c(0.50)  # Always include the median
  for (i in 1:length(coverages)) {
    ps <- c(ps, (1 - coverages[i])/2, 1 - (1 - coverages[i])/2)
  }
  ps <- sort(unique(ps))
  
  n_times <- length(observed)
  n_samples <- dim(median_samples)[2]
  n_quantiles <- length(ps)
  
  quantiles_array <- array(NA, dim = c(n_quantiles, n_times, n_samples))
  quantiles_array[which.min(abs(ps - 0.50)), , ] <- median_samples  # Assuming median is at 0.50
  
  for (i in 1:length(coverages)) {
    quantiles_array[which.min(abs(ps - (1 - coverages[i])/2)), , ] <- quantile_samples[[2 * i - 1]]
    quantiles_array[which.min(abs(ps - (1 - (1 - coverages[i])/2))), , ] <- quantile_samples[[2 * i]]
  }
  
  # Initialize results data frame with dynamic columns based on coverages
  result_columns <- c("t", "CRPS", "WIS", "Dispersion", paste0("IC_", coverages * 100), paste0("CD_", coverages * 100), "QS")
  results <- data.frame(matrix(ncol = length(result_columns), nrow = n_times))
  colnames(results) <- result_columns
  results$t <- 1:n_times
  
  # Function to compute all scores
  compute_scores <- function(y, quantiles, ps, coverages) {
    list(
      CRPS = a_CRPS(y, quantiles, ps),
      WIS = WIS_a(quantiles[which.min(abs(ps - 0.50))], quantiles, ps, y, coverages),
      Dispersion = Disp_a(quantiles, ps, coverages),
      IC = IC_a(y, quantiles, ps, coverages),
      CD = CD_a(y, quantiles, ps, coverages),
      QS = QS_a(quantiles, ps, coverages)
    )
  }
  
  # Loop through all time points
  for (t in 1:n_times) {
    quantile_values <- quantiles_array[, t, 1]  # Use the first sample for simplicity
    # print(paste("Time:", t, "Quantile values:", paste(quantile_values, collapse = ", ")))
    scores <- compute_scores(observed[t], quantile_values, ps, coverages)
    
    results$CRPS[t] <- scores$CRPS
    results$WIS[t] <- scores$WIS
    results$Dispersion[t] <- scores$Dispersion
    for (i in 1:length(coverages)) {
      results[t, paste0("IC_", coverages[i] * 100)] <- scores$IC[i]
      results[t, paste0("CD_", coverages[i] * 100)] <- scores$CD[i]
    }
    results$QS[t] <- scores$QS
  }
  
  overall_means <- colMeans(results[, -1], na.rm = TRUE)
  return(list(results = results, overall_means = overall_means))
}

# Function to compute rolling mean, cumulative mean, and cumulative sum
compute_metrics <- function(series, window_size = 360) {
  roll_mean <- rollmean(series, window_size, fill = NA)
  cum_mean <- cumsum(series) / seq_along(series)
  cum_sum <- cumsum(series) / length(series)
  list(roll_mean = roll_mean, cum_mean = cum_mean, cum_sum = cum_sum)
}

# Plotting Function with Theoretical Coverage
plot_metric_with_coverage <- function(time, series, roll_mean, cum_mean, cum_sum, metric_name, theoretical_coverage = NULL) {
  par(mfrow = c(1, 3))  # Arrange plots in 1 row, 3 columns
  
  plot(time, series, type = "l", col = "darkblue", main = paste("Time Series of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2)
  lines(time, roll_mean, col = "orange", lwd = 2)
  legend("topright", legend = c("Series", "Rolling Mean"), col = c("darkblue", "orange"), lty = 1, lwd = 2)
  abline(h=0)
  
  plot(time, cum_mean, type = "l", col = "darkred", main = paste("Cumulative Mean of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2)
  abline(h=0)
  
  plot(time, cum_sum, type = "l", col = "darkgreen", main = paste("Cumulative Sum of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2)
  abline(h=0)
  
  if (!is.null(theoretical_coverage)) {
    abline(h = theoretical_coverage, col = "purple", lwd = 2, lty = 2)
  }
}

# Function to plot all metrics
plot_all_metrics <- function(results, coverages) {
  time <- results$t
  
  # CRPS
  crps_metrics <- compute_metrics(results$CRPS)
  plot_metric_with_coverage(time, results$CRPS, crps_metrics$roll_mean, crps_metrics$cum_mean, crps_metrics$cum_sum, "CRPS")
  
  # WIS
  wis_metrics <- compute_metrics(results$WIS)
  plot_metric_with_coverage(time, results$WIS, wis_metrics$roll_mean, wis_metrics$cum_mean, wis_metrics$cum_sum, "WIS")
  
  # Dispersion
  disp_metrics <- compute_metrics(results$Dispersion)
  plot_metric_with_coverage(time, results$Dispersion, disp_metrics$roll_mean, disp_metrics$cum_mean, disp_metrics$cum_sum, "Dispersion")
  
  # IC
  for (i in 1:length(coverages)) {
    ic_metrics <- compute_metrics(results[, paste0("IC_", coverages[i] * 100)])
    plot_metric_with_coverage(time, results[, paste0("IC_", coverages[i] * 100)], ic_metrics$roll_mean, ic_metrics$cum_mean, ic_metrics$cum_sum, paste0("IC_", coverages[i] * 100), theoretical_coverage = coverages[i])
  }
  
  # CD
  for (i in 1:length(coverages)) {
    cd_metrics <- compute_metrics(results[, paste0("CD_", coverages[i] * 100)])
    plot_metric_with_coverage(time, results[, paste0("CD_", coverages[i] * 100)], cd_metrics$roll_mean, cd_metrics$cum_mean, cd_metrics$cum_sum, paste0("CD_", coverages[i] * 100))
  }
  
  # QS
  qs_metrics <- compute_metrics(results$QS)
  plot_metric_with_coverage(time, results$QS, qs_metrics$roll_mean, qs_metrics$cum_mean, qs_metrics$cum_sum, "QS")
}


median_samples <- xb_50_corrected[1, , ]
# quantile_samples <- list(xb_05_corrected[1, , ], xb_95_corrected[1, , ], xb_20_corrected[1, , ], xb_80_corrected[1, , ], xb_35_corrected[1, , ], xb_65_corrected[1, , ])
# coverages <- c(0.9, 0.6, 0.3)
quantile_samples <- list(xb_05_corrected[1, , ], xb_95_corrected[1, , ])
coverages <- c(0.9)
observed <- Y[1, ]

results <- compute_scores_general(median_samples, quantile_samples, coverages, observed)
# print(results$results)
plot_all_metrics(results$results, coverages)


In [ ]:
# Required Libraries
library(zoo)

# Check Loss Function
CheckLossFn <- function(p0, diff) {
  diff * p0 - diff * as.numeric(diff < 0)
}

# Approximated CRPS
a_CRPS <- function(y, quantiles, ps) {
  score <- 0
  for (i in 1:length(ps)) {
    diff <- y - quantiles[i]
    score <- score + CheckLossFn(ps[i], diff)
  }
  return(2 / length(ps) * score)
}

# Interval Score
IS_a <- function(l, u, a, y) {
  score <- (u - l) + 2/a * (l - y) * ifelse(y < l, 1, 0) + 2/a * (y - u) * ifelse(y >= u, 1, 0)
  return(score)
}

# Weighted Interval Score
WIS_a <- function(m, quantiles, ps, y, coverages) {
  K <- length(coverages)
  score <- 0.5 * abs(y - m)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    L <- quantiles[L_index]
    U <- quantiles[U_index]
    score <- score + alpha/2 * IS_a(L, U, alpha, y)
  }
  score <- score / (K + 0.5)
  return(score)
}

# Dispersion of WIS
Disp_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  score <- 0
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    score <- score + alpha/2 * (u - l)
  }
  return(score)
}

# Interval Coverage
IC_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
  }
  return(scores)
}

# Coverage Deviation
CD_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    IC_k <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
    scores[k] <- 1 - alpha - IC_k
  }
  return(scores)
}

# Quantile Sharpness
QS_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- (u - l)
  }
  return(mean(scores))
}

# Main function to compute all scores
compute_scores_general <- function(median_samples, quantile_samples, coverages, observed) {
  # Create the ps vector based on the provided quantile samples
  ps <- c(0.50)  # Always include the median
  for (i in 1:length(coverages)) {
    ps <- c(ps, (1 - coverages[i])/2, 1 - (1 - coverages[i])/2)
  }
  ps <- sort(unique(ps))
  
  n_times <- length(observed)
  n_samples <- dim(median_samples)[2]
  n_quantiles <- length(ps)
  
  quantiles_array <- array(NA, dim = c(n_quantiles, n_times, n_samples))
  quantiles_array[which.min(abs(ps - 0.50)), , ] <- median_samples  # Assuming median is at 0.50
  
  for (i in 1:length(coverages)) {
    quantiles_array[which.min(abs(ps - (1 - coverages[i])/2)), , ] <- quantile_samples[[2 * i - 1]]
    quantiles_array[which.min(abs(ps - (1 - (1 - coverages[i])/2))), , ] <- quantile_samples[[2 * i]]
  }
  
  # Initialize results data frame with dynamic columns based on coverages
  result_columns <- c("t", "CRPS", "WIS", "Dispersion", paste0("IC_", coverages * 100), paste0("CD_", coverages * 100), "QS")
  results <- data.frame(matrix(ncol = length(result_columns), nrow = n_times))
  colnames(results) <- result_columns
  results$t <- 1:n_times
  
  # Function to compute all scores
  compute_scores <- function(y, quantiles, ps, coverages) {
    list(
      CRPS = a_CRPS(y, quantiles, ps),
      WIS = WIS_a(quantiles[which.min(abs(ps - 0.50))], quantiles, ps, y, coverages),
      Dispersion = Disp_a(quantiles, ps, coverages),
      IC = IC_a(y, quantiles, ps, coverages),
      CD = CD_a(y, quantiles, ps, coverages),
      QS = QS_a(quantiles, ps, coverages)
    )
  }
  
  # Loop through all time points
  for (t in 1:n_times) {
    quantile_values <- quantiles_array[, t, 1]  # Use the first sample for simplicity
    scores <- compute_scores(observed[t], quantile_values, ps, coverages)
    
    results$CRPS[t] <- scores$CRPS
    results$WIS[t] <- scores$WIS
    results$Dispersion[t] <- scores$Dispersion
    for (i in 1:length(coverages)) {
      results[t, paste0("IC_", coverages[i] * 100)] <- scores$IC[i]
      results[t, paste0("CD_", coverages[i] * 100)] <- scores$CD[i]
    }
    results$QS[t] <- scores$QS
  }
  
  overall_means <- colMeans(results[, -1], na.rm = TRUE)
  return(list(results = results, overall_means = overall_means))
}

# Function to compute rolling mean, cumulative mean, and cumulative sum
compute_metrics <- function(series, window_size = 360) {
  roll_mean <- rollmean(series, window_size, fill = NA)
  cum_mean <- cumsum(series) / seq_along(series)
  cum_sum <- cumsum(series) / length(series)
  list(roll_mean = roll_mean, cum_mean = cum_mean, cum_sum = cum_sum)
}
par(mfrow = c(1, 2)) 
plot_cumulative_metrics <- function(metrics_list, metric_name, models) {
  
  plot(metrics_list[[1]]$cum_mean, type = "l", col = "darkblue", main = paste("Cumulative Mean of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2, ylim = range(sapply(metrics_list, function(x) x$cum_mean), na.rm = TRUE) * 1.1)
  for (i in 2:length(metrics_list)) {
    lines(metrics_list[[i]]$cum_mean, col = i, lwd = 2)
  }
  legend("topleft", legend = c("Av", "SL", "NDLM", "exAL"), col = 1:length(models), lty = 1, lwd = 2, cex = 0.8)
  
  plot(metrics_list[[1]]$cum_sum, type = "l", col = "darkblue", main = paste("Cumulative Sum of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2, ylim = range(sapply(metrics_list, function(x) x$cum_sum), na.rm = TRUE) * 1.1)
  for (i in 2:length(metrics_list)) {
    lines(metrics_list[[i]]$cum_sum, col = i, lwd = 2)
  }
  legend("topleft", legend = c("Av", "SL", "NDLM", "exAL"), col = 1:length(models), lty = 1, lwd = 2, cex = 0.8)
}

# Function to compute and plot results for all models
compute_and_plot_all_models <- function() {
  models <- list(
    list(
      name = "Averaged Model",
      median_samples = xb_50_AV_corrected[1, , ],
      quantile_samples = list(xb_05_AV_corrected[1, , ], xb_95_AV_corrected[1, , ]),
      coverages = c(0.9)
    ),
    list(
      name = "SL Model",
      median_samples = xb_50_SL_corrected[1, , ],
      quantile_samples = list(xb_05_SL_corrected[1, , ], xb_95_SL_corrected[1, , ]),
      coverages = c(0.9)
    ),
    list(
      name = "NDLM Model",
      median_samples = xb_M_50[1, , ],
      quantile_samples = list(xb_M_05[1, , ], xb_M_95[1, , ]),
      coverages = c(0.9)
    ),
    list(
      name = "exAL Model",
      median_samples = xb_50_corrected[1, , ],
      quantile_samples = list(xb_05_corrected[1, , ], xb_95_corrected[1, , ]),
      coverages = c(0.9)
    )
  )
  
  observed <- Y[1, ]
  all_results <- list()
  overall_means_table <- data.frame()
  
  for (model in models) {
    cat("Processing", model$name, "\n")
    results <- compute_scores_general(model$median_samples, model$quantile_samples, model$coverages, observed)
    all_results[[model$name]] <- results
    overall_means_table <- rbind(overall_means_table, c(model$name, results$overall_means))
  }
  
  colnames(overall_means_table) <- c("Model", names(results$overall_means))
  print(overall_means_table)
  
  # Plot cumulative means and sums for each score
  scores <- c("CRPS", "WIS", "Dispersion", "QS")
  for (score in scores) {
    metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[score]]))
    plot_cumulative_metrics(metrics_list, score, names(all_results))
  }
  
  coverages <- unique(unlist(lapply(models, function(model) model$coverages)))
  for (coverage in coverages) {
    ic_metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[paste0("IC_", coverage * 100)]]))
    plot_cumulative_metrics(ic_metrics_list, paste0("IC_", coverage * 100), names(all_results))
    
    cd_metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[paste0("CD_", coverage * 100)]]))
    plot_cumulative_metrics(cd_metrics_list, paste0("CD_", coverage * 100), names(all_results))
  }
}

# Run the computation and plotting for all models
compute_and_plot_all_models()


In [ ]:
# Required Libraries
library(zoo)

# Check Loss Function
CheckLossFn <- function(p0, diff) {
  diff * p0 - diff * as.numeric(diff < 0)
}

# Approximated CRPS
a_CRPS <- function(y, quantiles, ps) {
  score <- 0
  for (i in 1:length(ps)) {
    diff <- y - quantiles[i]
    score <- score + CheckLossFn(ps[i], diff)
  }
  return(2 / length(ps) * score)
}

# Interval Score
IS_a <- function(l, u, a, y) {
  score <- (u - l) + 2/a * (l - y) * ifelse(y < l, 1, 0) + 2/a * (y - u) * ifelse(y >= u, 1, 0)
  return(score)
}

# Weighted Interval Score
WIS_a <- function(m, quantiles, ps, y, coverages) {
  K <- length(coverages)
  score <- 0.5 * abs(y - m)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    L <- quantiles[L_index]
    U <- quantiles[U_index]
    score <- score + alpha/2 * IS_a(L, U, alpha, y)
  }
  score <- score / (K + 0.5)
  return(score)
}

# Dispersion of WIS
Disp_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  score <- 0
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    score <- score + alpha/2 * (u - l)
  }
  return(score)
}

# Interval Coverage
IC_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
  }
  return(scores)
}

# Coverage Deviation
CD_a <- function(y, quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    IC_k <- ifelse(l < y, 1, 0) * ifelse(y < u, 1, 0)
    scores[k] <- 1 - alpha - IC_k
  }
  return(scores)
}

# Quantile Sharpness
QS_a <- function(quantiles, ps, coverages) {
  K <- length(coverages)
  scores <- numeric(K)
  for (k in 1:K) {
    alpha <- 1 - coverages[k]
    L_index <- which.min(abs(ps - alpha/2))
    U_index <- which.min(abs(ps - (1 - alpha/2)))
    l <- quantiles[L_index]
    u <- quantiles[U_index]
    scores[k] <- (u - l)
  }
  return(mean(scores))
}

# Main function to compute all scores
compute_scores_general <- function(median_samples, quantile_samples, coverages, observed) {
  # Create the ps vector based on the provided quantile samples
  ps <- c(0.50)  # Always include the median
  for (i in 1:length(coverages)) {
    ps <- c(ps, (1 - coverages[i])/2, 1 - (1 - coverages[i])/2)
  }
  ps <- sort(unique(ps))

  n_times <- length(observed)
  n_samples <- dim(median_samples)[2]
  n_quantiles <- length(ps)
  
  quantiles_array <- array(NA, dim = c(n_quantiles, n_times, n_samples))
  quantiles_array[which.min(abs(ps - 0.50)), , ] <- median_samples  # Assuming median is at 0.50
  
  for (i in 1:length(coverages)) {
    quantiles_array[which.min(abs(ps - (1 - coverages[i])/2)), , ] <- quantile_samples[[2 * i - 1]]
    quantiles_array[which.min(abs(ps - (1 - (1 - coverages[i])/2))), , ] <- quantile_samples[[2 * i]]
  }

  # Initialize results data frame with dynamic columns based on coverages
  result_columns <- c("t", "CRPS", "WIS", "Dispersion", paste0("IC_", coverages * 100), paste0("CD_", coverages * 100), "QS")
  results <- data.frame(matrix(ncol = length(result_columns), nrow = n_times))
  colnames(results) <- result_columns
  results$t <- 1:n_times
  
  # Function to compute all scores
  compute_scores <- function(y, quantiles, ps, coverages) {
    list(
      CRPS = a_CRPS(y, quantiles, ps),
      WIS = WIS_a(quantiles[which.min(abs(ps - 0.50))], quantiles, ps, y, coverages),
      Dispersion = Disp_a(quantiles, ps, coverages),
      IC = IC_a(y, quantiles, ps, coverages),
      CD = CD_a(y, quantiles, ps, coverages),
      QS = QS_a(quantiles, ps, coverages)
    )
  }
  
  # Loop through all time points
  for (t in 1:n_times) {
    quantile_values <- quantiles_array[, t, 1]  # Use the first sample for simplicity
    scores <- compute_scores(observed[t], quantile_values, ps, coverages)
    
    results$CRPS[t] <- scores$CRPS
    results$WIS[t] <- scores$WIS
    results$Dispersion[t] <- scores$Dispersion
    for (i in 1:length(coverages)) {
      results[t, paste0("IC_", coverages[i] * 100)] <- scores$IC[i]
      results[t, paste0("CD_", coverages[i] * 100)] <- scores$CD[i]
    }
    results$QS[t] <- scores$QS
  }
  
  overall_means <- colMeans(results[, -1], na.rm = TRUE)
  return(list(results = results, overall_means = overall_means))
}

# Function to compute rolling mean, cumulative mean, and cumulative sum
compute_metrics <- function(series, window_size = 360) {
  roll_mean <- rollmean(series, window_size, fill = NA)
  cum_mean <- cumsum(series) / seq_along(series)
  cum_sum <- cumsum(series) / length(series)
  list(roll_mean = roll_mean, cum_mean = cum_mean, cum_sum = cum_sum)
}

par(mfrow = c(3, 2))  # Arrange plots in 1 row, 2 columns
plot_cumulative_metrics <- function(metrics_list, metric_name, models) {
  
  plot(metrics_list[[1]]$cum_mean, type = "l", col = "darkblue", main = paste("Cumulative Mean of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2, ylim = range(sapply(metrics_list, function(x) x$cum_mean), na.rm = TRUE) * 1.1)
  for (i in 2:length(metrics_list)) {
    lines(metrics_list[[i]]$cum_mean, col = i, lwd = 2)
  }
  legend("topleft", legend = models, col = 1:length(models), lty = 1, lwd = 2, cex = 0.8)
  
  plot(metrics_list[[1]]$cum_sum, type = "l", col = "darkblue", main = paste("Cumulative Sum of", metric_name), ylab = metric_name, xlab = "Time", lwd = 2, ylim = range(sapply(metrics_list, function(x) x$cum_sum), na.rm = TRUE) * 1.1)
  for (i in 2:length(metrics_list)) {
    lines(metrics_list[[i]]$cum_sum, col = i, lwd = 2)
  }
  legend("topleft", legend = models, col = 1:length(models), lty = 1, lwd = 2, cex = 0.8)
}

# Function to compute and plot results for all models
compute_and_plot_all_models <- function() {
  models <- list(
    list(
      name = "Averaged Model",
      median_samples = xb_50_AV_forecast[1,,],
      quantile_samples = list(xb_05_AV_forecast[1,,], xb_95_AV_forecast[1,,]),
      coverages = c(0.9)
    ),
    list(
      name = "SL Model",
      median_samples = xb_50_SL_forecast[1,,],
      quantile_samples = list(xb_05_SL_forecast[1,,], xb_95_SL_forecast[1,,]),
      coverages = c(0.9)
    ),
    list(
      name = "NDLM Model",
      median_samples = xb_50_NDLM_forecast[1,,],
      quantile_samples = list(xb_05_NDLM_forecast[1,,], xb_95_NDLM_forecast[1,,]),
      coverages = c(0.9)
    ),
    list(
      name = "exAL Model",
      median_samples = xb_50_M_forecast[1,,],
      quantile_samples = list(xb_05_M_forecast[1,,], xb_95_M_forecast[1,,]),
      coverages = c(0.9)
    )
  )
  
  observed <- truth$std_discharge_cms
  all_results <- list()
  overall_means_table <- data.frame()
  
  for (model in models) {
    cat("Processing", model$name, "\n")
    results <- compute_scores_general(model$median_samples, model$quantile_samples, model$coverages, observed)
    all_results[[model$name]] <- results
    overall_means_table <- rbind(overall_means_table, c(model$name, results$overall_means))
  }
  
  colnames(overall_means_table) <- c("Model", names(results$overall_means))
  print(overall_means_table)
  
  # Plot cumulative means and sums for each score
  scores <- c("CRPS", "WIS", "Dispersion", "QS")
  for (score in scores) {
    metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[score]]))
    plot_cumulative_metrics(metrics_list, score, names(all_results))
  }
  
  coverages <- unique(unlist(lapply(models, function(model) model$coverages)))
  for (coverage in coverages) {
    ic_metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[paste0("IC_", coverage * 100)]]))
    plot_cumulative_metrics(ic_metrics_list, paste0("IC_", coverage * 100), names(all_results))
    
    cd_metrics_list <- lapply(all_results, function(res) compute_metrics(res$results[[paste0("CD_", coverage * 100)]]))
    plot_cumulative_metrics(cd_metrics_list, paste0("CD_", coverage * 100), names(all_results))
  }
}

# Run the computation and plotting for all models
compute_and_plot_all_models()


## Spectral Analysis

In [ ]:
# Function to calculate w values for spectral analysis
calculate_w_values <- function(T) {
  m <- floor(T / 2)
  K <- 1:m
  2 * pi * K / T
}

periodogram <- function(w, y) {
  T <- length(y)
  n <- length(w)
  s <- matrix(exp(-1i * outer(w, 1:T, "*")), nrow = n, ncol = T)
  I <- numeric(n)
  
  for (j in 1:n) {
    sum_s <- sum(y * s[j, ])
    I[j] <- abs(sum_s)^2 * 2 / T
  }
  
  I
}


# Log-likelihood function
loglikelihood_wavelength <- function(I, y) {
  T <- length(y)
  like <- 1 - I / sum(y^2)
  ((2 - T) / 2) * log(like)
}

# Function to perform spectral analysis
perform_spectral_analysis <- function(ts_data) {
  T <- length(ts_data)
  w <- calculate_w_values(T)
  I <- periodogram(w, ts_data)
  like <- loglikelihood_wavelength(I, ts_data)
  
  x <- 2 * pi / w
  like_filtered <- like[x < 6000]
  x_filtered <- x[x < 6000]
  
  # Improved peak detection
  peaks <- which(diff(sign(diff(like_filtered))) == -2) + 1
  peak_x <- x_filtered[peaks]
  peak_y <- like_filtered[peaks]
  
  list(like = like_filtered, x = x_filtered, peak_x = peak_x, peak_y = peak_y)
}

####################################################################################
####################################################################################

# Reading the CSV file

data_path <- "/home/jaguir26/projects/notebooks/combined_streamflow_data_cleaned.csv"
streamflow_data <- read_csv(data_path, show_col_types = FALSE)
timestamps <- as.Date(streamflow_data$Date)
time_series_matrix <- as.matrix(streamflow_data[, c('USGS', 'NWS3.0',  'GloFAS')])

# Create time series
start_year <- as.numeric(format(min(timestamps), "%Y"))
usgs_ts <- ts(time_series_matrix[,1], start = c(start_year, 1), frequency = 365)
glofas_ts <- ts(time_series_matrix[,3], start = c(start_year, 1), frequency = 365)
nws_ts <- ts(time_series_matrix[,2], start = c(start_year, 1), frequency = 365)

####################################################################################
####################################################################################

# Perform spectral analysis
usgs_spectral <- perform_spectral_analysis(usgs_ts)
glofas_spectral <- perform_spectral_analysis(glofas_ts)
nws_spectral <- perform_spectral_analysis(nws_ts)

# Plotting spectral analysis
plot_spectral <- function(spectral_data, title, color) {
  plot(spectral_data$x, spectral_data$like, type = "l", xlab = "Period (in days)", ylab = "log-likelihood", main = title, xlim =c(0,1500))
  points(spectral_data$peak_x, spectral_data$peak_y, pch = 16, col = color)
}


# Adjusting graphical parameters for better fit and efficient use of space
par(mfrow = c(3, 1),  # Setting layout to 3 rows, 1 column
    mar = c(2, 4, 2, 1) + 0.1,  # Setting margins: bottom, left, top, right
    oma = c(0.5, 0.5, 0.5, 0.5),  # Outer margins
    omd = c(0.1, 0.9, 0.1, 0.9))  # Outer margin dimensions

# Plotting spectral analysis for each series
plot_spectral(usgs_spectral, "Spectral Analysis: USGS", "red")
plot_spectral(glofas_spectral, "Spectral Analysis: GloFAS", "blue")
plot_spectral(nws_spectral, "Spectral Analysis: NWS", "green")

dev.off()
####################################################################################
####################################################################################

# Function to extract and combine peak information
extract_peak_info <- function(spectral_data, label) {
  data.frame(
    Series = rep(label, length(spectral_data$peak_x)),
    Peak_Period_Days = spectral_data$peak_x,
    Peak_Period_Years = spectral_data$peak_x / 365,
    Peak_Value = spectral_data$peak_y
  )
}

# Function to extract top 5 peaks
extract_top_peaks <- function(peaks_df) {
  peaks_df[order(-peaks_df$Peak_Value),][1:5, ]
}

# Extract peak information for each time series
usgs_peaks <- extract_peak_info(usgs_spectral, "USGS")
glofas_peaks <- extract_peak_info(glofas_spectral, "GloFAS")
nws_peaks <- extract_peak_info(nws_spectral, "NWS")

# Extract top 5 peaks for each time series
usgs_top_peaks <- extract_top_peaks(usgs_peaks)
glofas_top_peaks <- extract_top_peaks(glofas_peaks)
nws_top_peaks <- extract_top_peaks(nws_peaks)

# Combine the top peak information into a single data frame
all_top_peaks <- rbind(usgs_top_peaks, glofas_top_peaks, nws_top_peaks)

# Print the combined top peak information
print(all_top_peaks)